# Indian Crime Dataset — Common Schema Integration & EDA

## Objective

This notebook builds a new unified Indian crime dataset from the original
crime-related CSV files stored in:

`data/raw/crime_data/`

The raw source files will remain completely untouched.

The workflow will:

1. Discover and inventory all source CSV files.
2. Inspect their schemas, data types, row counts, and sample records.
3. Determine what each dataset and row represents.
4. Identify semantically comparable features across datasets.
5. Establish a small common schema based on actual evidence from the datasets.
6. Classify datasets according to compatibility with the common schema.
7. Standardize only compatible datasets.
8. Combine compatible records row-wise.
9. Validate the resulting unified dataset.
10. Perform exploratory data analysis (EDA).

### Important principles

- Raw files are never modified.
- No assumptions are made about what a row represents.
- Similar column names are not automatically treated as equivalent.
- Incompatible datasets will not be forced into the unified dataset.
- Missing values will not automatically be replaced with zero.
- The final integration will use row-wise concatenation, not STATE + YEAR merging.
- Every integrated record will retain its source dataset identifier.
- The common schema will be established only after inspecting the actual datasets.

In [3]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES AND DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Locate the project root
# ------------------------------------------------------------
# The notebook is inside:
#
# Crime_Analysis/notebooks/
#
# Therefore the project root is one level above the notebook
# working directory.

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "raw" / "crime_data").exists():

    # If Jupyter is running from Crime_Analysis/
    PROJECT_ROOT = CURRENT_DIR

elif (CURRENT_DIR.parent / "data" / "raw" / "crime_data").exists():

    # If Jupyter is running from Crime_Analysis/notebooks/
    PROJECT_ROOT = CURRENT_DIR.parent

else:

    raise FileNotFoundError(
        "Could not locate the Crime_Analysis project root.\n\n"
        "Expected to find:\n"
        "data/raw/crime_data/\n\n"
        f"Current directory:\n{CURRENT_DIR.resolve()}"
    )

# ------------------------------------------------------------
# Define project directories
# ------------------------------------------------------------

RAW_DATA_DIR = (
    PROJECT_ROOT / "data" / "raw" / "crime_data"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

TABLES_DIR = (
    PROJECT_ROOT / "outputs" / "tables"
)

FIGURES_DIR = (
    PROJECT_ROOT / "outputs" / "figures"
)

# ------------------------------------------------------------
# Create output directories if needed
# ------------------------------------------------------------
# This does NOT touch the raw CSV files.

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

EXPECTED_FILE_COUNT = 76

CSV_ENCODINGS = [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1"
]

# ------------------------------------------------------------
# Display paths
# ------------------------------------------------------------

print("=" * 70)
print("PROJECT PATH CONFIGURATION")
print("=" * 70)

print("\nCurrent working directory:")
print(CURRENT_DIR.resolve())

print("\nProject root:")
print(PROJECT_ROOT.resolve())

print("\nRaw data directory:")
print(RAW_DATA_DIR.resolve())

print("\nProcessed data directory:")
print(PROCESSED_DATA_DIR.resolve())

print("\nTables directory:")
print(TABLES_DIR.resolve())

print("\nFigures directory:")
print(FIGURES_DIR.resolve())

print("\n✓ Path configuration completed.")

PROJECT PATH CONFIGURATION

Current working directory:
D:\Major_Project\Crime_Analysis\notebooks

Project root:
D:\Major_Project\Crime_Analysis

Raw data directory:
D:\Major_Project\Crime_Analysis\data\raw\crime_data

Processed data directory:
D:\Major_Project\Crime_Analysis\data\processed

Tables directory:
D:\Major_Project\Crime_Analysis\outputs\tables

Figures directory:
D:\Major_Project\Crime_Analysis\outputs\figures

✓ Path configuration completed.


In [4]:
# ============================================================
# CELL 3 — DISCOVER ALL CSV FILES
# ============================================================

# Find every CSV file inside data/raw/crime_data/
# Recursively, without modifying any source file.

csv_files = sorted(
    [
        file_path
        for file_path in RAW_DATA_DIR.rglob("*")
        if file_path.is_file()
        and file_path.suffix.lower() == ".csv"
    ],
    key=lambda x: x.name.lower()
)

print("=" * 70)
print("CSV FILE DISCOVERY")
print("=" * 70)

print(f"\nRaw data directory:")
print(RAW_DATA_DIR.resolve())

print(f"\nNumber of CSV files found: {len(csv_files)}")
print(f"Expected number of CSV files: {EXPECTED_FILE_COUNT}")

if len(csv_files) == EXPECTED_FILE_COUNT:
    print("\nSTATUS: ✓ Exactly 76 CSV files found.")
elif len(csv_files) > EXPECTED_FILE_COUNT:
    print("\nSTATUS: ⚠ More than 76 CSV files found.")
else:
    print("\nSTATUS: ⚠ Fewer than 76 CSV files found.")

print("\n" + "-" * 70)
print("DISCOVERED FILES")
print("-" * 70)

for i, file_path in enumerate(csv_files, start=1):
    print(f"{i:02d}. {file_path.name}")

CSV FILE DISCOVERY

Raw data directory:
D:\Major_Project\Crime_Analysis\data\raw\crime_data

Number of CSV files found: 76
Expected number of CSV files: 76

STATUS: ✓ Exactly 76 CSV files found.

----------------------------------------------------------------------
DISCOVERED FILES
----------------------------------------------------------------------
01. 01_District_wise_crimes_committed_IPC_2001_2012.csv
02. 01_District_wise_crimes_committed_IPC_2013.csv
03. 01_District_wise_crimes_committed_IPC_2014.csv
04. 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
05. 02_01_District_wise_crimes_committed_against_SC_2013.csv
06. 02_01_District_wise_crimes_committed_against_SC_2014.csv
07. 02_District_wise_crimes_committed_against_ST_2001_2012.csv
08. 02_District_wise_crimes_committed_against_ST_2013.csv
09. 02_District_wise_crimes_committed_against_ST_2014.csv
10. 03_District_wise_crimes_committed_against_children_2001_2012.csv
11. 03_District_wise_crimes_committed_against_child

In [5]:
# ============================================================
# CELL 4 — VERIFY SOURCE FILE LOCATIONS
# ============================================================

print("=" * 70)
print("SOURCE FILE LOCATION VERIFICATION")
print("=" * 70)

outside_expected_folder = []

for file_path in csv_files:

    try:
        file_path.relative_to(RAW_DATA_DIR)
    except ValueError:
        outside_expected_folder.append(file_path)

if len(outside_expected_folder) == 0:

    print("\n✓ All discovered CSV files are inside:")
    print(RAW_DATA_DIR.resolve())

else:

    print(
        f"\n⚠ {len(outside_expected_folder)} files "
        "were found outside the expected directory."
    )

    for file_path in outside_expected_folder:
        print(file_path.resolve())

print("\nRaw files will remain untouched.")

SOURCE FILE LOCATION VERIFICATION

✓ All discovered CSV files are inside:
D:\Major_Project\Crime_Analysis\data\raw\crime_data

Raw files will remain untouched.


In [6]:
# ============================================================
# CELL 5 — ROBUST CSV READER
# ============================================================

def read_csv_for_inspection(file_path):
    """
    Read a CSV for inspection.

    The original file is never modified.

    Reading strategy:
    1. Try normal pandas CSV parsing using several encodings.
    2. If that fails, try the Python parser.
    3. Record malformed rows instead of silently hiding them.

    Returns
    -------
    df : pandas.DataFrame or None

    read_info : dict
        Information about encoding, parser and parsing problems.
    """

    attempts = []

    # --------------------------------------------------------
    # Attempt 1: normal pandas parser
    # --------------------------------------------------------

    for encoding in CSV_ENCODINGS:

        try:

            df = pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )

            return df, {
                "status": "success",
                "encoding": encoding,
                "parser": "default",
                "parsing_issue": False,
                "bad_rows": 0,
                "error": ""
            }

        except Exception as e:

            attempts.append(
                f"{encoding} / default parser: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Attempt 2: Python parser
    # --------------------------------------------------------

    for encoding in CSV_ENCODINGS:

        bad_lines = []

        def collect_bad_line(line):
            bad_lines.append(line)
            return None

        try:

            df = pd.read_csv(
                file_path,
                encoding=encoding,
                engine="python",
                on_bad_lines=collect_bad_line
            )

            return df, {
                "status": "success_with_parsing_issues",
                "encoding": encoding,
                "parser": "python",
                "parsing_issue": True,
                "bad_rows": len(bad_lines),
                "error": (
                    f"{len(bad_lines)} malformed row(s) "
                    f"detected during inspection."
                )
            }

        except Exception as e:

            attempts.append(
                f"{encoding} / python parser: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # All attempts failed
    # --------------------------------------------------------

    return None, {
        "status": "failed",
        "encoding": None,
        "parser": None,
        "parsing_issue": True,
        "bad_rows": None,
        "error": " | ".join(attempts)
    }


print("✓ CSV inspection function created.")

✓ CSV inspection function created.


In [7]:
# ============================================================
# CELL 6 — COLUMN CANDIDATE DETECTION
# ============================================================

def identify_column_candidates(columns):

    candidates = {
        "year_date_columns": [],
        "state_columns": [],
        "location_columns": [],
        "district_columns": [],
        "crime_columns": [],
        "victim_columns": [],
        "count_value_columns": []
    }

    for column in columns:

        original = str(column).strip()

        normalized = re.sub(
            r"[^a-z0-9]+",
            "_",
            original.lower()
        ).strip("_")

        # ----------------------------------------------------
        # Date / time
        # ----------------------------------------------------

        if any(
            keyword in normalized
            for keyword in [
                "year",
                "date",
                "time",
                "month",
                "day",
                "quarter"
            ]
        ):
            candidates["year_date_columns"].append(original)

        # ----------------------------------------------------
        # State
        # ----------------------------------------------------

        if any(
            keyword in normalized
            for keyword in [
                "state",
                "state_ut",
                "state_union_territory",
                "union_territory"
            ]
        ):
            candidates["state_columns"].append(original)

        # ----------------------------------------------------
        # Location
        # ----------------------------------------------------

        if any(
            keyword in normalized
            for keyword in [
                "state",
                "district",
                "city",
                "location",
                "place",
                "region",
                "area",
                "zone",
                "village",
                "town",
                "taluk",
                "tehsil"
            ]
        ):
            candidates["location_columns"].append(original)

        # ----------------------------------------------------
        # District
        # ----------------------------------------------------

        if "district" in normalized:
            candidates["district_columns"].append(original)

        # ----------------------------------------------------
        # Crime-related
        # ----------------------------------------------------

        if any(
            keyword in normalized
            for keyword in [
                "crime",
                "offence",
                "offense",
                "ipc",
                "section",
                "case",
                "incident"
            ]
        ):
            candidates["crime_columns"].append(original)

        # ----------------------------------------------------
        # Victim-related
        # ----------------------------------------------------

        if any(
            keyword in normalized
            for keyword in [
                "victim",
                "age",
                "gender",
                "sex"
            ]
        ):
            candidates["victim_columns"].append(original)

        # ----------------------------------------------------
        # Count / value-related
        # ----------------------------------------------------

        if any(
            keyword in normalized
            for keyword in [
                "count",
                "total",
                "number",
                "value",
                "cases",
                "persons",
                "incidents"
            ]
        ):
            candidates["count_value_columns"].append(original)

    return candidates


print("✓ Candidate-column detector created.")

✓ Candidate-column detector created.


In [8]:
# ============================================================
# CELL 7 — YEAR / DATE CANDIDATE DETECTION
# ============================================================

def get_year_like_columns(df):

    results = []

    for column in df.columns:

        series = df[column]

        normalized = re.sub(
            r"[^a-z0-9]+",
            "_",
            str(column).lower()
        ).strip("_")

        # Name-based detection
        name_candidate = any(
            keyword in normalized
            for keyword in [
                "year",
                "date",
                "month",
                "time"
            ]
        )

        # Value-based detection
        value_candidate = False

        if pd.api.types.is_numeric_dtype(series):

            non_null = series.dropna()

            if len(non_null) > 0:

                numeric_values = pd.to_numeric(
                    non_null,
                    errors="coerce"
                ).dropna()

                if len(numeric_values) > 0:

                    valid_years = (
                        (numeric_values >= 1900) &
                        (numeric_values <= 2100)
                    )

                    if (
                        valid_years.sum() /
                        len(numeric_values)
                    ) >= 0.5:

                        value_candidate = True

        if name_candidate or value_candidate:
            results.append(str(column))

    return results


print("✓ Year/date detector created.")

✓ Year/date detector created.


In [9]:
# ============================================================
# CELL 8 — BUILD COMPLETE 76-FILE INVENTORY
# ============================================================

inventory_records = []

print("=" * 70)
print("BUILDING SOURCE DATA INVENTORY")
print("=" * 70)

for file_number, file_path in enumerate(csv_files, start=1):

    print(
        f"[{file_number:02d}/{len(csv_files):02d}] "
        f"Reading: {file_path.name}"
    )

    df, read_info = read_csv_for_inspection(file_path)

    relative_path = str(
        file_path.relative_to(PROJECT_ROOT)
    )

    # --------------------------------------------------------
    # If file could not be read
    # --------------------------------------------------------

    if df is None:

        inventory_records.append({

            "file_number": file_number,

            "filename": file_path.name,

            "relative_path": relative_path,

            "row_count": np.nan,

            "column_count": np.nan,

            "column_names": "",

            "data_types": "",

            "year_date_columns": "",

            "state_columns": "",

            "location_columns": "",

            "district_columns": "",

            "crime_columns": "",

            "victim_columns": "",

            "count_value_columns": "",

            "read_status": read_info["status"],

            "encoding": read_info["encoding"],

            "parser": read_info["parser"],

            "parsing_issue": read_info["parsing_issue"],

            "bad_rows": read_info["bad_rows"],

            "read_error": read_info["error"]
        })

        continue

    # --------------------------------------------------------
    # Identify candidate columns
    # --------------------------------------------------------

    candidates = identify_column_candidates(
        df.columns
    )

    year_columns = get_year_like_columns(df)

    # --------------------------------------------------------
    # Data type information
    # --------------------------------------------------------

    dtype_summary = {
        str(column): str(dtype)
        for column, dtype in df.dtypes.items()
    }

    # --------------------------------------------------------
    # Store inventory information
    # --------------------------------------------------------

    inventory_records.append({

        "file_number": file_number,

        "filename": file_path.name,

        "relative_path": relative_path,

        "row_count": len(df),

        "column_count": len(df.columns),

        "column_names": json.dumps(
            [str(column) for column in df.columns],
            ensure_ascii=False
        ),

        "data_types": json.dumps(
            dtype_summary,
            ensure_ascii=False
        ),

        "year_date_columns": json.dumps(
            year_columns,
            ensure_ascii=False
        ),

        "state_columns": json.dumps(
            candidates["state_columns"],
            ensure_ascii=False
        ),

        "location_columns": json.dumps(
            candidates["location_columns"],
            ensure_ascii=False
        ),

        "district_columns": json.dumps(
            candidates["district_columns"],
            ensure_ascii=False
        ),

        "crime_columns": json.dumps(
            candidates["crime_columns"],
            ensure_ascii=False
        ),

        "victim_columns": json.dumps(
            candidates["victim_columns"],
            ensure_ascii=False
        ),

        "count_value_columns": json.dumps(
            candidates["count_value_columns"],
            ensure_ascii=False
        ),

        "read_status": read_info["status"],

        "encoding": read_info["encoding"],

        "parser": read_info["parser"],

        "parsing_issue": read_info["parsing_issue"],

        "bad_rows": read_info["bad_rows"],

        "read_error": read_info["error"]
    })


# ------------------------------------------------------------
# Create inventory DataFrame
# ------------------------------------------------------------

inventory_df = pd.DataFrame(
    inventory_records
)

print("\n" + "=" * 70)
print("INVENTORY COMPLETE")
print("=" * 70)

print(
    f"Files inventoried: {len(inventory_df)}"
)

if len(inventory_df) == EXPECTED_FILE_COUNT:
    print("STATUS: ✓ All 76 expected files were inventoried.")
else:
    print(
        "STATUS: ⚠ Inventory count does not equal 76."
    )

BUILDING SOURCE DATA INVENTORY
[01/76] Reading: 01_District_wise_crimes_committed_IPC_2001_2012.csv
[02/76] Reading: 01_District_wise_crimes_committed_IPC_2013.csv
[03/76] Reading: 01_District_wise_crimes_committed_IPC_2014.csv
[04/76] Reading: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
[05/76] Reading: 02_01_District_wise_crimes_committed_against_SC_2013.csv
[06/76] Reading: 02_01_District_wise_crimes_committed_against_SC_2014.csv
[07/76] Reading: 02_District_wise_crimes_committed_against_ST_2001_2012.csv
[08/76] Reading: 02_District_wise_crimes_committed_against_ST_2013.csv
[09/76] Reading: 02_District_wise_crimes_committed_against_ST_2014.csv
[10/76] Reading: 03_District_wise_crimes_committed_against_children_2001_2012.csv
[11/76] Reading: 03_District_wise_crimes_committed_against_children_2013.csv
[12/76] Reading: 03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2012.csv
[13/76] Reading: 03_Persons_arrested_and_their_disposa

In [10]:
# ============================================================
# CELL 9 — DISPLAY BASIC FILE INVENTORY
# ============================================================

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    120
)

pd.set_option(
    "display.width",
    200
)

inventory_display = inventory_df[
    [
        "file_number",
        "filename",
        "row_count",
        "column_count",
        "read_status",
        "parsing_issue",
        "bad_rows"
    ]
].copy()

display(inventory_display)

,file_number,filename,row_count,column_count,read_status,parsing_issue,bad_rows
0,1,01_District_wise_crimes_committed_IPC_2001_2012.csv,9017,33,success,False,0
1,2,01_District_wise_crimes_committed_IPC_2013.csv,823,33,success,False,0
2,3,01_District_wise_crimes_committed_IPC_2014.csv,838,91,success,False,0
3,4,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,9018,13,success,False,0
4,5,02_01_District_wise_crimes_committed_against_SC_2013.csv,823,13,success,False,0
...,...,...,...,...,...,...,...
71,72,42_Cases_under_crime_against_women.csv,2765,22,success_with_parsing_issues,True,1400
72,73,42_District_wise_crimes_committed_against_women_2001_2012.csv,9017,10,success,False,0
73,74,42_District_wise_crimes_committed_against_women_2013.csv,823,10,success,False,0
74,75,42_District_wise_crimes_committed_against_women_2014.csv,837,62,success,False,0


In [11]:
# ============================================================
# CELL 10 — TOTAL SOURCE ROW COUNT
# ============================================================

successful_files = inventory_df[
    inventory_df["row_count"].notna()
].copy()

total_source_rows = int(
    successful_files["row_count"].sum()
)

print("=" * 70)
print("SOURCE ROW COUNT")
print("=" * 70)

print(
    f"Files successfully inspected : "
    f"{len(successful_files)}"
)

print(
    f"Total source rows            : "
    f"{total_source_rows:,}"
)

print("\nIMPORTANT:")
print(
    "This is only the sum of rows across the 76 CSV files."
)

print(
    "It is NOT assumed to represent the number of "
    "individual crime incidents."
)

SOURCE ROW COUNT
Files successfully inspected : 76
Total source rows            : 147,753

IMPORTANT:
This is only the sum of rows across the 76 CSV files.
It is NOT assumed to represent the number of individual crime incidents.


In [12]:
# ============================================================
# CELL 11 — COLUMN NAMES OF ALL DATASETS
# ============================================================

print("=" * 70)
print("COLUMN NAMES BY DATASET")
print("=" * 70)

for _, row in inventory_df.iterrows():

    print("\n" + "-" * 70)
    print(f"FILE: {row['filename']}")
    print(f"ROWS: {row['row_count']}")
    print(f"COLUMNS: {row['column_count']}")
    print("-" * 70)

    if row["column_names"]:

        columns = json.loads(
            row["column_names"]
        )

        for i, column in enumerate(
            columns,
            start=1
        ):
            print(
                f"{i:02d}. {column}"
            )

    else:

        print("No column information available.")

COLUMN NAMES BY DATASET

----------------------------------------------------------------------
FILE: 01_District_wise_crimes_committed_IPC_2001_2012.csv
ROWS: 9017
COLUMNS: 33
----------------------------------------------------------------------
01. STATE/UT
02. DISTRICT
03. YEAR
04. MURDER
05. ATTEMPT TO MURDER
06. CULPABLE HOMICIDE NOT AMOUNTING TO MURDER
07. RAPE
08. CUSTODIAL RAPE
09. OTHER RAPE
10. KIDNAPPING & ABDUCTION
11. KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS
12. KIDNAPPING AND ABDUCTION OF OTHERS
13. DACOITY
14. PREPARATION AND ASSEMBLY FOR DACOITY
15. ROBBERY
16. BURGLARY
17. THEFT
18. AUTO THEFT
19. OTHER THEFT
20. RIOTS
21. CRIMINAL BREACH OF TRUST
22. CHEATING
23. COUNTERFIETING
24. ARSON
25. HURT/GREVIOUS HURT
26. DOWRY DEATHS
27. ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY
28. INSULT TO MODESTY OF WOMEN
29. CRUELTY BY HUSBAND OR HIS RELATIVES
30. IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES
31. CAUSING DEATH BY NEGLIGENCE
32. OTHER IPC CRIMES
33. TOTAL IPC

In [13]:
# ============================================================
# CELL 12 — COLUMN INVENTORY TABLE
# ============================================================

column_records = []

for _, row in inventory_df.iterrows():

    if not row["column_names"]:
        continue

    columns = json.loads(
        row["column_names"]
    )

    for column in columns:

        normalized = re.sub(
            r"[^a-z0-9]+",
            "_",
            str(column).lower()
        ).strip("_")

        column_records.append({

            "filename": row["filename"],

            "column_name": str(column),

            "normalized_column_name": normalized
        })


column_inventory_df = pd.DataFrame(
    column_records
)

print("=" * 70)
print("COLUMN INVENTORY")
print("=" * 70)

print(
    f"Total column occurrences: "
    f"{len(column_inventory_df):,}"
)

print(
    f"Unique column names: "
    f"{column_inventory_df['column_name'].nunique():,}"
)

display(
    column_inventory_df.head(100)
)

COLUMN INVENTORY
Total column occurrences: 1,469
Unique column names: 777


,filename,column_name,normalized_column_name
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,STATE/UT,state_ut
1,01_District_wise_crimes_committed_IPC_2001_2012.csv,DISTRICT,district
2,01_District_wise_crimes_committed_IPC_2001_2012.csv,YEAR,year
3,01_District_wise_crimes_committed_IPC_2001_2012.csv,MURDER,murder
4,01_District_wise_crimes_committed_IPC_2001_2012.csv,ATTEMPT TO MURDER,attempt_to_murder
...,...,...,...
95,01_District_wise_crimes_committed_IPC_2014.csv,Theft,theft
96,01_District_wise_crimes_committed_IPC_2014.csv,Auto Theft,auto_theft
97,01_District_wise_crimes_committed_IPC_2014.csv,Other Thefts,other_thefts
98,01_District_wise_crimes_committed_IPC_2014.csv,Unlawful Assembly,unlawful_assembly


In [14]:
# ============================================================
# CELL 13 — EXACT COMMON COLUMN NAMES
# ============================================================

column_frequency = (
    column_inventory_df
    .groupby("column_name")["filename"]
    .nunique()
    .reset_index(
        name="dataset_count"
    )
    .sort_values(
        ["dataset_count", "column_name"],
        ascending=[False, True]
    )
)

print("=" * 70)
print("EXACT COLUMN-NAME FREQUENCY")
print("=" * 70)

print(
    "NOTE: Exact name similarity does NOT prove "
    "semantic equivalence."
)

display(
    column_frequency.head(100)
)

EXACT COLUMN-NAME FREQUENCY
NOTE: Exact name similarity does NOT prove semantic equivalence.


,column_name,dataset_count
776,Year,60
82,Area_Name,40
704,STATE/UT,26
710,Sub_Group_Name,24
291,Group_Name,21
...,...,...
572,Persons on bail during inv stage at beginning of Year_Total,3
573,Persons on bail during trial stage at Year End_Female,3
574,Persons on bail during trial stage at Year End_Male,3
575,Persons on bail during trial stage at Year End_Total,3


In [15]:
# ============================================================
# CELL 14 — PARSING PROBLEM REPORT
# ============================================================

problem_files = inventory_df[
    inventory_df["parsing_issue"] == True
].copy()

failed_files = inventory_df[
    inventory_df["read_status"] == "failed"
].copy()

print("=" * 70)
print("PARSING / READING PROBLEM REPORT")
print("=" * 70)

print(
    f"\nFiles with parsing issues : "
    f"{len(problem_files)}"
)

print(
    f"Files that failed reading : "
    f"{len(failed_files)}"
)

# ------------------------------------------------------------
# Parsing issues
# ------------------------------------------------------------

if len(problem_files) > 0:

    print("\nFiles with parsing issues:")

    display(
        problem_files[
            [
                "filename",
                "read_status",
                "encoding",
                "parser",
                "bad_rows",
                "read_error"
            ]
        ]
    )

else:

    print("\n✓ No parsing issues detected.")

# ------------------------------------------------------------
# Completely failed files
# ------------------------------------------------------------

if len(failed_files) > 0:

    print("\nFiles that could not be read:")

    display(
        failed_files[
            [
                "filename",
                "read_status",
                "read_error"
            ]
        ]
    )

else:

    print("\n✓ No files completely failed to read.")

PARSING / READING PROBLEM REPORT

Files with parsing issues : 3
Files that failed reading : 0

Files with parsing issues:


,filename,read_status,encoding,parser,bad_rows,read_error
61,36_Police_housing.csv,success_with_parsing_issues,utf-8,python,348,348 malformed row(s) detected during inspection.
71,42_Cases_under_crime_against_women.csv,success_with_parsing_issues,utf-8,python,1400,1400 malformed row(s) detected during inspection.
75,43_Arrests_under_crime_against_women.csv,success_with_parsing_issues,utf-8,python,1400,1400 malformed row(s) detected during inspection.



✓ No files completely failed to read.


In [16]:
# ============================================================
# CELL 15 — SAVE 76-FILE INVENTORY
# ============================================================

inventory_output_path = (
    TABLES_DIR / "76_file_inventory.csv"
)

inventory_df.to_csv(
    inventory_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("INVENTORY SAVED")
print("=" * 70)

print(
    "\nSaved to:"
)

print(
    inventory_output_path.resolve()
)

print(
    f"\nInventory rows: {len(inventory_df)}"
)

print(
    "\n✓ Original raw CSV files were not modified."
)

INVENTORY SAVED

Saved to:
D:\Major_Project\Crime_Analysis\outputs\tables\76_file_inventory.csv

Inventory rows: 76

✓ Original raw CSV files were not modified.


In [18]:
# ============================================================
# CELL 16 — PHASE 1 + PHASE 2 VERIFICATION
# ============================================================

print("=" * 70)
print("PHASE 1 + PHASE 2 VERIFICATION")
print("=" * 70)

print(
    f"\nCSV files discovered       : "
    f"{len(csv_files)}"
)

print(
    f"Files inventoried          : "
    f"{len(inventory_df)}"
)

print(
    f"Total source rows          : "
    f"{total_source_rows:,}"
)

print(
    f"Unique column names        : "
    f"{column_inventory_df['column_name'].nunique():,}"
)

print(
    f"Files with parsing issues  : "
    f"{inventory_df['parsing_issue'].sum()}"
)

print(
    f"Files completely unreadable: "
    f"{len(failed_files)}"
)

print("\n" + "-" * 70)
print("OUTPUT FILE")
print("-" * 70)

print(
    inventory_output_path.resolve()
)

print("\n" + "-" * 70)
print("STATUS")
print("-" * 70)

if len(csv_files) == 76:
    print("✓ Exactly 76 CSV files discovered")
else:
    print("⚠ CSV file count is not 76")

if len(inventory_df) == 76:
    print("✓ All 76 files inventoried")
else:
    print("⚠ Inventory count is not 76")

if len(failed_files) == 0:
    print("✓ No files completely failed")
else:
    print("⚠ Some files require additional inspection")


PHASE 1 + PHASE 2 VERIFICATION

CSV files discovered       : 76
Files inventoried          : 76
Total source rows          : 147,753
Unique column names        : 777
Files with parsing issues  : 3
Files completely unreadable: 0

----------------------------------------------------------------------
OUTPUT FILE
----------------------------------------------------------------------
D:\Major_Project\Crime_Analysis\outputs\tables\76_file_inventory.csv

----------------------------------------------------------------------
STATUS
----------------------------------------------------------------------
✓ Exactly 76 CSV files discovered
✓ All 76 files inventoried
✓ No files completely failed


# Phase 3 — Understanding Dataset and Row Semantics

Before creating a common schema, each source dataset must be understood
according to what one row represents.

For every CSV, this phase examines:

- number of rows and columns
- column names
- data types
- sample records
- geographic indicators
- temporal indicators
- crime-related fields
- count/measure fields
- victim-related fields
- police/arrest/court/property indicators

The automated analysis below provides evidence and candidate interpretations.

It does NOT automatically declare semantic equivalence between datasets.

Any dataset whose meaning cannot be established reliably from its schema
and sample records will be flagged for manual review.

In [19]:
# ============================================================
# CELL 18 — VERIFY INVENTORY FOR PHASE 3
# ============================================================

print("=" * 70)
print("PHASE 3 — INVENTORY CHECK")
print("=" * 70)

print(f"Inventory files : {len(inventory_df)}")
print(f"Source CSVs     : {len(csv_files)}")

if len(inventory_df) != len(csv_files):
    raise ValueError(
        "Inventory count and discovered CSV count do not match."
    )

print("\n✓ Inventory and discovered files match.")
print("✓ Ready for semantic dataset inspection.")

PHASE 3 — INVENTORY CHECK
Inventory files : 76
Source CSVs     : 76

✓ Inventory and discovered files match.
✓ Ready for semantic dataset inspection.


In [20]:
# ============================================================
# CELL 19 — DATASET PROFILE FUNCTION
# ============================================================

def create_dataset_profile(df, filename):
    """
    Generate evidence about what a dataset may represent.

    IMPORTANT:
    This function creates candidate evidence only.
    It does not establish final semantic compatibility.
    """

    columns = [str(c).strip() for c in df.columns]

    normalized_columns = [
        re.sub(
            r"[^a-z0-9]+",
            "_",
            c.lower()
        ).strip("_")
        for c in columns
    ]

    # --------------------------------------------------------
    # Geographic evidence
    # --------------------------------------------------------

    state_cols = []
    district_cols = []
    location_cols = []

    for original, normalized in zip(
        columns,
        normalized_columns
    ):

        if any(
            key in normalized
            for key in [
                "state",
                "state_ut",
                "states_uts",
                "union_territory"
            ]
        ):
            state_cols.append(original)

        if "district" in normalized:
            district_cols.append(original)

        if any(
            key in normalized
            for key in [
                "location",
                "place",
                "city",
                "region",
                "area",
                "zone",
                "village",
                "town",
                "taluk",
                "tehsil"
            ]
        ):
            location_cols.append(original)

    # --------------------------------------------------------
    # Temporal evidence
    # --------------------------------------------------------

    year_cols = []

    for original, normalized in zip(
        columns,
        normalized_columns
    ):

        if any(
            key in normalized
            for key in [
                "year",
                "date",
                "month",
                "quarter",
                "time"
            ]
        ):
            year_cols.append(original)

    # --------------------------------------------------------
    # Crime evidence
    # --------------------------------------------------------

    crime_cols = []

    for original, normalized in zip(
        columns,
        normalized_columns
    ):

        if any(
            key in normalized
            for key in [
                "crime",
                "offence",
                "offense",
                "ipc",
                "case",
                "incident",
                "murder",
                "rape",
                "robbery",
                "theft",
                "kidnapping"
            ]
        ):
            crime_cols.append(original)

    # --------------------------------------------------------
    # Victim evidence
    # --------------------------------------------------------

    victim_cols = []

    for original, normalized in zip(
        columns,
        normalized_columns
    ):

        if any(
            key in normalized
            for key in [
                "victim",
                "age",
                "gender",
                "sex"
            ]
        ):
            victim_cols.append(original)

    # --------------------------------------------------------
    # Arrest / police evidence
    # --------------------------------------------------------

    arrest_police_cols = []

    for original, normalized in zip(
        columns,
        normalized_columns
    ):

        if any(
            key in normalized
            for key in [
                "arrest",
                "police",
                "custody",
                "investigation",
                "charge_sheet",
                "chargesheet",
                "convicted",
                "acquitted",
                "trial"
            ]
        ):
            arrest_police_cols.append(original)

    # --------------------------------------------------------
    # Property evidence
    # --------------------------------------------------------

    property_cols = []

    for original, normalized in zip(
        columns,
        normalized_columns
    ):

        if any(
            key in normalized
            for key in [
                "property",
                "vehicle",
                "stolen",
                "recovered",
                "loss",
                "value"
            ]
        ):
            property_cols.append(original)

    # --------------------------------------------------------
    # Numeric columns
    # --------------------------------------------------------

    numeric_columns = [
        str(c)
        for c in df.select_dtypes(
            include=np.number
        ).columns
    ]

    # --------------------------------------------------------
    # Candidate subject indicators
    # --------------------------------------------------------

    subject_evidence = []

    if arrest_police_cols:
        subject_evidence.append(
            "police/arrest/justice-related fields"
        )

    if victim_cols:
        subject_evidence.append(
            "victim/demographic-related fields"
        )

    if property_cols:
        subject_evidence.append(
            "property/vehicle/value-related fields"
        )

    if crime_cols:
        subject_evidence.append(
            "crime/case/offence-related fields"
        )

    if not subject_evidence:
        subject_evidence.append(
            "no obvious crime-specific indicator detected"
        )

    # --------------------------------------------------------
    # Candidate geographic level
    # --------------------------------------------------------

    if district_cols:
        geographic_candidate = (
            "district-level information may be present"
        )

    elif state_cols:
        geographic_candidate = (
            "state/UT-level information may be present"
        )

    elif location_cols:
        geographic_candidate = (
            "location information may be present"
        )

    else:
        geographic_candidate = (
            "no obvious geographic field detected"
        )

    # --------------------------------------------------------
    # Candidate temporal level
    # --------------------------------------------------------

    if year_cols:

        if len(year_cols) == 1:
            temporal_candidate = (
                "year/date information may be present"
            )
        else:
            temporal_candidate = (
                "multiple year/date-related fields detected"
            )

    else:

        temporal_candidate = (
            "no obvious time field detected"
        )

    return {

        "filename": filename,

        "row_count": len(df),

        "column_count": len(df.columns),

        "geographic_level_candidate":
            geographic_candidate,

        "time_level_candidate":
            temporal_candidate,

        "crime_information_candidate":
            "; ".join(crime_cols),

        "victim_information_candidate":
            "; ".join(victim_cols),

        "police_arrest_information_candidate":
            "; ".join(arrest_police_cols),

        "property_information_candidate":
            "; ".join(property_cols),

        "state_columns":
            "; ".join(state_cols),

        "district_columns":
            "; ".join(district_cols),

        "location_columns":
            "; ".join(location_cols),

        "year_date_columns":
            "; ".join(year_cols),

        "numeric_columns":
            "; ".join(numeric_columns),

        "subject_evidence":
            "; ".join(subject_evidence),

        # IMPORTANT:
        # These remain blank until manually verified.
        "geographic_level":
            "",

        "time_level":
            "",

        "subject_type":
            "",

        "manual_review_required":
            True,

        "manual_notes":
            ""
    }

In [21]:
# ============================================================
# CELL 20 — BUILD DATASET SEMANTIC PROFILE
# ============================================================

dataset_profiles = []

print("=" * 70)
print("BUILDING SEMANTIC PROFILE FOR ALL 76 DATASETS")
print("=" * 70)

for i, file_path in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{i:02d}/{len(csv_files):02d}] "
        f"Profiling: {file_path.name}"
    )

    df, read_info = read_csv_for_inspection(
        file_path
    )

    if df is None:

        dataset_profiles.append({

            "filename": file_path.name,

            "row_count": np.nan,

            "column_count": np.nan,

            "geographic_level_candidate":
                "FILE COULD NOT BE READ",

            "time_level_candidate":
                "FILE COULD NOT BE READ",

            "crime_information_candidate": "",

            "victim_information_candidate": "",

            "police_arrest_information_candidate": "",

            "property_information_candidate": "",

            "state_columns": "",

            "district_columns": "",

            "location_columns": "",

            "year_date_columns": "",

            "numeric_columns": "",

            "subject_evidence": "",

            "geographic_level": "",

            "time_level": "",

            "subject_type": "",

            "manual_review_required": True,

            "manual_notes":
                "File could not be read."
        })

        continue

    profile = create_dataset_profile(
        df,
        file_path.name
    )

    dataset_profiles.append(profile)


dataset_profile_df = pd.DataFrame(
    dataset_profiles
)

print("\n" + "=" * 70)
print("SEMANTIC PROFILE COMPLETE")
print("=" * 70)

print(
    f"Datasets profiled: "
    f"{len(dataset_profile_df)}"
)

BUILDING SEMANTIC PROFILE FOR ALL 76 DATASETS
[01/76] Profiling: 01_District_wise_crimes_committed_IPC_2001_2012.csv
[02/76] Profiling: 01_District_wise_crimes_committed_IPC_2013.csv
[03/76] Profiling: 01_District_wise_crimes_committed_IPC_2014.csv
[04/76] Profiling: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
[05/76] Profiling: 02_01_District_wise_crimes_committed_against_SC_2013.csv
[06/76] Profiling: 02_01_District_wise_crimes_committed_against_SC_2014.csv
[07/76] Profiling: 02_District_wise_crimes_committed_against_ST_2001_2012.csv
[08/76] Profiling: 02_District_wise_crimes_committed_against_ST_2013.csv
[09/76] Profiling: 02_District_wise_crimes_committed_against_ST_2014.csv
[10/76] Profiling: 03_District_wise_crimes_committed_against_children_2001_2012.csv
[11/76] Profiling: 03_District_wise_crimes_committed_against_children_2013.csv
[12/76] Profiling: 03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2012.csv
[13/76] Profili

In [22]:
# ============================================================
# CELL 21 — DISPLAY DATASET PROFILE
# ============================================================

profile_display = dataset_profile_df[
    [
        "filename",
        "row_count",
        "column_count",
        "geographic_level_candidate",
        "time_level_candidate",
        "subject_evidence",
        "manual_review_required"
    ]
].copy()

display(
    profile_display
)

,filename,row_count,column_count,geographic_level_candidate,time_level_candidate,subject_evidence,manual_review_required
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,9017,33,district-level information may be present,year/date information may be present,victim/demographic-related fields; crime/case/offence-related fields,True
1,01_District_wise_crimes_committed_IPC_2013.csv,823,33,district-level information may be present,year/date information may be present,victim/demographic-related fields; crime/case/offence-related fields,True
2,01_District_wise_crimes_committed_IPC_2014.csv,838,91,district-level information may be present,year/date information may be present,police/arrest/justice-related fields; victim/demographic-related fields; crime/case/offence-related fields,True
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,9018,13,district-level information may be present,year/date information may be present,crime/case/offence-related fields,True
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,823,13,district-level information may be present,year/date information may be present,crime/case/offence-related fields,True
...,...,...,...,...,...,...,...
71,42_Cases_under_crime_against_women.csv,2765,22,location information may be present,multiple year/date-related fields detected,police/arrest/justice-related fields; crime/case/offence-related fields,True
72,42_District_wise_crimes_committed_against_women_2001_2012.csv,9017,10,district-level information may be present,year/date information may be present,victim/demographic-related fields; crime/case/offence-related fields,True
73,42_District_wise_crimes_committed_against_women_2013.csv,823,10,district-level information may be present,year/date information may be present,victim/demographic-related fields; crime/case/offence-related fields,True
74,42_District_wise_crimes_committed_against_women_2014.csv,837,62,district-level information may be present,year/date information may be present,victim/demographic-related fields; crime/case/offence-related fields,True


In [23]:
# ============================================================
# CELL 22 — DETAILED DATASET EVIDENCE
# ============================================================

for _, row in dataset_profile_df.iterrows():

    print("\n" + "=" * 80)
    print(f"DATASET: {row['filename']}")
    print("=" * 80)

    print(f"Rows       : {row['row_count']}")
    print(f"Columns    : {row['column_count']}")

    print(
        f"\nGeographic candidate:"
        f"\n{row['geographic_level_candidate']}"
    )

    print(
        f"\nTime candidate:"
        f"\n{row['time_level_candidate']}"
    )

    print(
        f"\nSubject evidence:"
        f"\n{row['subject_evidence']}"
    )

    print(
        f"\nCrime-related columns:"
        f"\n{row['crime_information_candidate'] or 'None detected'}"
    )

    print(
        f"\nVictim-related columns:"
        f"\n{row['victim_information_candidate'] or 'None detected'}"
    )

    print(
        f"\nPolice/arrest/justice columns:"
        f"\n{row['police_arrest_information_candidate'] or 'None detected'}"
    )

    print(
        f"\nProperty-related columns:"
        f"\n{row['property_information_candidate'] or 'None detected'}"
    )

    print(
        f"\nState columns:"
        f"\n{row['state_columns'] or 'None detected'}"
    )

    print(
        f"\nDistrict columns:"
        f"\n{row['district_columns'] or 'None detected'}"
    )

    print(
        f"\nYear/date columns:"
        f"\n{row['year_date_columns'] or 'None detected'}"
    )


DATASET: 01_District_wise_crimes_committed_IPC_2001_2012.csv
Rows       : 9017
Columns    : 33

Geographic candidate:
district-level information may be present

Time candidate:
year/date information may be present

Subject evidence:
victim/demographic-related fields; crime/case/offence-related fields

Crime-related columns:
MURDER; ATTEMPT TO MURDER; CULPABLE HOMICIDE NOT AMOUNTING TO MURDER; RAPE; CUSTODIAL RAPE; OTHER RAPE; KIDNAPPING & ABDUCTION; KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS; KIDNAPPING AND ABDUCTION OF OTHERS; ROBBERY; THEFT; AUTO THEFT; OTHER THEFT; OTHER IPC CRIMES; TOTAL IPC CRIMES

Victim-related columns:
ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY

Police/arrest/justice columns:
None detected

Property-related columns:
None detected

State columns:
STATE/UT

District columns:
DISTRICT

Year/date columns:
YEAR

DATASET: 01_District_wise_crimes_committed_IPC_2013.csv
Rows       : 823
Columns    : 33

Geographic candidate:
district-level information may be

In [24]:
# ============================================================
# CELL 23 — SAMPLE ROW INSPECTION FOR ALL DATASETS
# ============================================================

SAMPLE_ROWS = 5

for i, file_path in enumerate(
    csv_files,
    start=1
):

    print("\n" + "=" * 90)
    print(
        f"[{i:02d}/{len(csv_files):02d}] "
        f"{file_path.name}"
    )
    print("=" * 90)

    df, read_info = read_csv_for_inspection(
        file_path
    )

    if df is None:

        print("Unable to read this file.")
        continue

    print(
        f"Shape: {df.shape}"
    )

    print("\nColumns:")

    for column in df.columns:
        print(f"  - {column}")

    print("\nSample rows:")

    display(
        df.head(SAMPLE_ROWS)
    )


[01/76] 01_District_wise_crimes_committed_IPC_2001_2012.csv
Shape: (9017, 33)

Columns:
  - STATE/UT
  - DISTRICT
  - YEAR
  - MURDER
  - ATTEMPT TO MURDER
  - CULPABLE HOMICIDE NOT AMOUNTING TO MURDER
  - RAPE
  - CUSTODIAL RAPE
  - OTHER RAPE
  - KIDNAPPING & ABDUCTION
  - KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS
  - KIDNAPPING AND ABDUCTION OF OTHERS
  - DACOITY
  - PREPARATION AND ASSEMBLY FOR DACOITY
  - ROBBERY
  - BURGLARY
  - THEFT
  - AUTO THEFT
  - OTHER THEFT
  - RIOTS
  - CRIMINAL BREACH OF TRUST
  - CHEATING
  - COUNTERFIETING
  - ARSON
  - HURT/GREVIOUS HURT
  - DOWRY DEATHS
  - ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY
  - INSULT TO MODESTY OF WOMEN
  - CRUELTY BY HUSBAND OR HIS RELATIVES
  - IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES
  - CAUSING DEATH BY NEGLIGENCE
  - OTHER IPC CRIMES
  - TOTAL IPC CRIMES

Sample rows:


,STATE/UT,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES
0,ANDHRA PRADESH,ADILABAD,2001,101,60,17,50,0,50,46,30,16,9,0,41,198,199,22,177,78,16,104,1,30,1131,16,149,34,175,0,181,1518,4154
1,ANDHRA PRADESH,ANANTAPUR,2001,151,125,1,23,0,23,53,30,23,8,0,16,191,366,57,309,168,11,65,8,69,1543,7,118,24,154,0,270,754,4125
2,ANDHRA PRADESH,CHITTOOR,2001,101,57,2,27,0,27,59,34,25,4,0,14,237,723,164,559,156,33,209,9,38,2088,14,112,83,186,0,404,1262,5818
3,ANDHRA PRADESH,CUDDAPAH,2001,80,53,1,20,0,20,25,20,5,1,0,4,98,173,36,137,164,12,37,2,23,795,17,126,38,57,0,233,1181,3140
4,ANDHRA PRADESH,EAST GODAVARI,2001,82,67,1,23,0,23,49,26,23,4,0,25,437,1021,150,871,70,50,220,3,41,1244,12,109,58,247,0,431,2313,6507



[02/76] 01_District_wise_crimes_committed_IPC_2013.csv
Shape: (823, 33)

Columns:
  - STATE/UT
  - DISTRICT
  - YEAR
  - MURDER
  - ATTEMPT TO MURDER
  - CULPABLE HOMICIDE NOT AMOUNTING TO MURDER
  - RAPE
  - CUSTODIAL RAPE
  - OTHER RAPE
  - KIDNAPPING & ABDUCTION
  - KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS
  - KIDNAPPING AND ABDUCTION OF OTHERS
  - DACOITY
  - PREPARATION AND ASSEMBLY FOR DACOITY
  - ROBBERY
  - BURGLARY
  - THEFT
  - AUTO THEFT
  - OTHER THEFT
  - RIOTS
  - CRIMINAL BREACH OF TRUST
  - CHEATING
  - COUNTERFIETING
  - ARSON
  - HURT/GREVIOUS HURT
  - DOWRY DEATHS
  - ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY
  - INSULT TO MODESTY OF WOMEN
  - CRUELTY BY HUSBAND OR HIS RELATIVES
  - IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES
  - CAUSING DEATH BY NEGLIGENCE
  - OTHER IPC CRIMES
  - TOTAL IPC CRIMES

Sample rows:


,STATE/UT,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES
0,Andhra Pradesh,ADILABAD,2013,96,72,13,61,0,61,65,47,18,2,0,14,274,377,86,291,58,93,254,1,30,2394,12,197,138,464,0,376,1390,6381
1,Andhra Pradesh,ANANTAPUR,2013,156,149,3,28,0,28,110,84,26,5,0,23,279,597,154,443,56,5,160,5,29,2537,23,337,43,161,0,573,1634,6913
2,Andhra Pradesh,CHITTOOR,2013,72,61,2,31,0,31,52,27,25,3,0,11,157,512,158,354,57,17,238,6,18,937,13,119,84,435,0,546,2239,5610
3,Andhra Pradesh,CUDDAPAH,2013,93,107,7,19,0,19,84,50,34,2,0,9,220,702,255,447,156,81,317,5,34,2310,9,318,163,207,0,464,1741,7048
4,Andhra Pradesh,CYBERABAD,2013,162,123,16,138,0,138,192,129,63,15,0,89,1318,4779,1761,3018,34,179,2111,12,40,4284,43,350,338,1526,0,1104,3139,19992



[03/76] 01_District_wise_crimes_committed_IPC_2014.csv
Shape: (838, 91)

Columns:
  - States/UTs
  - District
  - Year
  - Murder
  - Attempt to commit Murder
  - Culpable Homicide not amounting to Murder
  - Attempt to commit Culpable Homicide
  - Rape
  - Custodial Rape
  - Custodial_Gang Rape
  - Custodial_Other Rape
  - Rape other than Custodial
  - Rape_Gang Rape
  - Rape_Others
  - Attempt to commit Rape
  - Kidnapping & Abduction_Total
  - Kidnapping & Abduction
  - Kidnapping & Abduction in order to Murder
  - Kidnapping for Ransom
  - Kidnapping & Abduction of Women to compel her for marriage
  - Other Kidnapping
  - Dacoity
  - Dacoity with Murder
  - Other Dacoity
  - Making Preparation and Assembly for committing Dacoity
  - Robbery
  - Criminal Trespass/Burglary
  - Criminal Trespass or Burglary
  - House Trespass & House Breaking
  - Theft
  - Auto Theft
  - Other Thefts
  - Unlawful Assembly
  - Riots
  - Riots_Communal
  - Riots_Industrial
  - Riots_Political
  - Riots

...

,STATE/UT,DISTRICT,Year,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls
0,Andhra Pradesh,ADILABAD,2013,61,47,12,197,138,464,0
1,Andhra Pradesh,ANANTAPUR,2013,28,84,23,337,43,161,0
2,Andhra Pradesh,CHITTOOR,2013,31,27,13,119,84,435,0
3,Andhra Pradesh,CUDDAPAH,2013,19,50,9,318,163,207,0
4,Andhra Pradesh,CYBERABAD,2013,138,129,43,350,338,1526,0



[75/76] 42_District_wise_crimes_committed_against_women_2014.csv
Shape: (837, 62)

Columns:
  - States/UTs
  - District
  - Year
  - Rape
  - Custodial Rape
  - Custodial_Gang Rape
  - Custodial_Other Rape
  - Rape other than Custodial
  - Rape_Gang Rape
  - Rape_Others
  - Attempt to commit Rape
  - Kidnapping & Abduction_Total
  - Kidnaping & Abduction
  - Kidnaping & Abduction in order to Murder
  - Kidnapping for Ransom
  - Kidnapping & Abduction of Women to compel her for marriage
  - Kidnaping & Abduction_Others
  - Dowry Deaths
  - Assault on Women with intent to outrage her Modesty_Total
  - Sexual Harassment
  - Assault on women with intent to Disrobe
  - Voyeurism
  - Stalking
  - Others
  - Insult to the Modesty of Women_Total
  - At Office premises
  - In places related to work
  - In Public Transport system
  - In other Places
  - Cruelty by Husband or his Relatives
  - Importation of Girls from Foreign Country
  - Murder
  - Attempt to commit Murder
  - Culpable Homicide

,States/UTs,District,Year,Rape,Custodial Rape,Custodial_Gang Rape,Custodial_Other Rape,Rape other than Custodial,Rape_Gang Rape,Rape_Others,Attempt to commit Rape,Kidnapping & Abduction_Total,Kidnaping & Abduction,Kidnaping & Abduction in order to Murder,Kidnapping for Ransom,Kidnapping & Abduction of Women to compel her for marriage,Kidnaping & Abduction_Others,Dowry Deaths,Assault on Women with intent to outrage her Modesty_Total,Sexual Harassment,Assault on women with intent to Disrobe,Voyeurism,Stalking,Others,Insult to the Modesty of Women_Total,At Office premises,In places related to work,In Public Transport system,In other Places,Cruelty by Husband or his Relatives,Importation of Girls from Foreign Country,Murder,Attempt to commit Murder,Culpable Homicide not amounting to Murder,Attempt to commit Culpable Homicide,Grievous Hurt,Hurt,Acid attack,Attempt to Acid Attack,Deaths caused with intent to cause miscarriage,Causing miscarriage without consent of women,Dacoity_Total,Dacoity with Murder,Other Dacoity,Robbery,Arson,HumanTrafficking,Abetment of Suicides of Women,UnNatural Offences,Other IPC Crimes,"Dowry Prohibition Act, 1961","Indecent Representation of Women (P) Act, 1986","Commission of Sati Prevention Act, 1987","Protection of Women from Domestic Violence Act, 2005",Immoral Traffic Prevention Act,ITP Under Section 5,ITP Under Section 6,ITP Under Section 7,ITP Under Section 8,ITP Under Other Sections,Other SLL Crimes against Women,Total Crimes against Women
0,Andhra Pradesh,Anantapur,2014,35,0,0,0,35,0,35,1,106,0,0,0,88,18,25,436,82,34,4,80,236,26,0,0,0,26,165,0,50,20,1,0,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,229,0,0,0,0,0,0,0,0,0,0,1097
1,Andhra Pradesh,Chittoor,2014,32,0,0,0,32,1,31,0,34,3,0,1,28,2,17,135,0,0,0,0,135,94,0,0,0,94,278,0,11,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,4,0,0,0,0,0,607
2,Andhra Pradesh,Cuddapah,2014,28,0,0,0,28,0,28,4,16,0,0,0,11,5,16,215,212,0,0,0,3,12,1,11,0,0,91,0,27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,20,0,0,175,0,0,0,5,0,0,0,0,5,0,609
3,Andhra Pradesh,East Godavari,2014,85,0,0,0,85,0,85,18,25,0,0,0,0,25,7,519,159,61,11,55,233,62,0,0,0,62,464,0,32,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,27,0,0,22,0,0,0,16,0,0,0,0,16,0,1277
4,Andhra Pradesh,Guntakal Railway,2014,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4



[76/76] 43_Arrests_under_crime_against_women.csv
Shape: (2765, 16)

Columns:
  - Area_Name
  - Year
  - Group_Name
  - Sub_Group_Name
  - Persons_Acquitted
  - Persons_against_whom_cases_Compounded_or_Withdrawn
  - Persons_Arrested
  - Persons_Chargesheeted
  - Persons_Convicted
  - Persons_in_Custody_or_on_Bail_during_Investigation_at_Year_beginning
  - Persons_in_Custody_or_on_Bail_during_Investigation_at_Year_end
  - Persons_in_Custody_or_on_Bail_during_Trial_at_Year_End
  - Persons_Released_or_Freed_by_Police_or_Magistrate_before_Trial_for_want_of_evidence_or_any_other_reason
  - Persons_Trial_Completed
  - Persons_under_Trial_at_Year_beginning
  - Total_Persons_under_Trial

Sample rows:


,Area_Name,Year,Group_Name,Sub_Group_Name,Persons_Acquitted,Persons_against_whom_cases_Compounded_or_Withdrawn,Persons_Arrested,Persons_Chargesheeted,Persons_Convicted,Persons_in_Custody_or_on_Bail_during_Investigation_at_Year_beginning,Persons_in_Custody_or_on_Bail_during_Investigation_at_Year_end,Persons_in_Custody_or_on_Bail_during_Trial_at_Year_End,Persons_Released_or_Freed_by_Police_or_Magistrate_before_Trial_for_want_of_evidence_or_any_other_reason,Persons_Trial_Completed,Persons_under_Trial_at_Year_beginning,Total_Persons_under_Trial
0,Andaman & Nicobar Islands,2001,Rape,01. Rape,6,0,3,3,0,6,6,45,0,6,48,51
1,Andhra Pradesh,2001,Rape,01. Rape,1168,13,1150,1021,246,450,545,2191,34,1414,2597,3618
2,Arunachal Pradesh,2001,Rape,01. Rape,1,0,51,31,2,25,30,347,15,3,319,350
3,Assam,2001,Rape,01. Rape,403,14,928,585,120,806,959,2331,190,523,2283,2868
4,Bihar,2001,Rape,01. Rape,756,0,1400,1302,217,719,576,5963,241,973,5634,6936


In [25]:
# ============================================================
# CELL 24 — COMPACT SAMPLE EVIDENCE TABLE
# ============================================================

sample_evidence_records = []

for file_path in csv_files:

    df, read_info = read_csv_for_inspection(
        file_path
    )

    if df is None:
        continue

    # Take up to 3 rows
    sample_df = df.head(3)

    sample_records = []

    for _, sample_row in sample_df.iterrows():

        row_dict = {}

        for column in df.columns:

            value = sample_row[column]

            if pd.isna(value):
                value = None

            else:
                value = str(value)

            row_dict[str(column)] = value

        sample_records.append(row_dict)

    sample_evidence_records.append({

        "filename": file_path.name,

        "columns": json.dumps(
            [str(c) for c in df.columns],
            ensure_ascii=False
        ),

        "sample_rows": json.dumps(
            sample_records,
            ensure_ascii=False
        )
    })


sample_evidence_df = pd.DataFrame(
    sample_evidence_records
)

print("=" * 70)
print("COMPACT SAMPLE EVIDENCE")
print("=" * 70)

print(
    f"Datasets with sample evidence: "
    f"{len(sample_evidence_df)}"
)

display(
    sample_evidence_df.head(10)
)

COMPACT SAMPLE EVIDENCE
Datasets with sample evidence: 76


,filename,columns,sample_rows
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,"[""STATE/UT"", ""DISTRICT"", ""YEAR"", ""MURDER"", ""ATTEMPT TO MURDER"", ""CULPABLE HOMICIDE NOT AMOUNTING TO MURDER"", ""RAPE"",...","[{""STATE/UT"": ""ANDHRA PRADESH"", ""DISTRICT"": ""ADILABAD"", ""YEAR"": ""2001"", ""MURDER"": ""101"", ""ATTEMPT TO MURDER"": ""60"", ..."
1,01_District_wise_crimes_committed_IPC_2013.csv,"[""STATE/UT"", ""DISTRICT"", ""YEAR"", ""MURDER"", ""ATTEMPT TO MURDER"", ""CULPABLE HOMICIDE NOT AMOUNTING TO MURDER"", ""RAPE"",...","[{""STATE/UT"": ""Andhra Pradesh"", ""DISTRICT"": ""ADILABAD"", ""YEAR"": ""2013"", ""MURDER"": ""96"", ""ATTEMPT TO MURDER"": ""72"", ""..."
2,01_District_wise_crimes_committed_IPC_2014.csv,"[""States/UTs"", ""District"", ""Year"", ""Murder"", ""Attempt to commit Murder"", ""Culpable Homicide not amounting to Murder""...","[{""States/UTs"": ""Andhra Pradesh"", ""District"": ""Anantapur"", ""Year"": ""2014"", ""Murder"": ""134"", ""Attempt to commit Murde..."
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,"[""STATE/UT"", ""DISTRICT"", ""Year"", ""Murder"", ""Rape"", ""Kidnapping and Abduction"", ""Dacoity"", ""Robbery"", ""Arson"", ""Hurt""...","[{""STATE/UT"": ""ANDHRA PRADESH"", ""DISTRICT"": ""ADILABAD"", ""Year"": ""2001"", ""Murder"": ""0"", ""Rape"": ""1"", ""Kidnapping and ..."
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,"[""STATE/UT"", ""DISTRICT"", ""Year"", ""Murder"", ""Rape"", ""Kidnapping and Abduction"", ""Dacoity"", ""Robbery"", ""Arson"", ""Hurt""...","[{""STATE/UT"": ""Andhra Pradesh"", ""DISTRICT"": ""ADILABAD"", ""Year"": ""2013"", ""Murder"": ""2"", ""Rape"": ""3"", ""Kidnapping and ..."
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,"[""States/UTs"", ""District"", ""Year"", ""Protection of Civil Rights Act, 1955"", ""POA_Murder"", ""POA_Attempt to commit Murd...","[{""States/UTs"": ""Andhra Pradesh"", ""District"": ""Anantapur"", ""Year"": ""2014"", ""Protection of Civil Rights Act, 1955"": ""..."
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,"[""STATE/UT"", ""DISTRICT"", ""Year"", ""Murder"", ""Rape"", ""Kidnapping Abduction"", ""Dacoity"", ""Robbery"", ""Arson"", ""Hurt"", ""P...","[{""STATE/UT"": ""ANDHRA PRADESH"", ""DISTRICT"": ""ADILABAD"", ""Year"": ""2001"", ""Murder"": ""0"", ""Rape"": ""1"", ""Kidnapping Abdu..."
7,02_District_wise_crimes_committed_against_ST_2013.csv,"[""STATE/UT"", ""DISTRICT"", ""Year"", ""Murder"", ""Rape"", ""Kidnapping Abduction"", ""Dacoity"", ""Robbery"", ""Arson"", ""Hurt"", ""P...","[{""STATE/UT"": ""Andhra Pradesh"", ""DISTRICT"": ""ADILABAD"", ""Year"": ""2013"", ""Murder"": ""0"", ""Rape"": ""7"", ""Kidnapping Abdu..."
8,02_District_wise_crimes_committed_against_ST_2014.csv,"[""States/UTs"", ""District"", ""Year"", ""Protection of Civil Rights Act, 1955"", ""POA_Murder"", ""POA_Attempt to commit Murd...","[{""States/UTs"": ""Andhra Pradesh"", ""District"": ""Anantapur"", ""Year"": ""2014"", ""Protection of Civil Rights Act, 1955"": ""..."
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,"[""STATE/UT"", ""DISTRICT"", ""Year"", ""Murder"", ""Rape"", ""Kidnapping and Abduction"", ""Foeticide"", ""Abetment of suicide"", ""...","[{""STATE/UT"": ""ANDHRA PRADESH"", ""DISTRICT"": ""ADILABAD"", ""Year"": ""2001"", ""Murder"": ""0.0"", ""Rape"": ""0.0"", ""Kidnapping ..."


In [26]:
# ============================================================
# CELL 25 — INSPECT FILES WITH PARSING PROBLEMS
# ============================================================

parsing_problem_names = (
    inventory_df.loc[
        inventory_df["parsing_issue"] == True,
        "filename"
    ]
    .tolist()
)

print("=" * 70)
print("FILES WITH PARSING PROBLEMS")
print("=" * 70)

print(
    f"Number of affected files: "
    f"{len(parsing_problem_names)}"
)

for filename in parsing_problem_names:

    print("\n" + "-" * 70)
    print(filename)
    print("-" * 70)

    row = inventory_df[
        inventory_df["filename"] == filename
    ].iloc[0]

    print(
        f"Rows retained by inspection parser: "
        f"{row['row_count']}"
    )

    print(
        f"Reported malformed rows: "
        f"{row['bad_rows']}"
    )

    print(
        f"Encoding used: "
        f"{row['encoding']}"
    )

    print(
        f"Parser used: "
        f"{row['parser']}"
    )

    print(
        f"Status: "
        f"{row['read_status']}"
    )

FILES WITH PARSING PROBLEMS
Number of affected files: 3

----------------------------------------------------------------------
36_Police_housing.csv
----------------------------------------------------------------------
Rows retained by inspection parser: 696
Reported malformed rows: 348
Encoding used: utf-8
Parser used: python
Status: success_with_parsing_issues

----------------------------------------------------------------------
42_Cases_under_crime_against_women.csv
----------------------------------------------------------------------
Rows retained by inspection parser: 2765
Reported malformed rows: 1400
Encoding used: utf-8
Parser used: python
Status: success_with_parsing_issues

----------------------------------------------------------------------
43_Arrests_under_crime_against_women.csv
----------------------------------------------------------------------
Rows retained by inspection parser: 2765
Reported malformed rows: 1400
Encoding used: utf-8
Parser used: python
Status:

In [27]:
# ============================================================
# CELL 26 — RAW STRUCTURE CHECK FOR MALFORMED FILES
# ============================================================

import csv

for filename in parsing_problem_names:

    file_path = RAW_DATA_DIR / filename

    print("\n" + "=" * 90)
    print(f"RAW STRUCTURE CHECK: {filename}")
    print("=" * 90)

    # --------------------------------------------------------
    # Read first 15 physical lines exactly as text
    # --------------------------------------------------------

    print("\nFirst 15 physical lines:")
    print("-" * 90)

    with open(
        file_path,
        "r",
        encoding="latin1",
        errors="replace"
    ) as f:

        for line_number in range(15):

            line = f.readline()

            if not line:
                break

            print(
                f"{line_number + 1:03d}: "
                f"{line.rstrip()[:500]}"
            )

    # --------------------------------------------------------
    # Inspect comma-separated field counts
    # --------------------------------------------------------

    print("\nField-count inspection:")
    print("-" * 90)

    field_counts = {}

    with open(
        file_path,
        "r",
        encoding="latin1",
        errors="replace",
        newline=""
    ) as f:

        reader = csv.reader(f)

        for line_number, row in enumerate(
            reader,
            start=1
        ):

            count = len(row)

            field_counts[count] = (
                field_counts.get(count, 0) + 1
            )

            # Stop after enough physical rows for inspection
            if line_number >= 1000:
                break

    print(
        "Observed field-count frequencies "
        "in first 1000 physical rows:"
    )

    for count, frequency in sorted(
        field_counts.items()
    ):
        print(
            f"  {count} fields : "
            f"{frequency} rows"
        )


RAW STRUCTURE CHECK: 36_Police_housing.csv

First 15 physical lines:
------------------------------------------------------------------------------------------
001: ï»¿Area_Name,Year,Group_Name,Sub_Group_Name,PH_Houses_Provided_by_Department,PH_Houses_provided_on_LeaseRentGPRA,PH_Sanctioned_Strength
002: Andaman & Nicobar Islands,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),7,NULL,17
003: Andhra Pradesh,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),102,189,569
004: Arunachal Pradesh,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),81,11,92
005: Assam,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),0,0,531
006: Bihar,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),130,89,222
007: Chandigarh,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),10,0,17
008: Chhattisgarh,2001,PH_Officers (DySP & Above),1. For Officers (Dy.SP & Above),70,168,140
009: Dadra & Nagar Haveli,2001,PH_Officers (DySP & 

In [28]:
# ============================================================
# CELL 27 — SAVE PHASE 3 DATASET PROFILE
# ============================================================

profile_output_path = (
    TABLES_DIR / "dataset_semantic_profile.csv"
)

dataset_profile_df.to_csv(
    profile_output_path,
    index=False,
    encoding="utf-8-sig"
)

sample_evidence_output_path = (
    TABLES_DIR / "dataset_sample_evidence.csv"
)

sample_evidence_df.to_csv(
    sample_evidence_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("PHASE 3 PROFILE REPORTS SAVED")
print("=" * 70)

print("\nSemantic profile:")
print(
    profile_output_path.resolve()
)

print("\nSample evidence:")
print(
    sample_evidence_output_path.resolve()
)

print("\n✓ Raw source CSV files remain untouched.")

PHASE 3 PROFILE REPORTS SAVED

Semantic profile:
D:\Major_Project\Crime_Analysis\outputs\tables\dataset_semantic_profile.csv

Sample evidence:
D:\Major_Project\Crime_Analysis\outputs\tables\dataset_sample_evidence.csv

✓ Raw source CSV files remain untouched.


In [29]:
# ============================================================
# CELL 28 — PHASE 3 STATUS SUMMARY
# ============================================================

print("=" * 70)
print("PHASE 3 STATUS")
print("=" * 70)

print(
    f"\nDatasets inspected          : "
    f"{len(dataset_profile_df)}"
)

print(
    f"Datasets requiring review  : "
    f"{dataset_profile_df['manual_review_required'].sum()}"
)

print(
    f"Files with parsing issues  : "
    f"{len(parsing_problem_names)}"
)

print(
    f"Total source rows observed : "
    f"{total_source_rows:,}"
)

print("\nReports created:")

print(
    f"  {profile_output_path.resolve()}"
)

print(
    f"  {sample_evidence_output_path.resolve()}"
)

print("\n" + "-" * 70)
print("IMPORTANT")
print("-" * 70)

print(
    "No final geographic, temporal, or subject classification "
    "has been assigned automatically."
)

print(
    "No common schema has been created."
)

print(
    "No datasets have been integrated."
)

print(
    "No raw source files have been modified."
)

print("\n✓ Phase 3 evidence collection complete.")

PHASE 3 STATUS

Datasets inspected          : 76
Datasets requiring review  : 76
Files with parsing issues  : 3
Total source rows observed : 147,753

Reports created:
  D:\Major_Project\Crime_Analysis\outputs\tables\dataset_semantic_profile.csv
  D:\Major_Project\Crime_Analysis\outputs\tables\dataset_sample_evidence.csv

----------------------------------------------------------------------
IMPORTANT
----------------------------------------------------------------------
No final geographic, temporal, or subject classification has been assigned automatically.
No common schema has been created.
No datasets have been integrated.
No raw source files have been modified.

✓ Phase 3 evidence collection complete.


In [30]:
# ============================================================
# CELL 29 — DATASET FAMILY CLASSIFICATION
# ============================================================

def classify_dataset_family(filename):
    """
    Candidate family classification based primarily on the
    dataset filename.

    This is a structural grouping step, NOT a final semantic
    validation of the underlying data.
    """

    name = filename.lower()

    # --------------------------------------------------------
    # District-wise crime datasets
    # --------------------------------------------------------

    if "district_wise" in name:
        return "DISTRICT_WISE_CRIME"

    # --------------------------------------------------------
    # Persons arrested / disposal
    # --------------------------------------------------------

    if (
        "persons_arrested" in name
        or "person_arrested" in name
        or "arrests_under" in name
    ):
        return "ARREST_DISPOSAL"

    # --------------------------------------------------------
    # Juvenile datasets
    # --------------------------------------------------------

    if (
        "juvenile" in name
        or "juveniles" in name
    ):
        return "JUVENILE"

    # --------------------------------------------------------
    # Property / vehicle
    # --------------------------------------------------------

    if (
        "property" in name
        or "auto_theft" in name
        or "serious_fraud" in name
        or "property_taken" in name
    ):
        return "PROPERTY_CRIME"

    # --------------------------------------------------------
    # Police personnel / police administration
    # --------------------------------------------------------

    if (
        "police" in name
        or "home_guards" in name
        or "auxilliary" in name
    ):
        return "POLICE_ADMINISTRATION"

    # --------------------------------------------------------
    # Victim demographic datasets
    # --------------------------------------------------------

    if (
        "victim" in name
        or "offenders_known" in name
        or "motive_or_cause" in name
        or "specific_purpose" in name
        or "use_of_fire" in name
    ):
        return "VICTIM_OFFENDER_CHARACTERISTICS"

    # --------------------------------------------------------
    # Court / trial / justice
    # --------------------------------------------------------

    if (
        "trial" in name
        or "court" in name
        or "custodial" in name
        or "escape" in name
        or "cases_under_crime" in name
    ):
        return "JUSTICE_COURT"

    # --------------------------------------------------------
    # Anti-corruption
    # --------------------------------------------------------

    if (
        "corruption" in name
        or "corruprion" in name
    ):
        return "ANTI_CORRUPTION"

    # --------------------------------------------------------
    # Police complaints / human rights
    # --------------------------------------------------------

    if (
        "complaint" in name
        or "human_rights" in name
    ):
        return "POLICE_ACCOUNTABILITY"

    # --------------------------------------------------------
    # Crime by place
    # --------------------------------------------------------

    if "place_of_occurrence" in name:
        return "CRIME_PLACE_OF_OCCURRENCE"

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    return "OTHER_SPECIALIZED"


dataset_profile_df["dataset_family"] = (
    dataset_profile_df["filename"]
    .apply(classify_dataset_family)
)

print("=" * 70)
print("DATASET FAMILY CLASSIFICATION")
print("=" * 70)

family_counts = (
    dataset_profile_df["dataset_family"]
    .value_counts()
    .sort_index()
)

display(
    family_counts.to_frame("file_count")
)

DATASET FAMILY CLASSIFICATION


,file_count
dataset_family,
ANTI_CORRUPTION,2
ARREST_DISPOSAL,17
CRIME_PLACE_OF_OCCURRENCE,3
DISTRICT_WISE_CRIME,14
JUSTICE_COURT,8
JUVENILE,7
OTHER_SPECIALIZED,1
POLICE_ADMINISTRATION,11
PROPERTY_CRIME,6


In [31]:
# ============================================================
# CELL 30 — FILE-TO-FAMILY MAPPING
# ============================================================

family_mapping_df = (
    dataset_profile_df[
        [
            "filename",
            "row_count",
            "column_count",
            "dataset_family"
        ]
    ]
    .sort_values(
        [
            "dataset_family",
            "filename"
        ]
    )
    .reset_index(drop=True)
)

display(
    family_mapping_df
)

,filename,row_count,column_count,dataset_family
0,23_Anti_corruprion_cases.csv,346,29,ANTI_CORRUPTION
1,24_Anti_corruption_arrests.csv,347,24,ANTI_CORRUPTION
2,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2012.csv,494,14,ARREST_DISPOSAL
3,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2013.csv,494,14,ARREST_DISPOSAL
4,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv,2028,57,ARREST_DISPOSAL
...,...,...,...,...
71,21_Offenders_known_to_the_victim.csv,350,7,VICTIM_OFFENDER_CHARACTERISTICS
72,32_Murder_victim_age_sex.csv,1018,11,VICTIM_OFFENDER_CHARACTERISTICS
73,33_CH_not_murder_victim_age_sex.csv,936,10,VICTIM_OFFENDER_CHARACTERISTICS
74,34_Use_of_fire_arms_in_murder_cases.csv,284,5,VICTIM_OFFENDER_CHARACTERISTICS


In [32]:
# ============================================================
# CELL 31 — COMMON DIMENSION DETECTION
# ============================================================

COMMON_DIMENSION_PATTERNS = {
    "year": [
        r"^year$",
        r"year$"
    ],

    "state": [
        r"state",
        r"states",
        r"state/ut",
        r"states/uts"
    ],

    "district": [
        r"district"
    ],

    "crime_head": [
        r"crime head",
        r"crime_head",
        r"crime$"
    ],

    "area": [
        r"area",
        r"region",
        r"location",
        r"place"
    ]
}


def detect_dimension_columns(columns):

    detected = {
        key: []
        for key in COMMON_DIMENSION_PATTERNS
    }

    for column in columns:

        col = str(column).strip().lower()

        for dimension, patterns in (
            COMMON_DIMENSION_PATTERNS.items()
        ):

            for pattern in patterns:

                if re.search(
                    pattern,
                    col
                ):
                    detected[dimension].append(
                        str(column)
                    )
                    break

    return detected

In [33]:
# ============================================================
# CELL 32 — DIMENSION INVENTORY
# ============================================================

dimension_records = []

for file_path in csv_files:

    df, read_info = read_csv_for_inspection(
        file_path
    )

    if df is None:
        continue

    detected = detect_dimension_columns(
        df.columns
    )

    dimension_records.append({

        "filename":
            file_path.name,

        "dataset_family":
            classify_dataset_family(
                file_path.name
            ),

        "has_year":
            bool(detected["year"]),

        "year_columns":
            "; ".join(
                detected["year"]
            ),

        "has_state":
            bool(detected["state"]),

        "state_columns":
            "; ".join(
                detected["state"]
            ),

        "has_district":
            bool(detected["district"]),

        "district_columns":
            "; ".join(
                detected["district"]
            ),

        "has_crime_head":
            bool(detected["crime_head"]),

        "crime_head_columns":
            "; ".join(
                detected["crime_head"]
            ),

        "has_area":
            bool(detected["area"]),

        "area_columns":
            "; ".join(
                detected["area"]
            )
    })


dimension_inventory_df = pd.DataFrame(
    dimension_records
)

display(
    dimension_inventory_df
)

,filename,dataset_family,has_year,year_columns,has_state,state_columns,has_district,district_columns,has_crime_head,crime_head_columns,has_area,area_columns
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,DISTRICT_WISE_CRIME,True,YEAR,True,STATE/UT,True,DISTRICT,False,,False,
1,01_District_wise_crimes_committed_IPC_2013.csv,DISTRICT_WISE_CRIME,True,YEAR,True,STATE/UT,True,DISTRICT,False,,False,
2,01_District_wise_crimes_committed_IPC_2014.csv,DISTRICT_WISE_CRIME,True,Year,True,States/UTs; Offences against State; Other offences against State,True,District,False,,True,"Other places related to work; Places other than 231, 232 & 233"
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,DISTRICT_WISE_CRIME,True,Year,True,STATE/UT,True,DISTRICT,False,,False,
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,DISTRICT_WISE_CRIME,True,Year,True,STATE/UT,True,DISTRICT,False,,False,
...,...,...,...,...,...,...,...,...,...,...,...,...
71,42_Cases_under_crime_against_women.csv,JUSTICE_COURT,True,Year; Cases_Pending_Investigation_from_previous_year; Cases_Pending_Trial_from_the_previous_year,False,,False,,False,,True,Area_Name
72,42_District_wise_crimes_committed_against_women_2001_2012.csv,DISTRICT_WISE_CRIME,True,Year,True,STATE/UT,True,DISTRICT,False,,False,
73,42_District_wise_crimes_committed_against_women_2013.csv,DISTRICT_WISE_CRIME,True,Year,True,STATE/UT,True,DISTRICT,False,,False,
74,42_District_wise_crimes_committed_against_women_2014.csv,DISTRICT_WISE_CRIME,True,Year,True,States/UTs,True,District,False,,True,In places related to work; In other Places


In [34]:
# ============================================================
# CELL 33 — DIMENSION COVERAGE
# ============================================================

dimension_summary = pd.DataFrame({

    "dimension": [
        "Year",
        "State/UT",
        "District",
        "Crime Head",
        "Generic Area/Location"
    ],

    "datasets_containing_dimension": [

        dimension_inventory_df["has_year"].sum(),

        dimension_inventory_df["has_state"].sum(),

        dimension_inventory_df["has_district"].sum(),

        dimension_inventory_df["has_crime_head"].sum(),

        dimension_inventory_df["has_area"].sum()
    ]
})

display(
    dimension_summary
)

,dimension,datasets_containing_dimension
0,Year,72
1,State/UT,36
2,District,14
3,Crime Head,17
4,Generic Area/Location,47


In [35]:
# ============================================================
# CELL 34 — DATASET COMPARABILITY MATRIX
# ============================================================

dimension_columns = [
    "has_year",
    "has_state",
    "has_district",
    "has_crime_head"
]

comparability_records = []

for i in range(
    len(dimension_inventory_df)
):

    for j in range(
        i + 1,
        len(dimension_inventory_df)
    ):

        a = dimension_inventory_df.iloc[i]
        b = dimension_inventory_df.iloc[j]

        shared_dimensions = []

        for dimension in dimension_columns:

            if (
                a[dimension]
                and b[dimension]
            ):
                shared_dimensions.append(
                    dimension.replace(
                        "has_",
                        ""
                    )
                )

        comparability_records.append({

            "dataset_a":
                a["filename"],

            "dataset_b":
                b["filename"],

            "family_a":
                a["dataset_family"],

            "family_b":
                b["dataset_family"],

            "shared_dimensions":
                ", ".join(
                    shared_dimensions
                ),

            "shared_dimension_count":
                len(shared_dimensions)
        })


comparability_df = pd.DataFrame(
    comparability_records
)

# Only display pairs with at least 2 shared dimensions
potential_comparisons = (
    comparability_df[
        comparability_df[
            "shared_dimension_count"
        ] >= 2
    ]
    .sort_values(
        "shared_dimension_count",
        ascending=False
    )
)

print("=" * 70)
print("POTENTIALLY COMPARABLE DATASET PAIRS")
print("=" * 70)

display(
    potential_comparisons.head(100)
)

POTENTIALLY COMPARABLE DATASET PAIRS


,dataset_a,dataset_b,family_a,family_b,shared_dimensions,shared_dimension_count
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,01_District_wise_crimes_committed_IPC_2013.csv,DISTRICT_WISE_CRIME,DISTRICT_WISE_CRIME,"year, state, district",3
846,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2013.csv,07_02_Persons_arrested_by_sex_and_age_group_SLL_2014.csv,ARREST_DISPOSAL,ARREST_DISPOSAL,"year, state, crime_head",3
639,03_District_wise_crimes_committed_against_children_2001_2012.csv,03_District_wise_crimes_committed_against_children_2013.csv,DISTRICT_WISE_CRIME,DISTRICT_WISE_CRIME,"year, state, district",3
701,03_District_wise_crimes_committed_against_children_2001_2012.csv,42_District_wise_crimes_committed_against_women_2001_2012.csv,DISTRICT_WISE_CRIME,DISTRICT_WISE_CRIME,"year, state, district",3
702,03_District_wise_crimes_committed_against_children_2001_2012.csv,42_District_wise_crimes_committed_against_women_2013.csv,DISTRICT_WISE_CRIME,DISTRICT_WISE_CRIME,"year, state, district",3
...,...,...,...,...,...,...
898,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv,04_01_Person_arrested_and_their_disposal_by_police_and_court_SLL_crime_2013.csv,ARREST_DISPOSAL,ARREST_DISPOSAL,"year, state, crime_head",3
899,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv,04_01_Person_arrested_and_their_disposal_by_police_and_court_SLL_crime_2014.csv,ARREST_DISPOSAL,ARREST_DISPOSAL,"year, state, crime_head",3
900,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv,04_02_Person_arrested_and_their_disposal_by_police_and_court_IPC_crime_2012.csv,ARREST_DISPOSAL,ARREST_DISPOSAL,"year, state, crime_head",3
901,03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv,04_02_Person_arrested_and_their_disposal_by_police_and_court_IPC_crime_2013.csv,ARREST_DISPOSAL,ARREST_DISPOSAL,"year, state, crime_head",3


In [36]:
# ============================================================
# CELL 35 — GEOGRAPHIC INTEGRATION CANDIDATES
# ============================================================

geographic_candidates = (
    dimension_inventory_df[
        (
            dimension_inventory_df["has_year"]
        )
        &
        (
            dimension_inventory_df["has_state"]
        )
    ]
    .copy()
)

print("=" * 70)
print("YEAR + STATE/UT INTEGRATION CANDIDATES")
print("=" * 70)

display(
    geographic_candidates[
        [
            "filename",
            "dataset_family",
            "year_columns",
            "state_columns",
            "district_columns",
            "crime_head_columns"
        ]
    ]
)

YEAR + STATE/UT INTEGRATION CANDIDATES


,filename,dataset_family,year_columns,state_columns,district_columns,crime_head_columns
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,DISTRICT_WISE_CRIME,YEAR,STATE/UT,DISTRICT,
1,01_District_wise_crimes_committed_IPC_2013.csv,DISTRICT_WISE_CRIME,YEAR,STATE/UT,DISTRICT,
2,01_District_wise_crimes_committed_IPC_2014.csv,DISTRICT_WISE_CRIME,Year,States/UTs; Offences against State; Other offences against State,District,
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,DISTRICT_WISE_CRIME,Year,States/UTs,District,
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
7,02_District_wise_crimes_committed_against_ST_2013.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
8,02_District_wise_crimes_committed_against_ST_2014.csv,DISTRICT_WISE_CRIME,Year,States/UTs,District,
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,


In [37]:
# ============================================================
# CELL 36 — DISTRICT-LEVEL CANDIDATES
# ============================================================

district_candidates = (
    dimension_inventory_df[
        (
            dimension_inventory_df["has_year"]
        )
        &
        (
            dimension_inventory_df["has_state"]
        )
        &
        (
            dimension_inventory_df["has_district"]
        )
    ]
    .copy()
)

print("=" * 70)
print("DISTRICT-LEVEL DATASET CANDIDATES")
print("=" * 70)

display(
    district_candidates[
        [
            "filename",
            "dataset_family",
            "year_columns",
            "state_columns",
            "district_columns",
            "crime_head_columns"
        ]
    ]
)

DISTRICT-LEVEL DATASET CANDIDATES


,filename,dataset_family,year_columns,state_columns,district_columns,crime_head_columns
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,DISTRICT_WISE_CRIME,YEAR,STATE/UT,DISTRICT,
1,01_District_wise_crimes_committed_IPC_2013.csv,DISTRICT_WISE_CRIME,YEAR,STATE/UT,DISTRICT,
2,01_District_wise_crimes_committed_IPC_2014.csv,DISTRICT_WISE_CRIME,Year,States/UTs; Offences against State; Other offences against State,District,
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,DISTRICT_WISE_CRIME,Year,States/UTs,District,
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
7,02_District_wise_crimes_committed_against_ST_2013.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,
8,02_District_wise_crimes_committed_against_ST_2014.csv,DISTRICT_WISE_CRIME,Year,States/UTs,District,
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,DISTRICT_WISE_CRIME,Year,STATE/UT,DISTRICT,


In [38]:
# ============================================================
# CELL 37 — PRELIMINARY INTEGRATION DECISION
# ============================================================

def preliminary_integration_decision(row):

    family = row["dataset_family"]

    filename = row["filename"].lower()

    # --------------------------------------------------------
    # District-wise crime data
    # --------------------------------------------------------

    if family == "DISTRICT_WISE_CRIME":

        return (
            "DISTRICT_CRIME_CORE",
            "Candidate for district-level crime analysis"
        )

    # --------------------------------------------------------
    # Arrest/disposal
    # --------------------------------------------------------

    if family == "ARREST_DISPOSAL":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Different row grain: arrest/disposal statistics"
        )

    # --------------------------------------------------------
    # Property
    # --------------------------------------------------------

    if family == "PROPERTY_CRIME":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Property/value measures have different semantics"
        )

    # --------------------------------------------------------
    # Juvenile
    # --------------------------------------------------------

    if family == "JUVENILE":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Juvenile-specific population and measures"
        )

    # --------------------------------------------------------
    # Police administration
    # --------------------------------------------------------

    if family == "POLICE_ADMINISTRATION":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Police personnel/administration statistics"
        )

    # --------------------------------------------------------
    # Victim/offender characteristics
    # --------------------------------------------------------

    if family == "VICTIM_OFFENDER_CHARACTERISTICS":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Victim/offender characteristic statistics"
        )

    # --------------------------------------------------------
    # Justice / courts
    # --------------------------------------------------------

    if family == "JUSTICE_COURT":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Justice/court process statistics"
        )

    # --------------------------------------------------------
    # Anti-corruption
    # --------------------------------------------------------

    if family == "ANTI_CORRUPTION":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Specialized anti-corruption statistics"
        )

    # --------------------------------------------------------
    # Police accountability
    # --------------------------------------------------------

    if family == "POLICE_ACCOUNTABILITY":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Police complaints/accountability statistics"
        )

    # --------------------------------------------------------
    # Crime place
    # --------------------------------------------------------

    if family == "CRIME_PLACE_OF_OCCURRENCE":

        return (
            "SEPARATE_ANALYTICAL_TABLE",
            "Place-of-occurrence structure differs"
        )

    # --------------------------------------------------------
    # Everything else
    # --------------------------------------------------------

    return (
        "MANUAL_REVIEW",
        "Specialized dataset requiring semantic review"
    )


decision_values = (
    dataset_profile_df
    .apply(
        preliminary_integration_decision,
        axis=1
    )
)

dataset_profile_df[
    [
        "integration_status",
        "integration_reason"
    ]
] = pd.DataFrame(
    decision_values.tolist(),
    index=dataset_profile_df.index
)

display(
    dataset_profile_df[
        [
            "filename",
            "dataset_family",
            "integration_status",
            "integration_reason"
        ]
    ]
)

,filename,dataset_family,integration_status,integration_reason
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
1,01_District_wise_crimes_committed_IPC_2013.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
2,01_District_wise_crimes_committed_IPC_2014.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
...,...,...,...,...
71,42_Cases_under_crime_against_women.csv,JUSTICE_COURT,SEPARATE_ANALYTICAL_TABLE,Justice/court process statistics
72,42_District_wise_crimes_committed_against_women_2001_2012.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
73,42_District_wise_crimes_committed_against_women_2013.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis
74,42_District_wise_crimes_committed_against_women_2014.csv,DISTRICT_WISE_CRIME,DISTRICT_CRIME_CORE,Candidate for district-level crime analysis


In [39]:
# ============================================================
# CELL 38 — INTEGRATION DECISION SUMMARY
# ============================================================

decision_summary = (
    dataset_profile_df[
        "integration_status"
    ]
    .value_counts()
    .to_frame(
        "file_count"
    )
)

display(
    decision_summary
)

,file_count
integration_status,
SEPARATE_ANALYTICAL_TABLE,61
DISTRICT_CRIME_CORE,14
MANUAL_REVIEW,1


In [40]:
# ============================================================
# CELL 39 — SAVE PHASE 4 AUDIT
# ============================================================

phase4_output = (
    TABLES_DIR /
    "phase4_dataset_integration_audit.csv"
)

dataset_profile_df.to_csv(
    phase4_output,
    index=False,
    encoding="utf-8-sig"
)

dimension_output = (
    TABLES_DIR /
    "phase4_dimension_inventory.csv"
)

dimension_inventory_df.to_csv(
    dimension_output,
    index=False,
    encoding="utf-8-sig"
)

comparison_output = (
    TABLES_DIR /
    "phase4_dataset_comparability.csv"
)

comparability_df.to_csv(
    comparison_output,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("PHASE 4 AUDIT FILES CREATED")
print("=" * 70)

print(
    f"\n{phase4_output.resolve()}"
)

print(
    f"\n{dimension_output.resolve()}"
)

print(
    f"\n{comparison_output.resolve()}"
)

PHASE 4 AUDIT FILES CREATED

D:\Major_Project\Crime_Analysis\outputs\tables\phase4_dataset_integration_audit.csv

D:\Major_Project\Crime_Analysis\outputs\tables\phase4_dimension_inventory.csv

D:\Major_Project\Crime_Analysis\outputs\tables\phase4_dataset_comparability.csv


In [41]:
# ============================================================
# CELL 40 — SELECT DISTRICT CRIME DATASETS
# ============================================================

district_files = [
    Path(RAW_DATA_DIR) / filename
    for filename in dataset_profile_df.loc[
        dataset_profile_df["integration_status"]
        == "DISTRICT_CRIME_CORE",
        "filename"
    ]
]

print("=" * 70)
print("DISTRICT CRIME DATASETS SELECTED")
print("=" * 70)

print(f"\nTotal selected: {len(district_files)}")

for i, path in enumerate(
    district_files,
    start=1
):
    print(f"{i:02d}. {path.name}")

DISTRICT CRIME DATASETS SELECTED

Total selected: 14
01. 01_District_wise_crimes_committed_IPC_2001_2012.csv
02. 01_District_wise_crimes_committed_IPC_2013.csv
03. 01_District_wise_crimes_committed_IPC_2014.csv
04. 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
05. 02_01_District_wise_crimes_committed_against_SC_2013.csv
06. 02_01_District_wise_crimes_committed_against_SC_2014.csv
07. 02_District_wise_crimes_committed_against_ST_2001_2012.csv
08. 02_District_wise_crimes_committed_against_ST_2013.csv
09. 02_District_wise_crimes_committed_against_ST_2014.csv
10. 03_District_wise_crimes_committed_against_children_2001_2012.csv
11. 03_District_wise_crimes_committed_against_children_2013.csv
12. 42_District_wise_crimes_committed_against_women_2001_2012.csv
13. 42_District_wise_crimes_committed_against_women_2013.csv
14. 42_District_wise_crimes_committed_against_women_2014.csv


In [42]:
# ============================================================
# CELL 41 — DISTRICT DATASET COLUMN COMPARISON
# ============================================================

district_structure = []

for path in district_files:

    df, read_info = read_csv_for_inspection(
        path
    )

    if df is None:
        continue

    district_structure.append({

        "filename":
            path.name,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "state_column":
            next(
                (
                    c for c in df.columns
                    if str(c).strip().upper()
                    in [
                        "STATE/UT",
                        "STATES/UTS"
                    ]
                ),
                None
            ),

        "district_column":
            next(
                (
                    c for c in df.columns
                    if str(c).strip().upper()
                    == "DISTRICT"
                ),
                None
            ),

        "year_column":
            next(
                (
                    c for c in df.columns
                    if str(c).strip().upper()
                    == "YEAR"
                ),
                None
            ),

        "read_status":
            read_info.get(
                "status",
                "unknown"
            )
    })


district_structure_df = pd.DataFrame(
    district_structure
)

display(
    district_structure_df
)

,filename,rows,columns,state_column,district_column,year_column,read_status
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,9017,33,STATE/UT,DISTRICT,YEAR,success
1,01_District_wise_crimes_committed_IPC_2013.csv,823,33,STATE/UT,DISTRICT,YEAR,success
2,01_District_wise_crimes_committed_IPC_2014.csv,838,91,States/UTs,District,Year,success
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,9018,13,STATE/UT,DISTRICT,Year,success
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,823,13,STATE/UT,DISTRICT,Year,success
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,837,66,States/UTs,District,Year,success
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,9018,13,STATE/UT,DISTRICT,Year,success
7,02_District_wise_crimes_committed_against_ST_2013.csv,823,13,STATE/UT,DISTRICT,Year,success
8,02_District_wise_crimes_committed_against_ST_2014.csv,837,66,States/UTs,District,Year,success
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,9015,15,STATE/UT,DISTRICT,Year,success


In [43]:
# ============================================================
# CELL 42 — NORMALIZE COMMON DIMENSIONS
# ============================================================

def normalize_dimension_columns(df):

    rename_map = {}

    for column in df.columns:

        normalized = (
            str(column)
            .strip()
            .upper()
        )

        if normalized in [
            "STATE/UT",
            "STATES/UTS"
        ]:
            rename_map[column] = "STATE"

        elif normalized == "DISTRICT":
            rename_map[column] = "DISTRICT"

        elif normalized == "YEAR":
            rename_map[column] = "YEAR"

    return df.rename(
        columns=rename_map
    )


normalized_district_data = {}

for path in district_files:

    df, read_info = read_csv_for_inspection(
        path
    )

    if df is None:
        continue

    df = normalize_dimension_columns(
        df
    )

    normalized_district_data[
        path.name
    ] = df

print(
    f"Normalized datasets: "
    f"{len(normalized_district_data)}"
)

Normalized datasets: 14


In [44]:
# ============================================================
# CELL 43 — CLEAN COMMON DIMENSION VALUES
# ============================================================

for filename, df in normalized_district_data.items():

    for column in [
        "STATE",
        "DISTRICT"
    ]:

        if column in df.columns:

            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
            )

            df[column] = (
                df[column]
                .replace(
                    {
                        "":
                            pd.NA,
                        "NAN":
                            pd.NA,
                        "NULL":
                            pd.NA,
                        "N/A":
                            pd.NA
                    }
                )
            )

    if "YEAR" in df.columns:

        df["YEAR"] = pd.to_numeric(
            df["YEAR"],
            errors="coerce"
        )

print("Common dimensions cleaned.")

Common dimensions cleaned.


In [45]:
# ============================================================
# CELL 44 — ASSIGN CRIME GROUP
# ============================================================

def assign_crime_group(filename):

    name = filename.lower()

    if "_ipc_" in name:
        return "IPC"

    if "_sc_" in name:
        return "SC"

    if "_st_" in name:
        return "ST"

    if "children" in name:
        return "CHILDREN"

    if "against_women" in name:
        return "WOMEN"

    return "OTHER"


for filename, df in normalized_district_data.items():

    df["CRIME_GROUP"] = assign_crime_group(
        filename
    )

    df["SOURCE_FILE"] = filename

In [46]:
# ============================================================
# CELL 45 — CRIME GROUP SUMMARY
# ============================================================

group_summary = []

for filename, df in normalized_district_data.items():

    group_summary.append({

        "filename":
            filename,

        "crime_group":
            df["CRIME_GROUP"].iloc[0],

        "rows":
            len(df),

        "columns":
            len(df.columns)
    })


group_summary_df = pd.DataFrame(
    group_summary
)

display(
    group_summary_df
)

,filename,crime_group,rows,columns
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,9017,35
1,01_District_wise_crimes_committed_IPC_2013.csv,IPC,823,35
2,01_District_wise_crimes_committed_IPC_2014.csv,IPC,838,93
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,SC,9018,15
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,SC,823,15
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,SC,837,68
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,ST,9018,15
7,02_District_wise_crimes_committed_against_ST_2013.csv,ST,823,15
8,02_District_wise_crimes_committed_against_ST_2014.csv,ST,837,68
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN,9015,17


In [47]:
# ============================================================
# CELL 46 — YEAR COVERAGE
# ============================================================

year_records = []

for filename, df in normalized_district_data.items():

    if "YEAR" not in df.columns:
        continue

    year_records.append({

        "filename":
            filename,

        "min_year":
            df["YEAR"].min(),

        "max_year":
            df["YEAR"].max(),

        "unique_years":
            df["YEAR"].nunique()
    })


year_coverage_df = pd.DataFrame(
    year_records
)

display(
    year_coverage_df
)

,filename,min_year,max_year,unique_years
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,2001,2012,12
1,01_District_wise_crimes_committed_IPC_2013.csv,2013,2013,1
2,01_District_wise_crimes_committed_IPC_2014.csv,2014,2014,1
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,2001,2012,12
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,2013,2013,1
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,2014,2014,1
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,2001,2012,12
7,02_District_wise_crimes_committed_against_ST_2013.csv,2013,2013,1
8,02_District_wise_crimes_committed_against_ST_2014.csv,2014,2014,1
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,2001,2012,12


In [48]:
# ============================================================
# CELL 47 — GEOGRAPHIC QUALITY CHECK
# ============================================================

geo_records = []

for filename, df in normalized_district_data.items():

    state_missing = (
        df["STATE"].isna().sum()
        if "STATE" in df.columns
        else len(df)
    )

    district_missing = (
        df["DISTRICT"].isna().sum()
        if "DISTRICT" in df.columns
        else len(df)
    )

    year_missing = (
        df["YEAR"].isna().sum()
        if "YEAR" in df.columns
        else len(df)
    )

    geo_records.append({

        "filename":
            filename,

        "rows":
            len(df),

        "missing_state":
            state_missing,

        "missing_district":
            district_missing,

        "missing_year":
            year_missing,

        "state_missing_pct":
            round(
                100 * state_missing / len(df),
                2
            ),

        "district_missing_pct":
            round(
                100 * district_missing / len(df),
                2
            ),

        "year_missing_pct":
            round(
                100 * year_missing / len(df),
                2
            )
    })


geo_quality_df = pd.DataFrame(
    geo_records
)

display(
    geo_quality_df
)

,filename,rows,missing_state,missing_district,missing_year,state_missing_pct,district_missing_pct,year_missing_pct
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,9017,0,0,0,0.0,0.0,0.0
1,01_District_wise_crimes_committed_IPC_2013.csv,823,0,0,0,0.0,0.0,0.0
2,01_District_wise_crimes_committed_IPC_2014.csv,838,0,0,0,0.0,0.0,0.0
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,9018,0,0,0,0.0,0.0,0.0
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,823,0,0,0,0.0,0.0,0.0
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,837,0,0,0,0.0,0.0,0.0
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,9018,0,0,0,0.0,0.0,0.0
7,02_District_wise_crimes_committed_against_ST_2013.csv,823,0,0,0,0.0,0.0,0.0
8,02_District_wise_crimes_committed_against_ST_2014.csv,837,0,0,0,0.0,0.0,0.0
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,9015,0,0,0,0.0,0.0,0.0


In [49]:
# ============================================================
# CELL 48 — DUPLICATE GEOGRAPHIC-TEMPORAL KEYS
# ============================================================

duplicate_records = []

for filename, df in normalized_district_data.items():

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    existing_keys = [
        c for c in key_columns
        if c in df.columns
    ]

    duplicate_count = (
        df.duplicated(
            subset=existing_keys,
            keep=False
        )
        .sum()
    )

    duplicate_records.append({

        "filename":
            filename,

        "key_columns":
            ", ".join(existing_keys),

        "duplicate_rows":
            int(duplicate_count),

        "duplicate_pct":
            round(
                100 * duplicate_count / len(df),
                2
            )
    })


duplicate_key_df = pd.DataFrame(
    duplicate_records
)

display(
    duplicate_key_df
)

,filename,key_columns,duplicate_rows,duplicate_pct
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,"STATE, DISTRICT, YEAR",2,0.02
1,01_District_wise_crimes_committed_IPC_2013.csv,"STATE, DISTRICT, YEAR",0,0.00
2,01_District_wise_crimes_committed_IPC_2014.csv,"STATE, DISTRICT, YEAR",0,0.00
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,"STATE, DISTRICT, YEAR",4,0.04
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,"STATE, DISTRICT, YEAR",0,0.00
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,"STATE, DISTRICT, YEAR",0,0.00
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,"STATE, DISTRICT, YEAR",4,0.04
7,02_District_wise_crimes_committed_against_ST_2013.csv,"STATE, DISTRICT, YEAR",0,0.00
8,02_District_wise_crimes_committed_against_ST_2014.csv,"STATE, DISTRICT, YEAR",0,0.00
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,"STATE, DISTRICT, YEAR",2,0.02


In [50]:
# ============================================================
# CELL 49 — SAMPLE EACH CRIME GROUP
# ============================================================

for group in sorted(
    group_summary_df[
        "crime_group"
    ].unique()
):

    matching = [
        filename
        for filename, df
        in normalized_district_data.items()
        if df["CRIME_GROUP"].iloc[0] == group
    ]

    if not matching:
        continue

    filename = matching[0]

    print("\n" + "=" * 70)
    print(f"GROUP: {group}")
    print(f"FILE : {filename}")
    print("=" * 70)

    display(
        normalized_district_data[
            filename
        ].head(5)
    )


GROUP: CHILDREN
FILE : 03_District_wise_crimes_committed_against_children_2001_2012.csv


,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping and Abduction,Foeticide,Abetment of suicide,Exposure and abandonment,Procuration of minor girls,Buying of girls for prostitution,Selling of girls for prostitution,Prohibition of child marriage act,Other Crimes,Total,CRIME_GROUP,SOURCE_FILE
0,ANDHRA PRADESH,ADILABAD,2001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
1,ANDHRA PRADESH,ANANTAPUR,2001,19.0,12.0,29.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,66,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
2,ANDHRA PRADESH,CHITTOOR,2001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
3,ANDHRA PRADESH,CUDDAPAH,2001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
4,ANDHRA PRADESH,EAST GODAVARI,2001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv



GROUP: IPC
FILE : 01_District_wise_crimes_committed_IPC_2001_2012.csv


,STATE,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES,CRIME_GROUP,SOURCE_FILE
0,ANDHRA PRADESH,ADILABAD,2001,101,60,17,50,0,50,46,30,16,9,0,41,198,199,22,177,78,16,104,1,30,1131,16,149,34,175,0,181,1518,4154,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
1,ANDHRA PRADESH,ANANTAPUR,2001,151,125,1,23,0,23,53,30,23,8,0,16,191,366,57,309,168,11,65,8,69,1543,7,118,24,154,0,270,754,4125,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
2,ANDHRA PRADESH,CHITTOOR,2001,101,57,2,27,0,27,59,34,25,4,0,14,237,723,164,559,156,33,209,9,38,2088,14,112,83,186,0,404,1262,5818,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
3,ANDHRA PRADESH,CUDDAPAH,2001,80,53,1,20,0,20,25,20,5,1,0,4,98,173,36,137,164,12,37,2,23,795,17,126,38,57,0,233,1181,3140,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
4,ANDHRA PRADESH,EAST GODAVARI,2001,82,67,1,23,0,23,49,26,23,4,0,25,437,1021,150,871,70,50,220,3,41,1244,12,109,58,247,0,431,2313,6507,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv



GROUP: SC
FILE : 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv


,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping and Abduction,Dacoity,Robbery,Arson,Hurt,Prevention of atrocities (POA) Act,Protection of Civil Rights (PCR) Act,Other Crimes Against SCs,CRIME_GROUP,SOURCE_FILE
0,ANDHRA PRADESH,ADILABAD,2001,0,1,4,0,0,0,3,0,15,32,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
1,ANDHRA PRADESH,ANANTAPUR,2001,0,4,0,0,0,0,49,21,0,53,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
2,ANDHRA PRADESH,CHITTOOR,2001,3,3,0,0,0,0,38,36,0,34,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
3,ANDHRA PRADESH,CUDDAPAH,2001,0,3,0,0,0,0,20,52,0,25,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
4,ANDHRA PRADESH,EAST GODAVARI,2001,1,3,0,0,0,0,3,12,63,7,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv



GROUP: ST
FILE : 02_District_wise_crimes_committed_against_ST_2001_2012.csv


,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping Abduction,Dacoity,Robbery,Arson,Hurt,Protection of Civil Rights (PCR) Act,Prevention of atrocities (POA) Act,Other Crimes Against STs,CRIME_GROUP,SOURCE_FILE
0,ANDHRA PRADESH,ADILABAD,2001,0,1,2,0,0,0,2,0,0,13,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
1,ANDHRA PRADESH,ANANTAPUR,2001,0,0,0,0,0,0,7,0,1,6,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
2,ANDHRA PRADESH,CHITTOOR,2001,0,0,0,0,0,0,2,0,0,0,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
3,ANDHRA PRADESH,CUDDAPAH,2001,0,0,0,0,0,0,2,0,2,0,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
4,ANDHRA PRADESH,EAST GODAVARI,2001,0,0,0,0,0,0,0,0,0,14,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv



GROUP: WOMEN
FILE : 42_District_wise_crimes_committed_against_women_2001_2012.csv


,STATE,DISTRICT,YEAR,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls,CRIME_GROUP,SOURCE_FILE
0,ANDHRA PRADESH,ADILABAD,2001,50,30,16,149,34,175,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
1,ANDHRA PRADESH,ANANTAPUR,2001,23,30,7,118,24,154,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
2,ANDHRA PRADESH,CHITTOOR,2001,27,34,14,112,83,186,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
3,ANDHRA PRADESH,CUDDAPAH,2001,20,20,17,126,38,57,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
4,ANDHRA PRADESH,EAST GODAVARI,2001,23,26,12,109,58,247,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv


In [51]:
# ============================================================
# CELL 50 — SAVE NORMALIZED DISTRICT DATA
# ============================================================

normalized_dir = (
    PROCESSED_DATA_DIR /
    "district_normalized"
)

normalized_dir.mkdir(
    parents=True,
    exist_ok=True
)

for filename, df in normalized_district_data.items():

    output_path = (
        normalized_dir /
        filename
    )

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

print("=" * 70)
print("NORMALIZED DISTRICT DATA SAVED")
print("=" * 70)

print(
    normalized_dir.resolve()
)

NORMALIZED DISTRICT DATA SAVED
D:\Major_Project\Crime_Analysis\data\processed\district_normalized


In [52]:
# ============================================================
# CELL 51 — INVESTIGATE DUPLICATE STATE-DISTRICT-YEAR KEYS
# ============================================================

print("=" * 80)
print("DUPLICATE KEY INVESTIGATION")
print("=" * 80)

duplicate_details = {}

for filename, df in normalized_district_data.items():

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    duplicate_mask = df.duplicated(
        subset=key_columns,
        keep=False
    )

    duplicate_rows = df[
        duplicate_mask
    ].copy()

    if len(duplicate_rows) == 0:
        continue

    duplicate_details[filename] = duplicate_rows

    print("\n" + "-" * 80)
    print(f"FILE: {filename}")
    print("-" * 80)

    print(
        f"Duplicate rows: {len(duplicate_rows)}"
    )

    # Display only keys first
    display(
        duplicate_rows[
            key_columns
        ].sort_values(
            key_columns
        )
    )

DUPLICATE KEY INVESTIGATION

--------------------------------------------------------------------------------
FILE: 01_District_wise_crimes_committed_IPC_2001_2012.csv
--------------------------------------------------------------------------------
Duplicate rows: 2


,STATE,DISTRICT,YEAR
6880,JAMMU & KASHMIR,RAILWAYS,2010
6881,JAMMU & KASHMIR,RAILWAYS,2010



--------------------------------------------------------------------------------
FILE: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
--------------------------------------------------------------------------------
Duplicate rows: 4


,STATE,DISTRICT,YEAR
204,JAMMU & KASHMIR,ANANTNAG,2001
205,JAMMU & KASHMIR,ANANTNAG,2001
3343,NAGALAND,TOTAL,2005
3344,NAGALAND,TOTAL,2005



--------------------------------------------------------------------------------
FILE: 02_District_wise_crimes_committed_against_ST_2001_2012.csv
--------------------------------------------------------------------------------
Duplicate rows: 4


,STATE,DISTRICT,YEAR
204,JAMMU & KASHMIR,ANANTNAG,2001
205,JAMMU & KASHMIR,ANANTNAG,2001
3343,NAGALAND,TOTAL,2005
3344,NAGALAND,TOTAL,2005



--------------------------------------------------------------------------------
FILE: 03_District_wise_crimes_committed_against_children_2001_2012.csv
--------------------------------------------------------------------------------
Duplicate rows: 2


,STATE,DISTRICT,YEAR
3341,NAGALAND,TOTAL,2005
3342,NAGALAND,TOTAL,2005



--------------------------------------------------------------------------------
FILE: 42_District_wise_crimes_committed_against_women_2001_2012.csv
--------------------------------------------------------------------------------
Duplicate rows: 2


,STATE,DISTRICT,YEAR
6880,JAMMU & KASHMIR,RAILWAYS,2010
6881,JAMMU & KASHMIR,RAILWAYS,2010


In [53]:
# ============================================================
# CELL 52 — COMPLETE DUPLICATE RECORD INSPECTION
# ============================================================

for filename, duplicate_rows in duplicate_details.items():

    print("\n" + "=" * 90)
    print(f"DUPLICATE RECORDS: {filename}")
    print("=" * 90)

    display(
        duplicate_rows
    )


DUPLICATE RECORDS: 01_District_wise_crimes_committed_IPC_2001_2012.csv


,STATE,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES,CRIME_GROUP,SOURCE_FILE
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,1,0,1,0,0,0,1,12,1,11,16,1,0,0,0,0,0,0,0,0,0,0,5,36,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,1,1,0,1,0,0,0,0,0,0,0,8,1,7,0,0,0,0,0,0,0,0,0,0,0,0,10,21,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv



DUPLICATE RECORDS: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv


,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping and Abduction,Dacoity,Robbery,Arson,Hurt,Prevention of atrocities (POA) Act,Protection of Civil Rights (PCR) Act,Other Crimes Against SCs,CRIME_GROUP,SOURCE_FILE
204,JAMMU & KASHMIR,ANANTNAG,2001,0,0,0,0,0,0,0,0,0,0,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
205,JAMMU & KASHMIR,ANANTNAG,2001,0,0,0,0,0,0,0,0,0,0,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
3343,NAGALAND,TOTAL,2005,0,0,0,0,0,0,0,0,0,0,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
3344,NAGALAND,TOTAL,2005,0,0,0,0,0,0,0,0,0,0,SC,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv



DUPLICATE RECORDS: 02_District_wise_crimes_committed_against_ST_2001_2012.csv


,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping Abduction,Dacoity,Robbery,Arson,Hurt,Protection of Civil Rights (PCR) Act,Prevention of atrocities (POA) Act,Other Crimes Against STs,CRIME_GROUP,SOURCE_FILE
204,JAMMU & KASHMIR,ANANTNAG,2001,0,0,0,0,0,0,0,0,0,0,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
205,JAMMU & KASHMIR,ANANTNAG,2001,0,0,0,0,0,0,0,0,0,0,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
3343,NAGALAND,TOTAL,2005,0,0,0,0,0,0,0,0,0,0,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv
3344,NAGALAND,TOTAL,2005,0,0,0,0,0,0,0,0,0,0,ST,02_District_wise_crimes_committed_against_ST_2001_2012.csv



DUPLICATE RECORDS: 03_District_wise_crimes_committed_against_children_2001_2012.csv


,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping and Abduction,Foeticide,Abetment of suicide,Exposure and abandonment,Procuration of minor girls,Buying of girls for prostitution,Selling of girls for prostitution,Prohibition of child marriage act,Other Crimes,Total,CRIME_GROUP,SOURCE_FILE
3341,NAGALAND,TOTAL,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
3342,NAGALAND,TOTAL,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv



DUPLICATE RECORDS: 42_District_wise_crimes_committed_against_women_2001_2012.csv


,STATE,DISTRICT,YEAR,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls,CRIME_GROUP,SOURCE_FILE
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,0,0,0,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv


In [54]:
# ============================================================
# CELL 53 — EXACT DUPLICATE CHECK
# ============================================================

exact_duplicate_summary = []

for filename, df in normalized_district_data.items():

    duplicate_mask = df.duplicated(
        keep=False
    )

    exact_duplicates = df[
        duplicate_mask
    ].duplicated(
        keep="first"
    ).sum()

    exact_duplicate_summary.append({

        "filename":
            filename,

        "duplicate_rows":
            int(
                df.duplicated(
                    subset=[
                        "STATE",
                        "DISTRICT",
                        "YEAR"
                    ],
                    keep=False
                ).sum()
            ),

        "exact_duplicate_rows":
            int(exact_duplicates)
    })


exact_duplicate_summary_df = pd.DataFrame(
    exact_duplicate_summary
)

display(
    exact_duplicate_summary_df[
        exact_duplicate_summary_df[
            "duplicate_rows"
        ] > 0
    ]
)

,filename,duplicate_rows,exact_duplicate_rows
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,2,0
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,4,2
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,4,2
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,2,1
11,42_District_wise_crimes_committed_against_women_2001_2012.csv,2,0


In [55]:
# ============================================================
# CELL 54 — DIFFERENCE ANALYSIS FOR DUPLICATE KEYS
# ============================================================

for filename, duplicate_rows in duplicate_details.items():

    print("\n" + "=" * 90)
    print(f"DUPLICATE DIFFERENCE ANALYSIS: {filename}")
    print("=" * 90)

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    non_key_columns = [
        c for c in duplicate_rows.columns
        if c not in key_columns
    ]

    # Group duplicate records by key
    for key, group in duplicate_rows.groupby(
        key_columns,
        dropna=False
    ):

        print("\nKEY:")
        print(key)

        differing_columns = []

        for column in non_key_columns:

            unique_values = (
                group[column]
                .astype("string")
                .fillna("<NA>")
                .unique()
            )

            if len(unique_values) > 1:
                differing_columns.append(
                    column
                )

        if differing_columns:

            print(
                "\nColumns with different values:"
            )

            for column in differing_columns:

                print(
                    f"  {column}: "
                    f"{group[column].tolist()}"
                )

        else:

            print(
                "\n✓ All non-key values are identical."
            )


DUPLICATE DIFFERENCE ANALYSIS: 01_District_wise_crimes_committed_IPC_2001_2012.csv

KEY:
('JAMMU & KASHMIR', 'RAILWAYS', 2010)

Columns with different values:
  MURDER: [0, 1]
  CULPABLE HOMICIDE NOT AMOUNTING TO MURDER: [0, 1]
  RAPE: [0, 1]
  OTHER RAPE: [0, 1]
  KIDNAPPING & ABDUCTION: [1, 0]
  KIDNAPPING AND ABDUCTION OF OTHERS: [1, 0]
  BURGLARY: [1, 0]
  THEFT: [12, 8]
  OTHER THEFT: [11, 7]
  RIOTS: [16, 0]
  CRIMINAL BREACH OF TRUST: [1, 0]
  OTHER IPC CRIMES: [5, 10]
  TOTAL IPC CRIMES: [36, 21]

DUPLICATE DIFFERENCE ANALYSIS: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv

KEY:
('JAMMU & KASHMIR', 'ANANTNAG', 2001)

✓ All non-key values are identical.

KEY:
('NAGALAND', 'TOTAL', 2005)

✓ All non-key values are identical.

DUPLICATE DIFFERENCE ANALYSIS: 02_District_wise_crimes_committed_against_ST_2001_2012.csv

KEY:
('JAMMU & KASHMIR', 'ANANTNAG', 2001)

✓ All non-key values are identical.

KEY:
('NAGALAND', 'TOTAL', 2005)

✓ All non-key values are identical.


In [56]:
# ============================================================
# CELL 55 — IDENTIFY THE HIDDEN DIMENSION IN DUPLICATES
# ============================================================

print("=" * 80)
print("HIDDEN-DIMENSION INVESTIGATION")
print("=" * 80)

for filename, duplicate_rows in duplicate_details.items():

    print("\n" + "=" * 90)
    print(f"FILE: {filename}")
    print("=" * 90)

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    # --------------------------------------------------------
    # For every duplicate key, inspect ALL columns that might
    # distinguish the records.
    # --------------------------------------------------------

    for key, group in duplicate_rows.groupby(
        key_columns,
        dropna=False
    ):

        if len(group) <= 1:
            continue

        print("\n" + "-" * 80)
        print("DUPLICATE KEY:")
        print(key)
        print("-" * 80)

        # Find columns with different values
        differing_columns = []

        for column in group.columns:

            if column in key_columns:
                continue

            unique_values = (
                group[column]
                .astype("string")
                .fillna("<NA>")
                .unique()
            )

            if len(unique_values) > 1:
                differing_columns.append(
                    column
                )

        if differing_columns:

            print(
                "\nColumns distinguishing the records:"
            )

            for column in differing_columns:

                print(
                    f"\n{column}:"
                )

                print(
                    group[
                        key_columns + [column]
                    ].to_string(
                        index=False
                    )
                )

        else:

            print(
                "\n⚠ No differing non-key columns found."
            )

HIDDEN-DIMENSION INVESTIGATION

FILE: 01_District_wise_crimes_committed_IPC_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY:
('JAMMU & KASHMIR', 'RAILWAYS', 2010)
--------------------------------------------------------------------------------

Columns distinguishing the records:

MURDER:
          STATE DISTRICT  YEAR  MURDER
JAMMU & KASHMIR RAILWAYS  2010       0
JAMMU & KASHMIR RAILWAYS  2010       1

CULPABLE HOMICIDE NOT AMOUNTING TO MURDER:
          STATE DISTRICT  YEAR  CULPABLE HOMICIDE NOT AMOUNTING TO MURDER
JAMMU & KASHMIR RAILWAYS  2010                                          0
JAMMU & KASHMIR RAILWAYS  2010                                          1

RAPE:
          STATE DISTRICT  YEAR  RAPE
JAMMU & KASHMIR RAILWAYS  2010     0
JAMMU & KASHMIR RAILWAYS  2010     1

OTHER RAPE:
          STATE DISTRICT  YEAR  OTHER RAPE
JAMMU & KASHMIR RAILWAYS  2010           0
JAMMU & KASHMIR RAILWAYS  2010           1

KIDN

In [57]:
# ============================================================
# CELL 56 — CATEGORICAL HIDDEN-DIMENSION CHECK
# ============================================================

for filename, duplicate_rows in duplicate_details.items():

    print("\n" + "=" * 90)
    print(f"CATEGORICAL ANALYSIS: {filename}")
    print("=" * 90)

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    categorical_columns = [
        column
        for column in duplicate_rows.columns
        if column not in key_columns
        and (
            duplicate_rows[column].dtype == "object"
            or str(
                duplicate_rows[column].dtype
            ).startswith("string")
        )
    ]

    if categorical_columns:

        print(
            "\nCategorical columns found:"
        )

        for column in categorical_columns:

            print(
                f"\n{column}:"
            )

            print(
                duplicate_rows[
                    column
                ].value_counts(
                    dropna=False
                )
            )

    else:

        print(
            "\nNo categorical columns detected."
        )


CATEGORICAL ANALYSIS: 01_District_wise_crimes_committed_IPC_2001_2012.csv

Categorical columns found:

CRIME_GROUP:
CRIME_GROUP
IPC    2
Name: count, dtype: int64

SOURCE_FILE:
SOURCE_FILE
01_District_wise_crimes_committed_IPC_2001_2012.csv    2
Name: count, dtype: int64

CATEGORICAL ANALYSIS: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv

Categorical columns found:

CRIME_GROUP:
CRIME_GROUP
SC    4
Name: count, dtype: int64

SOURCE_FILE:
SOURCE_FILE
02_01_District_wise_crimes_committed_against_SC_2001_2012.csv    4
Name: count, dtype: int64

CATEGORICAL ANALYSIS: 02_District_wise_crimes_committed_against_ST_2001_2012.csv

Categorical columns found:

CRIME_GROUP:
CRIME_GROUP
ST    4
Name: count, dtype: int64

SOURCE_FILE:
SOURCE_FILE
02_District_wise_crimes_committed_against_ST_2001_2012.csv    4
Name: count, dtype: int64

CATEGORICAL ANALYSIS: 03_District_wise_crimes_committed_against_children_2001_2012.csv

Categorical columns found:

CRIME_GROUP:
CRIME_GROUP
CHILDRE

In [58]:
# ============================================================
# CELL 57 — INVESTIGATE TOTAL ROWS
# ============================================================

for filename, df in normalized_district_data.items():

    if "DISTRICT" not in df.columns:
        continue

    total_mask = (
        df["DISTRICT"]
        .astype("string")
        .str.strip()
        .str.upper()
        .eq("TOTAL")
    )

    total_rows = df[
        total_mask
    ].copy()

    if len(total_rows) == 0:
        continue

    print("\n" + "=" * 80)
    print(f"TOTAL ROWS: {filename}")
    print("=" * 80)

    print(
        f"Number of TOTAL rows: "
        f"{len(total_rows)}"
    )

    display(
        total_rows[
            [
                "STATE",
                "DISTRICT",
                "YEAR"
            ]
        ].head(30)
    )


TOTAL ROWS: 01_District_wise_crimes_committed_IPC_2001_2012.csv
Number of TOTAL rows: 408


,STATE,DISTRICT,YEAR
28,ANDHRA PRADESH,TOTAL,2001
42,ARUNACHAL PRADESH,TOTAL,2001
70,ASSAM,TOTAL,2001
115,BIHAR,TOTAL,2001
135,CHHATTISGARH,TOTAL,2001
138,GOA,TOTAL,2001
169,GUJARAT,TOTAL,2001
190,HARYANA,TOTAL,2001
204,HIMACHAL PRADESH,TOTAL,2001
228,JAMMU & KASHMIR,TOTAL,2001



TOTAL ROWS: 01_District_wise_crimes_committed_IPC_2014.csv
Number of TOTAL rows: 36


,STATE,DISTRICT,YEAR
20,Andhra Pradesh,Total,2014
40,Arunachal Pradesh,Total,2014
69,Assam,Total,2014
116,Bihar,Total,2014
145,Chhattisgarh,Total,2014
149,Goa,Total,2014
191,Gujarat,Total,2014
216,Haryana,Total,2014
232,Himachal Pradesh,Total,2014
263,Jammu & Kashmir,Total,2014



TOTAL ROWS: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv
Number of TOTAL rows: 421


...

,STATE,DISTRICT,YEAR
33,Andhra Pradesh,TOTAL,2013
52,Arunachal Pradesh,TOTAL,2013
86,Assam,TOTAL,2013
131,Bihar,TOTAL,2013
160,Chhattisgarh,TOTAL,2013
163,Goa,TOTAL,2013
198,Gujarat,TOTAL,2013
223,Haryana,TOTAL,2013
239,Himachal Pradesh,TOTAL,2013
270,Jammu & Kashmir,TOTAL,2013



TOTAL ROWS: 42_District_wise_crimes_committed_against_women_2001_2012.csv
Number of TOTAL rows: 408


,STATE,DISTRICT,YEAR
28,ANDHRA PRADESH,TOTAL,2001
42,ARUNACHAL PRADESH,TOTAL,2001
70,ASSAM,TOTAL,2001
115,BIHAR,TOTAL,2001
135,CHHATTISGARH,TOTAL,2001
138,GOA,TOTAL,2001
169,GUJARAT,TOTAL,2001
190,HARYANA,TOTAL,2001
204,HIMACHAL PRADESH,TOTAL,2001
228,JAMMU & KASHMIR,TOTAL,2001



TOTAL ROWS: 42_District_wise_crimes_committed_against_women_2014.csv
Number of TOTAL rows: 36


,STATE,DISTRICT,YEAR
20,Andhra Pradesh,Total,2014
40,Arunachal Pradesh,Total,2014
68,Assam,Total,2014
115,Bihar,Total,2014
144,Chhattisgarh,Total,2014
148,Goa,Total,2014
190,Gujarat,Total,2014
215,Haryana,Total,2014
231,Himachal Pradesh,Total,2014
262,Jammu & Kashmir,Total,2014


In [59]:
# ============================================================
# CELL 58 — SPECIAL DISTRICT VALUE AUDIT
# ============================================================

special_terms = [
    "TOTAL",
    "RAILWAYS",
    "CITY",
    "COMMISSIONERATE",
    "RURAL",
    "URBAN"
]

special_records = []

for filename, df in normalized_district_data.items():

    if "DISTRICT" not in df.columns:
        continue

    district_values = (
        df["DISTRICT"]
        .astype("string")
        .str.strip()
    )

    for term in special_terms:

        matches = district_values[
            district_values
            .str.upper()
            .str.contains(
                term,
                na=False
            )
        ]

        if len(matches) > 0:

            special_records.append({

                "filename":
                    filename,

                "term":
                    term,

                "matching_rows":
                    len(matches),

                "unique_values":
                    "; ".join(
                        sorted(
                            matches
                            .dropna()
                            .unique()
                            .tolist()
                        )[:20]
                    )
            })


special_district_df = pd.DataFrame(
    special_records
)

display(
    special_district_df
)

,filename,term,matching_rows,unique_values
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,TOTAL,420,DELHI UT TOTAL; TOTAL
1,01_District_wise_crimes_committed_IPC_2001_2012.csv,RAILWAYS,41,RAILWAYS; RAILWAYS JAMMU; RAILWAYS KASHMIR; RAILWAYS KATRA; RAILWAYS KMR
2,01_District_wise_crimes_committed_IPC_2001_2012.csv,CITY,53,GUWAHATI CITY; HOWRAH CITY; HYDERABAD CITY; JODHPUR CITY; KOTA CITY; MANGALORE CITY; VIJAYAWADA CITY
3,01_District_wise_crimes_committed_IPC_2001_2012.csv,RURAL,310,AHMEDABAD RURAL; AMBALA RURAL; AMRAVATI RURAL; AMRITSAR RURAL; AURANGABAD RURAL; BANGALORE RURAL; COIMBATORE RURAL; ...
4,01_District_wise_crimes_committed_IPC_2001_2012.csv,URBAN,73,AMBALA URBAN; CHENNAISUBURBAN; COIMBATORE URBAN; GUNTUR URBAN; MADURAI URBAN; SALEM URBAN; THIRUNELVELI URBAN; TIRUP...
...,...,...,...,...
65,42_District_wise_crimes_committed_against_women_2014.csv,TOTAL,36,Total
66,42_District_wise_crimes_committed_against_women_2014.csv,RAILWAYS,5,K.Railways; Railways; Railways Jammu; Railways Kashmir; Railways Katra
67,42_District_wise_crimes_committed_against_women_2014.csv,CITY,20,Ahmedabad City; Belagavi City; Bengaluru City; Coimbatore City; Guwahati City; Hubballi Dharwad City; Hyderabad City...
68,42_District_wise_crimes_committed_against_women_2014.csv,RURAL,26,Ahmedabad Rural; Ambala (Rural); Amravati Rural; Amritsar Rural; Aurangabad Rural; Ernakulam Rural; Howrah Rural; Ja...


In [60]:
# ============================================================
# CELL 59 — STATE/YEAR/DISTRICT CARDINALITY
# ============================================================

for filename, df in normalized_district_data.items():

    if not all(
        column in df.columns
        for column in [
            "STATE",
            "DISTRICT",
            "YEAR"
        ]
    ):
        continue

    cardinality = (
        df.groupby(
            [
                "STATE",
                "YEAR"
            ],
            dropna=False
        )["DISTRICT"]
        .nunique()
        .reset_index(
            name="unique_district_count"
        )
        .sort_values(
            "unique_district_count"
        )
    )

    suspicious = cardinality[
        cardinality[
            "unique_district_count"
        ] <= 2
    ]

    if len(suspicious) > 0:

        print("\n" + "=" * 80)
        print(f"CARDINALITY CHECK: {filename}")
        print("=" * 80)

        display(
            suspicious.head(20)
        )


CARDINALITY CHECK: 01_District_wise_crimes_committed_IPC_2001_2012.csv


,STATE,YEAR,unique_district_count
71,CHANDIGARH,2012,2
86,D & N HAVELI,2003,2
316,PUDUCHERRY,2005,2
315,PUDUCHERRY,2004,2
314,PUDUCHERRY,2003,2
313,PUDUCHERRY,2002,2
312,PUDUCHERRY,2001,2
70,CHANDIGARH,2011,2
84,D & N HAVELI,2001,2
95,D & N HAVELI,2012,2



CARDINALITY CHECK: 01_District_wise_crimes_committed_IPC_2013.csv


,STATE,YEAR,unique_district_count
18,Lakshadweep,2013,2
7,D&N Haveli,2013,2
5,Chandigarh,2013,2



CARDINALITY CHECK: 01_District_wise_crimes_committed_IPC_2014.csv


...

,STATE,YEAR,unique_district_count
71,CHANDIGARH,2012,2
86,D & N HAVELI,2003,2
316,PUDUCHERRY,2005,2
315,PUDUCHERRY,2004,2
314,PUDUCHERRY,2003,2
313,PUDUCHERRY,2002,2
312,PUDUCHERRY,2001,2
70,CHANDIGARH,2011,2
84,D & N HAVELI,2001,2
95,D & N HAVELI,2012,2



CARDINALITY CHECK: 42_District_wise_crimes_committed_against_women_2013.csv


,STATE,YEAR,unique_district_count
18,Lakshadweep,2013,2
7,D&N Haveli,2013,2
5,Chandigarh,2013,2



CARDINALITY CHECK: 42_District_wise_crimes_committed_against_women_2014.csv


,STATE,YEAR,unique_district_count
18,Lakshadweep,2014,2
7,D&N Haveli,2014,2
5,Chandigarh,2014,2


In [63]:
# ============================================================
# CELL 60 — INSPECT DUPLICATE ROW CONTEXT
# FIXED VERSION
# ============================================================

print("=" * 90)
print("RAW/NORMALIZED ROW CONTEXT AROUND DUPLICATE KEYS")
print("=" * 90)


for filename, duplicate_rows in duplicate_details.items():

    print("\n" + "=" * 90)
    print(f"FILE: {filename}")
    print("=" * 90)

    # --------------------------------------------------------
    # IMPORTANT:
    # Use the already-normalized dataframe.
    #
    # This dataframe has:
    # STATE
    # DISTRICT
    # YEAR
    # --------------------------------------------------------

    if filename not in normalized_district_data:

        print(
            f"⚠ Normalized dataframe not found: "
            f"{filename}"
        )

        continue

    df = normalized_district_data[
        filename
    ].copy()

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    # --------------------------------------------------------
    # Make sure required columns exist
    # --------------------------------------------------------

    missing_columns = [
        column
        for column in key_columns
        if column not in df.columns
    ]

    if missing_columns:

        print(
            f"⚠ Missing columns: "
            f"{missing_columns}"
        )

        continue

    # --------------------------------------------------------
    # Get unique duplicate keys
    # --------------------------------------------------------

    duplicate_keys = (
        duplicate_rows[
            key_columns
        ]
        .drop_duplicates()
        .itertuples(
            index=False,
            name=None
        )
    )

    # --------------------------------------------------------
    # Inspect every duplicate key
    # --------------------------------------------------------

    for key in duplicate_keys:

        print("\n" + "-" * 80)
        print(
            f"DUPLICATE KEY: {key}"
        )
        print("-" * 80)

        # ----------------------------------------------------
        # Find matching rows in normalized dataframe
        # ----------------------------------------------------

        mask = pd.Series(
            True,
            index=df.index
        )

        for column, value in zip(
            key_columns,
            key
        ):

            if pd.isna(value):

                mask &= df[column].isna()

            else:

                mask &= (
                    df[column]
                    .astype("string")
                    .str.strip()
                    .eq(
                        str(value).strip()
                    )
                )

        matching_indices = (
            df.index[mask]
            .tolist()
        )

        print(
            "Matching dataframe indices:",
            matching_indices
        )

        # ----------------------------------------------------
        # Show surrounding rows
        # ----------------------------------------------------

        for idx in matching_indices:

            start = max(
                0,
                idx - 2
            )

            end = min(
                len(df),
                idx + 3
            )

            print(
                f"\nContext around dataframe "
                f"index {idx}:"
            )

            context_df = df.iloc[
                start:end
            ].copy()

            display(
                context_df
            )

RAW/NORMALIZED ROW CONTEXT AROUND DUPLICATE KEYS

FILE: 01_District_wise_crimes_committed_IPC_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY: ('JAMMU & KASHMIR', 'RAILWAYS', 2010)
--------------------------------------------------------------------------------
Matching dataframe indices: [6880, 6881]

Context around dataframe index 6880:


,STATE,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES,CRIME_GROUP,SOURCE_FILE
6878,JAMMU & KASHMIR,POONCH,2010,12,37,1,20,0,20,39,34,5,0,0,0,23,31,3,28,132,4,8,0,0,54,0,30,10,16,0,4,529,950,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6879,JAMMU & KASHMIR,PULWAMA,2010,4,10,2,8,0,8,33,31,2,1,0,4,37,80,10,70,172,3,7,3,8,55,0,11,13,0,0,2,164,617,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,1,0,1,0,0,0,1,12,1,11,16,1,0,0,0,0,0,0,0,0,0,0,5,36,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,1,1,0,1,0,0,0,0,0,0,0,8,1,7,0,0,0,0,0,0,0,0,0,0,0,0,10,21,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6882,JAMMU & KASHMIR,RAJOURI,2010,15,36,6,36,0,36,47,42,5,2,0,0,44,52,20,32,33,10,15,0,22,30,0,26,4,44,0,38,1000,1460,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv



Context around dataframe index 6881:


,STATE,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES,CRIME_GROUP,SOURCE_FILE
6879,JAMMU & KASHMIR,PULWAMA,2010,4,10,2,8,0,8,33,31,2,1,0,4,37,80,10,70,172,3,7,3,8,55,0,11,13,0,0,2,164,617,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,1,0,1,0,0,0,1,12,1,11,16,1,0,0,0,0,0,0,0,0,0,0,5,36,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,1,1,0,1,0,0,0,0,0,0,0,8,1,7,0,0,0,0,0,0,0,0,0,0,0,0,10,21,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6882,JAMMU & KASHMIR,RAJOURI,2010,15,36,6,36,0,36,47,42,5,2,0,0,44,52,20,32,33,10,15,0,22,30,0,26,4,44,0,38,1000,1460,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv
6883,JAMMU & KASHMIR,RAMBAN,2010,3,9,0,6,0,6,17,17,0,0,0,0,24,37,20,17,17,3,10,1,1,2,0,11,2,0,0,2,491,636,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv



FILE: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY: ('JAMMU & KASHMIR', 'ANANTNAG', 2001)
--------------------------------------------------------------------------------
Matching dataframe indices: [204, 205]

Context around dataframe index 204:


...

,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping and Abduction,Foeticide,Abetment of suicide,Exposure and abandonment,Procuration of minor girls,Buying of girls for prostitution,Selling of girls for prostitution,Prohibition of child marriage act,Other Crimes,Total,CRIME_GROUP,SOURCE_FILE
3340,NAGALAND,ZUNHEBOTO,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
3341,NAGALAND,TOTAL,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
3342,NAGALAND,TOTAL,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
3343,ODISHA,ANGUL,2005,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.0,17,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv
3344,ODISHA,BALASORE,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv



FILE: 42_District_wise_crimes_committed_against_women_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY: ('JAMMU & KASHMIR', 'RAILWAYS', 2010)
--------------------------------------------------------------------------------
Matching dataframe indices: [6880, 6881]

Context around dataframe index 6880:


,STATE,DISTRICT,YEAR,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls,CRIME_GROUP,SOURCE_FILE
6878,JAMMU & KASHMIR,POONCH,2010,20,34,0,30,10,16,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6879,JAMMU & KASHMIR,PULWAMA,2010,8,31,0,11,13,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,0,0,0,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6882,JAMMU & KASHMIR,RAJOURI,2010,36,42,0,26,4,44,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv



Context around dataframe index 6881:


,STATE,DISTRICT,YEAR,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls,CRIME_GROUP,SOURCE_FILE
6879,JAMMU & KASHMIR,PULWAMA,2010,8,31,0,11,13,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,0,0,0,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6882,JAMMU & KASHMIR,RAJOURI,2010,36,42,0,26,4,44,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv
6883,JAMMU & KASHMIR,RAMBAN,2010,6,17,0,11,2,0,0,WOMEN,42_District_wise_crimes_committed_against_women_2001_2012.csv


In [64]:
# ============================================================
# CELL 60B — RAW SOURCE CONTEXT
# ============================================================

print("=" * 90)
print("RAW SOURCE CONTEXT")
print("=" * 90)


for filename, duplicate_rows in duplicate_details.items():

    print("\n" + "=" * 90)
    print(f"RAW FILE: {filename}")
    print("=" * 90)

    file_path = RAW_DATA_DIR / filename

    # --------------------------------------------------------
    # Read raw source
    # --------------------------------------------------------

    raw_df, read_info = read_csv_for_inspection(
        file_path
    )

    if raw_df is None:

        print(
            "⚠ Could not read raw file."
        )

        continue

    # --------------------------------------------------------
    # Normalize raw column names ONLY for searching.
    # Data values are not changed.
    # --------------------------------------------------------

    raw_df = normalize_dimension_columns(
        raw_df
    )

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    missing_columns = [
        column
        for column in key_columns
        if column not in raw_df.columns
    ]

    if missing_columns:

        print(
            f"⚠ Required columns missing: "
            f"{missing_columns}"
        )

        print(
            "Available columns:"
        )

        print(
            raw_df.columns.tolist()
        )

        continue

    # --------------------------------------------------------
    # Unique duplicate keys
    # --------------------------------------------------------

    duplicate_keys = (
        duplicate_rows[
            key_columns
        ]
        .drop_duplicates()
        .itertuples(
            index=False,
            name=None
        )
    )

    # --------------------------------------------------------
    # Inspect each duplicate key
    # --------------------------------------------------------

    for key in duplicate_keys:

        mask = pd.Series(
            True,
            index=raw_df.index
        )

        for column, value in zip(
            key_columns,
            key
        ):

            if pd.isna(value):

                mask &= raw_df[column].isna()

            else:

                mask &= (
                    raw_df[column]
                    .astype("string")
                    .str.strip()
                    .eq(
                        str(value).strip()
                    )
                )

        matching_indices = (
            raw_df.index[mask]
            .tolist()
        )

        print("\n" + "-" * 80)
        print(
            f"DUPLICATE KEY: {key}"
        )
        print(
            f"RAW ROW INDICES: "
            f"{matching_indices}"
        )
        print("-" * 80)

        for idx in matching_indices:

            start = max(
                0,
                idx - 2
            )

            end = min(
                len(raw_df),
                idx + 3
            )

            print(
                f"\nRaw source context "
                f"around row {idx}:"
            )

            display(
                raw_df.iloc[
                    start:end
                ]
            )

RAW SOURCE CONTEXT

RAW FILE: 01_District_wise_crimes_committed_IPC_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY: ('JAMMU & KASHMIR', 'RAILWAYS', 2010)
RAW ROW INDICES: [6880, 6881]
--------------------------------------------------------------------------------

Raw source context around row 6880:


,STATE,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES
6878,JAMMU & KASHMIR,POONCH,2010,12,37,1,20,0,20,39,34,5,0,0,0,23,31,3,28,132,4,8,0,0,54,0,30,10,16,0,4,529,950
6879,JAMMU & KASHMIR,PULWAMA,2010,4,10,2,8,0,8,33,31,2,1,0,4,37,80,10,70,172,3,7,3,8,55,0,11,13,0,0,2,164,617
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,1,0,1,0,0,0,1,12,1,11,16,1,0,0,0,0,0,0,0,0,0,0,5,36
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,1,1,0,1,0,0,0,0,0,0,0,8,1,7,0,0,0,0,0,0,0,0,0,0,0,0,10,21
6882,JAMMU & KASHMIR,RAJOURI,2010,15,36,6,36,0,36,47,42,5,2,0,0,44,52,20,32,33,10,15,0,22,30,0,26,4,44,0,38,1000,1460



Raw source context around row 6881:


,STATE,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING AND ABDUCTION OF OTHERS,DACOITY,PREPARATION AND ASSEMBLY FOR DACOITY,ROBBERY,BURGLARY,THEFT,AUTO THEFT,OTHER THEFT,RIOTS,CRIMINAL BREACH OF TRUST,CHEATING,COUNTERFIETING,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES
6879,JAMMU & KASHMIR,PULWAMA,2010,4,10,2,8,0,8,33,31,2,1,0,4,37,80,10,70,172,3,7,3,8,55,0,11,13,0,0,2,164,617
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,1,0,1,0,0,0,1,12,1,11,16,1,0,0,0,0,0,0,0,0,0,0,5,36
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,1,1,0,1,0,0,0,0,0,0,0,8,1,7,0,0,0,0,0,0,0,0,0,0,0,0,10,21
6882,JAMMU & KASHMIR,RAJOURI,2010,15,36,6,36,0,36,47,42,5,2,0,0,44,52,20,32,33,10,15,0,22,30,0,26,4,44,0,38,1000,1460
6883,JAMMU & KASHMIR,RAMBAN,2010,3,9,0,6,0,6,17,17,0,0,0,0,24,37,20,17,17,3,10,1,1,2,0,11,2,0,0,2,491,636



RAW FILE: 02_01_District_wise_crimes_committed_against_SC_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY: ('JAMMU & KASHMIR', 'ANANTNAG', 2001)
RAW ROW INDICES: [204, 205]
--------------------------------------------------------------------------------

Raw source context around row 204:


...

,STATE,DISTRICT,YEAR,Murder,Rape,Kidnapping and Abduction,Foeticide,Abetment of suicide,Exposure and abandonment,Procuration of minor girls,Buying of girls for prostitution,Selling of girls for prostitution,Prohibition of child marriage act,Other Crimes,Total
3340,NAGALAND,ZUNHEBOTO,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3341,NAGALAND,TOTAL,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3342,NAGALAND,TOTAL,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3343,ODISHA,ANGUL,2005,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.0,17
3344,ODISHA,BALASORE,2005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0



RAW FILE: 42_District_wise_crimes_committed_against_women_2001_2012.csv

--------------------------------------------------------------------------------
DUPLICATE KEY: ('JAMMU & KASHMIR', 'RAILWAYS', 2010)
RAW ROW INDICES: [6880, 6881]
--------------------------------------------------------------------------------

Raw source context around row 6880:


,STATE,DISTRICT,YEAR,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls
6878,JAMMU & KASHMIR,POONCH,2010,20,34,0,30,10,16,0
6879,JAMMU & KASHMIR,PULWAMA,2010,8,31,0,11,13,0,0
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,0
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,0,0,0,0,0
6882,JAMMU & KASHMIR,RAJOURI,2010,36,42,0,26,4,44,0



Raw source context around row 6881:


,STATE,DISTRICT,YEAR,Rape,Kidnapping and Abduction,Dowry Deaths,Assault on women with intent to outrage her modesty,Insult to modesty of Women,Cruelty by Husband or his Relatives,Importation of Girls
6879,JAMMU & KASHMIR,PULWAMA,2010,8,31,0,11,13,0,0
6880,JAMMU & KASHMIR,RAILWAYS,2010,0,0,0,0,0,0,0
6881,JAMMU & KASHMIR,RAILWAYS,2010,1,0,0,0,0,0,0
6882,JAMMU & KASHMIR,RAJOURI,2010,36,42,0,26,4,44,0
6883,JAMMU & KASHMIR,RAMBAN,2010,6,17,0,11,2,0,0


In [65]:
# ============================================================
# CELL 61 — DEFINITIVE DUPLICATE CLASSIFICATION
# ============================================================

duplicate_audit_records = []

for filename, df in normalized_district_data.items():

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    # --------------------------------------------------------
    # Find duplicate keys
    # --------------------------------------------------------

    duplicate_mask = df.duplicated(
        subset=key_columns,
        keep=False
    )

    duplicate_df = df[
        duplicate_mask
    ].copy()

    if duplicate_df.empty:
        continue

    # --------------------------------------------------------
    # Analyze each duplicated key
    # --------------------------------------------------------

    for key, group in duplicate_df.groupby(
        key_columns,
        dropna=False
    ):

        # Remove key columns for comparison
        comparison_columns = [
            c
            for c in group.columns
            if c not in key_columns
        ]

        # ----------------------------------------------------
        # Check whether complete rows are identical
        # ----------------------------------------------------

        unique_records = (
            group[
                comparison_columns
            ]
            .drop_duplicates()
        )

        if len(unique_records) == 1:

            duplicate_type = (
                "EXACT_DUPLICATE"
            )

        else:

            duplicate_type = (
                "CONFLICTING_DUPLICATE"
            )

        duplicate_audit_records.append({

            "filename":
                filename,

            "state":
                key[0],

            "unit_name":
                key[1],

            "year":
                key[2],

            "row_count":
                len(group),

            "unique_record_count":
                len(unique_records),

            "duplicate_type":
                duplicate_type
        })


duplicate_audit_df = pd.DataFrame(
    duplicate_audit_records
)

print("=" * 80)
print("DEFINITIVE DUPLICATE CLASSIFICATION")
print("=" * 80)

display(
    duplicate_audit_df
)

DEFINITIVE DUPLICATE CLASSIFICATION


,filename,state,unit_name,year,row_count,unique_record_count,duplicate_type
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,JAMMU & KASHMIR,RAILWAYS,2010,2,2,CONFLICTING_DUPLICATE
1,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,JAMMU & KASHMIR,ANANTNAG,2001,2,1,EXACT_DUPLICATE
2,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,NAGALAND,TOTAL,2005,2,1,EXACT_DUPLICATE
3,02_District_wise_crimes_committed_against_ST_2001_2012.csv,JAMMU & KASHMIR,ANANTNAG,2001,2,1,EXACT_DUPLICATE
4,02_District_wise_crimes_committed_against_ST_2001_2012.csv,NAGALAND,TOTAL,2005,2,1,EXACT_DUPLICATE
5,03_District_wise_crimes_committed_against_children_2001_2012.csv,NAGALAND,TOTAL,2005,2,1,EXACT_DUPLICATE
6,42_District_wise_crimes_committed_against_women_2001_2012.csv,JAMMU & KASHMIR,RAILWAYS,2010,2,2,CONFLICTING_DUPLICATE


In [66]:
# ============================================================
# CELL 62 — DUPLICATE TYPE SUMMARY
# ============================================================

duplicate_type_summary = (
    duplicate_audit_df[
        "duplicate_type"
    ]
    .value_counts()
    .to_frame(
        "key_count"
    )
)

display(
    duplicate_type_summary
)

,key_count
duplicate_type,
EXACT_DUPLICATE,5
CONFLICTING_DUPLICATE,2


In [67]:
# ============================================================
# CELL 63 — CONFLICTING DUPLICATES ONLY
# ============================================================

conflicting_duplicates = (
    duplicate_audit_df[
        duplicate_audit_df[
            "duplicate_type"
        ]
        == "CONFLICTING_DUPLICATE"
    ]
    .sort_values(
        [
            "filename",
            "state",
            "unit_name",
            "year"
        ]
    )
)

print("=" * 80)
print("CONFLICTING DUPLICATES")
print("=" * 80)

display(
    conflicting_duplicates
)

CONFLICTING DUPLICATES


,filename,state,unit_name,year,row_count,unique_record_count,duplicate_type
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,JAMMU & KASHMIR,RAILWAYS,2010,2,2,CONFLICTING_DUPLICATE
6,42_District_wise_crimes_committed_against_women_2001_2012.csv,JAMMU & KASHMIR,RAILWAYS,2010,2,2,CONFLICTING_DUPLICATE


In [68]:
# ============================================================
# CELL 64 — EXACT DUPLICATES ONLY
# ============================================================

exact_duplicates = (
    duplicate_audit_df[
        duplicate_audit_df[
            "duplicate_type"
        ]
        == "EXACT_DUPLICATE"
    ]
    .sort_values(
        [
            "filename",
            "state",
            "unit_name",
            "year"
        ]
    )
)

print("=" * 80)
print("EXACT DUPLICATES")
print("=" * 80)

display(
    exact_duplicates
)

EXACT DUPLICATES


,filename,state,unit_name,year,row_count,unique_record_count,duplicate_type
1,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,JAMMU & KASHMIR,ANANTNAG,2001,2,1,EXACT_DUPLICATE
2,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,NAGALAND,TOTAL,2005,2,1,EXACT_DUPLICATE
3,02_District_wise_crimes_committed_against_ST_2001_2012.csv,JAMMU & KASHMIR,ANANTNAG,2001,2,1,EXACT_DUPLICATE
4,02_District_wise_crimes_committed_against_ST_2001_2012.csv,NAGALAND,TOTAL,2005,2,1,EXACT_DUPLICATE
5,03_District_wise_crimes_committed_against_children_2001_2012.csv,NAGALAND,TOTAL,2005,2,1,EXACT_DUPLICATE


In [69]:
# ============================================================
# CELL 65 — ADD SOURCE ROW IDENTIFIERS
# ============================================================

for filename, df in normalized_district_data.items():

    # Preserve the original row position
    df["SOURCE_ROW_ID"] = range(
        len(df)
    )

    # Unique identifier across all source files
    df["SOURCE_RECORD_ID"] = (
        filename
        + "::"
        + df["SOURCE_ROW_ID"]
        .astype(str)
    )

print(
    "✓ SOURCE_ROW_ID and SOURCE_RECORD_ID "
    "added to all district datasets."
)

✓ SOURCE_ROW_ID and SOURCE_RECORD_ID added to all district datasets.


In [70]:
# ============================================================
# CELL 66 — ADD DUPLICATE QUALITY FLAGS
# ============================================================

for filename, df in normalized_district_data.items():

    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    duplicate_mask = df.duplicated(
        subset=key_columns,
        keep=False
    )

    df["DUPLICATE_KEY_FLAG"] = (
        duplicate_mask
    )

    df["DATA_QUALITY_FLAG"] = (
        "UNIQUE"
    )

    df.loc[
        duplicate_mask,
        "DATA_QUALITY_FLAG"
    ] = "DUPLICATE_KEY_REVIEW"

In [71]:
# ============================================================
# CELL 67 — CLASSIFY REPORTING UNIT
# ============================================================

def classify_unit_type(unit_value):

    if pd.isna(unit_value):
        return "MISSING"

    value = str(
        unit_value
    ).strip().upper()

    if value == "TOTAL":
        return "AGGREGATE"

    if "RAILWAY" in value:
        return "SPECIAL_UNIT"

    if "CITY" in value:
        return "SPECIAL_UNIT"

    if "RURAL" in value:
        return "SPECIAL_UNIT"

    if "URBAN" in value:
        return "SPECIAL_UNIT"

    return "DISTRICT_OR_REPORTING_UNIT"


for filename, df in normalized_district_data.items():

    df["UNIT_NAME"] = (
        df["DISTRICT"]
        .astype("string")
        .str.strip()
    )

    df["UNIT_TYPE"] = (
        df["UNIT_NAME"]
        .apply(
            classify_unit_type
        )
    )

print(
    "✓ UNIT_NAME and UNIT_TYPE added."
)

✓ UNIT_NAME and UNIT_TYPE added.


In [72]:
# ============================================================
# CELL 68 — FINAL DISTRICT AUDIT SUMMARY
# ============================================================

all_district_audit = []

for filename, df in normalized_district_data.items():

    all_district_audit.append({

        "filename":
            filename,

        "rows":
            len(df),

        "unique_rows":
            len(
                df.drop_duplicates()
            ),

        "duplicate_key_rows":
            int(
                df[
                    "DUPLICATE_KEY_FLAG"
                ].sum()
            ),

        "aggregate_rows":
            int(
                (
                    df["UNIT_TYPE"]
                    == "AGGREGATE"
                ).sum()
            ),

        "special_unit_rows":
            int(
                (
                    df["UNIT_TYPE"]
                    == "SPECIAL_UNIT"
                ).sum()
            ),

        "ordinary_reporting_rows":
            int(
                (
                    df["UNIT_TYPE"]
                    == "DISTRICT_OR_REPORTING_UNIT"
                ).sum()
            )
    })


district_audit_summary = pd.DataFrame(
    all_district_audit
)

display(
    district_audit_summary
)

,filename,rows,unique_rows,duplicate_key_rows,aggregate_rows,special_unit_rows,ordinary_reporting_rows
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,9017,9017,2,408,477,8132
1,01_District_wise_crimes_committed_IPC_2013.csv,823,823,0,0,53,770
2,01_District_wise_crimes_committed_IPC_2014.csv,838,838,0,36,73,729
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,9018,9018,4,421,477,8120
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,823,823,0,35,53,735
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,837,837,0,36,73,728
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,9018,9018,4,421,477,8120
7,02_District_wise_crimes_committed_against_ST_2013.csv,823,823,0,0,53,770
8,02_District_wise_crimes_committed_against_ST_2014.csv,837,837,0,36,73,728
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,9015,9015,2,412,474,8129


In [73]:
# ============================================================
# CELL 69 — REMOVE EXACT DUPLICATE COPIES
# ============================================================

cleaned_district_data = {}

exact_duplicate_removed_records = []

for filename, df in normalized_district_data.items():

    df = df.copy()

    original_rows = len(df)

    # --------------------------------------------------------
    # Remove only completely identical records.
    #
    # IMPORTANT:
    # We do NOT use STATE + DISTRICT + YEAR here.
    # --------------------------------------------------------

    df_clean = df.drop_duplicates(
        keep="first"
    ).copy()

    removed_rows = (
        original_rows -
        len(df_clean)
    )

    cleaned_district_data[
        filename
    ] = df_clean

    exact_duplicate_removed_records.append({

        "filename":
            filename,

        "original_rows":
            original_rows,

        "rows_after_exact_dedup":
            len(df_clean),

        "exact_duplicate_rows_removed":
            removed_rows
    })


exact_dedup_summary = pd.DataFrame(
    exact_duplicate_removed_records
)

print("=" * 80)
print("EXACT DUPLICATE REMOVAL")
print("=" * 80)

display(
    exact_dedup_summary[
        exact_dedup_summary[
            "exact_duplicate_rows_removed"
        ] > 0
    ]
)

print(
    "\nTotal exact duplicate rows removed:",
    exact_dedup_summary[
        "exact_duplicate_rows_removed"
    ].sum()
)

EXACT DUPLICATE REMOVAL


,filename,original_rows,rows_after_exact_dedup,exact_duplicate_rows_removed



Total exact duplicate rows removed: 0


In [74]:
# ============================================================
# CELL 70 — FLAG CONFLICTING SOURCE RECORDS
# ============================================================

conflicting_keys = (
    conflicting_duplicates[
        [
            "filename",
            "state",
            "unit_name",
            "year"
        ]
    ]
    .copy()
)

print("=" * 80)
print("CONFLICTING SOURCE RECORDS")
print("=" * 80)

display(
    conflicting_keys
)

CONFLICTING SOURCE RECORDS


,filename,state,unit_name,year
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,JAMMU & KASHMIR,RAILWAYS,2010
6,42_District_wise_crimes_committed_against_women_2001_2012.csv,JAMMU & KASHMIR,RAILWAYS,2010


In [75]:
# ============================================================
# CELL 71 — SOURCE QUALITY STATUS
# ============================================================

for filename, df in cleaned_district_data.items():

    # Default status
    df["SOURCE_QUALITY_STATUS"] = "VALID"

    # --------------------------------------------------------
    # Exact duplicate information
    # --------------------------------------------------------

    # Re-check duplicate keys
    key_columns = [
        "STATE",
        "DISTRICT",
        "YEAR"
    ]

    duplicate_key_mask = df.duplicated(
        subset=key_columns,
        keep=False
    )

    # --------------------------------------------------------
    # Conflicting keys
    # --------------------------------------------------------

    conflict_mask = pd.Series(
        False,
        index=df.index
    )

    for _, conflict in conflicting_keys[
        conflicting_keys["filename"]
        == filename
    ].iterrows():

        match = (
            df["STATE"].astype("string")
            .eq(
                str(
                    conflict["state"]
                )
            )
            &
            df["DISTRICT"].astype("string")
            .eq(
                str(
                    conflict["unit_name"]
                )
            )
            &
            df["YEAR"].eq(
                conflict["year"]
            )
        )

        conflict_mask |= match

    df.loc[
        duplicate_key_mask,
        "SOURCE_QUALITY_STATUS"
    ] = "DUPLICATE_KEY"

    df.loc[
        conflict_mask,
        "SOURCE_QUALITY_STATUS"
    ] = "CONFLICTING_SOURCE_RECORD"

print(
    "✓ Source quality status assigned."
)

✓ Source quality status assigned.


In [76]:
# ============================================================
# CELL 72 — BUILD CORE DISTRICT DATA
# ============================================================

core_district_data = {}

excluded_district_records = []

for filename, df in cleaned_district_data.items():

    # --------------------------------------------------------
    # Core modelling population
    # --------------------------------------------------------

    core_mask = (
        df["UNIT_TYPE"]
        == "DISTRICT_OR_REPORTING_UNIT"
    )

    valid_mask = (
        df["SOURCE_QUALITY_STATUS"]
        == "VALID"
    )

    final_mask = (
        core_mask
        &
        valid_mask
    )

    core_df = df[
        final_mask
    ].copy()

    excluded_df = df[
        ~final_mask
    ].copy()

    core_district_data[
        filename
    ] = core_df

    excluded_district_records.append({

        "filename":
            filename,

        "total_rows":
            len(df),

        "core_rows":
            len(core_df),

        "excluded_rows":
            len(excluded_df),

        "aggregate_rows":
            int(
                (
                    df["UNIT_TYPE"]
                    == "AGGREGATE"
                ).sum()
            ),

        "special_unit_rows":
            int(
                (
                    df["UNIT_TYPE"]
                    == "SPECIAL_UNIT"
                ).sum()
            ),

        "conflicting_rows":
            int(
                (
                    df[
                        "SOURCE_QUALITY_STATUS"
                    ]
                    == "CONFLICTING_SOURCE_RECORD"
                ).sum()
            )
    })


core_exclusion_summary = pd.DataFrame(
    excluded_district_records
)

display(
    core_exclusion_summary
)

,filename,total_rows,core_rows,excluded_rows,aggregate_rows,special_unit_rows,conflicting_rows
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,9017,8132,885,408,477,2
1,01_District_wise_crimes_committed_IPC_2013.csv,823,770,53,0,53,0
2,01_District_wise_crimes_committed_IPC_2014.csv,838,729,109,36,73,0
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,9018,8118,900,421,477,0
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,823,735,88,35,53,0
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,837,728,109,36,73,0
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,9018,8118,900,421,477,0
7,02_District_wise_crimes_committed_against_ST_2013.csv,823,770,53,0,53,0
8,02_District_wise_crimes_committed_against_ST_2014.csv,837,728,109,36,73,0
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,9015,8129,886,412,474,0


In [77]:
# ============================================================
# CELL 73 — CORE DATA RETENTION AUDIT
# ============================================================

original_total = sum(
    len(df)
    for df in normalized_district_data.values()
)

cleaned_total = sum(
    len(df)
    for df in cleaned_district_data.values()
)

core_total = sum(
    len(df)
    for df in core_district_data.values()
)

print("=" * 80)
print("DATA RETENTION AUDIT")
print("=" * 80)

print(
    f"\nOriginal normalized rows : "
    f"{original_total:,}"
)

print(
    f"After exact dedup        : "
    f"{cleaned_total:,}"
)

print(
    f"Core district rows       : "
    f"{core_total:,}"
)

print(
    f"Rows excluded from core  : "
    f"{cleaned_total - core_total:,}"
)

DATA RETENTION AUDIT

Original normalized rows : 52,549
After exact dedup        : 52,549
Core district rows       : 47,322
Rows excluded from core  : 5,227


In [78]:
# ============================================================
# CELL 74 — CORE KEY UNIQUENESS CHECK
# ============================================================

core_key_results = []

for filename, df in core_district_data.items():

    key_columns = [
        "STATE",
        "UNIT_NAME",
        "YEAR"
    ]

    duplicate_count = (
        df.duplicated(
            subset=key_columns,
            keep=False
        )
        .sum()
    )

    core_key_results.append({

        "filename":
            filename,

        "core_rows":
            len(df),

        "duplicate_core_key_rows":
            int(duplicate_count),

        "core_key_unique":
            duplicate_count == 0
    })


core_key_audit = pd.DataFrame(
    core_key_results
)

display(
    core_key_audit
)

,filename,core_rows,duplicate_core_key_rows,core_key_unique
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,8132,0,True
1,01_District_wise_crimes_committed_IPC_2013.csv,770,0,True
2,01_District_wise_crimes_committed_IPC_2014.csv,729,0,True
3,02_01_District_wise_crimes_committed_against_SC_2001_2012.csv,8118,0,True
4,02_01_District_wise_crimes_committed_against_SC_2013.csv,735,0,True
5,02_01_District_wise_crimes_committed_against_SC_2014.csv,728,0,True
6,02_District_wise_crimes_committed_against_ST_2001_2012.csv,8118,0,True
7,02_District_wise_crimes_committed_against_ST_2013.csv,770,0,True
8,02_District_wise_crimes_committed_against_ST_2014.csv,728,0,True
9,03_District_wise_crimes_committed_against_children_2001_2012.csv,8129,0,True


In [79]:
# ============================================================
# CELL 75 — SAVE CLEANED DISTRICT SOURCES
# ============================================================

cleaned_dir = (
    PROCESSED_DATA_DIR /
    "district_cleaned"
)

cleaned_dir.mkdir(
    parents=True,
    exist_ok=True
)

for filename, df in cleaned_district_data.items():

    output_path = (
        cleaned_dir /
        filename
    )

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

print("=" * 80)
print("CLEANED DISTRICT SOURCES SAVED")
print("=" * 80)

print(
    cleaned_dir.resolve()
)

CLEANED DISTRICT SOURCES SAVED
D:\Major_Project\Crime_Analysis\data\processed\district_cleaned


In [80]:
# ============================================================
# CELL 76 — SAVE CORE MODELLING SOURCES
# ============================================================

core_dir = (
    PROCESSED_DATA_DIR /
    "district_core"
)

core_dir.mkdir(
    parents=True,
    exist_ok=True
)

for filename, df in core_district_data.items():

    output_path = (
        core_dir /
        filename
    )

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

print("=" * 80)
print("CORE DISTRICT SOURCES SAVED")
print("=" * 80)

print(
    core_dir.resolve()
)

CORE DISTRICT SOURCES SAVED
D:\Major_Project\Crime_Analysis\data\processed\district_core


In [81]:
# ============================================================
# CELL 77 — PREPARE CORE DATASETS FOR INTEGRATION
# ============================================================

integration_tables = {}

integration_key = [
    "STATE",
    "UNIT_NAME",
    "YEAR"
]

for filename, df in core_district_data.items():

    temp = df.copy()

    # --------------------------------------------------------
    # Keep the integration key
    # --------------------------------------------------------

    keep_key = [
        column
        for column in integration_key
        if column in temp.columns
    ]

    # --------------------------------------------------------
    # Identify crime-measure columns
    # --------------------------------------------------------

    excluded_columns = {
        "STATE",
        "DISTRICT",
        "UNIT_NAME",
        "YEAR",
        "CRIME_GROUP",
        "SOURCE_FILE",
        "SOURCE_ROW_ID",
        "SOURCE_RECORD_ID",
        "DUPLICATE_KEY_FLAG",
        "DATA_QUALITY_FLAG",
        "SOURCE_QUALITY_STATUS",
        "UNIT_TYPE"
    }

    measure_columns = [
        column
        for column in temp.columns
        if column not in excluded_columns
    ]

    # --------------------------------------------------------
    # Add crime-group prefix to every measure
    # --------------------------------------------------------

    crime_group = (
        temp["CRIME_GROUP"]
        .iloc[0]
    )

    rename_map = {}

    for column in measure_columns:

        rename_map[column] = (
            f"{crime_group}_{column}"
        )

    temp = temp[
        keep_key + measure_columns
    ].rename(
        columns=rename_map
    )

    # --------------------------------------------------------
    # Store prepared table
    # --------------------------------------------------------

    integration_tables[
        filename
    ] = temp

print("=" * 80)
print("INTEGRATION TABLES PREPARED")
print("=" * 80)

print(
    f"\nTables prepared: "
    f"{len(integration_tables)}"
)

for filename, df in integration_tables.items():

    print(
        f"{filename}: "
        f"{df.shape[0]:,} rows × "
        f"{df.shape[1]:,} columns"
    )

INTEGRATION TABLES PREPARED

Tables prepared: 14
01_District_wise_crimes_committed_IPC_2001_2012.csv: 8,132 rows × 33 columns
01_District_wise_crimes_committed_IPC_2013.csv: 770 rows × 33 columns
01_District_wise_crimes_committed_IPC_2014.csv: 729 rows × 91 columns
02_01_District_wise_crimes_committed_against_SC_2001_2012.csv: 8,118 rows × 13 columns
02_01_District_wise_crimes_committed_against_SC_2013.csv: 735 rows × 13 columns
02_01_District_wise_crimes_committed_against_SC_2014.csv: 728 rows × 66 columns
02_District_wise_crimes_committed_against_ST_2001_2012.csv: 8,118 rows × 13 columns
02_District_wise_crimes_committed_against_ST_2013.csv: 770 rows × 13 columns
02_District_wise_crimes_committed_against_ST_2014.csv: 728 rows × 66 columns
03_District_wise_crimes_committed_against_children_2001_2012.csv: 8,129 rows × 15 columns
03_District_wise_crimes_committed_against_children_2013.csv: 735 rows × 16 columns
42_District_wise_crimes_committed_against_women_2001_2012.csv: 8,132 rows × 

In [82]:
# ============================================================
# CELL 78 — CHECK FEATURE NAME COLLISIONS
# ============================================================

all_measure_columns = []

for filename, df in integration_tables.items():

    measure_columns = [
        column
        for column in df.columns
        if column not in integration_key
    ]

    all_measure_columns.extend(
        measure_columns
    )

column_counts = (
    pd.Series(
        all_measure_columns
    )
    .value_counts()
)

column_collisions = (
    column_counts[
        column_counts > 1
    ]
)

print("=" * 80)
print("FEATURE NAME COLLISION CHECK")
print("=" * 80)

if len(column_collisions) == 0:

    print(
        "\n✓ No feature-name collisions."
    )

else:

    print(
        "\n⚠ Feature-name collisions found:"
    )

    display(
        column_collisions
    )

FEATURE NAME COLLISION CHECK

⚠ Feature-name collisions found:


WOMEN_Cruelty by Husband or his Relatives        3
WOMEN_Dowry Deaths                               3
WOMEN_Rape                                       3
CHILDREN_Procuration of minor girls              2
SC_Murder                                        2
                                                ..
IPC_KIDNAPPING & ABDUCTION                       2
IPC_DACOITY                                      2
IPC_CUSTODIAL RAPE                               2
IPC_RAPE                                         2
IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER    2
Name: count, Length: 68, dtype: int64

In [83]:
# ============================================================
# CELL 79 — COMBINE FILES WITHIN EACH CRIME GROUP
# ============================================================

group_tables = {}

for filename, df in integration_tables.items():

    crime_group = (
        core_district_data[
            filename
        ]["CRIME_GROUP"]
        .iloc[0]
    )

    if crime_group not in group_tables:

        group_tables[
            crime_group
        ] = []

    group_tables[
        crime_group
    ].append(
        df
    )


combined_group_tables = {}

for crime_group, tables in group_tables.items():

    combined = pd.concat(
        tables,
        axis=0,
        ignore_index=True
    )

    combined_group_tables[
        crime_group
    ] = combined


print("=" * 80)
print("COMBINED CRIME GROUP TABLES")
print("=" * 80)

for crime_group, df in (
    combined_group_tables.items()
):

    print(
        f"{crime_group}: "
        f"{len(df):,} rows × "
        f"{len(df.columns):,} columns"
    )

COMBINED CRIME GROUP TABLES
IPC: 9,631 rows × 121 columns
SC: 9,581 rows × 76 columns
ST: 9,616 rows × 76 columns
CHILDREN: 8,864 rows × 17 columns
WOMEN: 9,630 rows × 66 columns


In [84]:
# ============================================================
# CELL 80 — GROUP KEY UNIQUENESS
# ============================================================

group_key_audit = []

for crime_group, df in (
    combined_group_tables.items()
):

    duplicate_count = (
        df.duplicated(
            subset=integration_key,
            keep=False
        )
        .sum()
    )

    group_key_audit.append({

        "crime_group":
            crime_group,

        "rows":
            len(df),

        "duplicate_key_rows":
            int(duplicate_count),

        "unique_keys":
            df[
                integration_key
            ].drop_duplicates().shape[0],

        "key_unique":
            duplicate_count == 0
    })


group_key_audit_df = pd.DataFrame(
    group_key_audit
)

display(
    group_key_audit_df
)

,crime_group,rows,duplicate_key_rows,unique_keys,key_unique
0,IPC,9631,0,9631,True
1,SC,9581,0,9581,True
2,ST,9616,0,9616,True
3,CHILDREN,8864,0,8864,True
4,WOMEN,9630,0,9630,True


In [85]:
# ============================================================
# CELL 81 — HORIZONTAL INTEGRATION
# ============================================================

integrated_df = None

merge_order = [
    "IPC",
    "SC",
    "ST",
    "CHILDREN",
    "WOMEN"
]

for crime_group in merge_order:

    if crime_group not in combined_group_tables:
        continue

    group_df = (
        combined_group_tables[
            crime_group
        ]
        .copy()
    )

    if integrated_df is None:

        integrated_df = group_df

    else:

        integrated_df = integrated_df.merge(
            group_df,
            on=integration_key,
            how="outer",
            validate="one_to_one"
        )

    print(
        f"After {crime_group}: "
        f"{integrated_df.shape[0]:,} rows × "
        f"{integrated_df.shape[1]:,} columns"
    )

After IPC: 9,631 rows × 121 columns
After SC: 9,811 rows × 194 columns
After ST: 9,811 rows × 267 columns
After CHILDREN: 9,829 rows × 281 columns
After WOMEN: 9,856 rows × 344 columns


In [86]:
# ============================================================
# CELL 82 — INTEGRATION COVERAGE
# ============================================================

coverage_records = []

for crime_group in merge_order:

    if crime_group not in combined_group_tables:
        continue

    group_df = combined_group_tables[
        crime_group
    ]

    group_keys = set(
        map(
            tuple,
            group_df[
                integration_key
            ]
            .drop_duplicates()
            .values
        )
    )

    integrated_keys = set(
        map(
            tuple,
            integrated_df[
                integration_key
            ]
            .drop_duplicates()
            .values
        )
    )

    coverage_records.append({

        "crime_group":
            crime_group,

        "source_keys":
            len(group_keys),

        "integrated_keys":
            len(integrated_keys),

        "coverage_pct":
            round(
                100 *
                len(
                    group_keys &
                    integrated_keys
                )
                /
                len(integrated_keys),
                2
            )
    })


coverage_df = pd.DataFrame(
    coverage_records
)

display(
    coverage_df
)

,crime_group,source_keys,integrated_keys,coverage_pct
0,IPC,9631,9856,97.72
1,SC,9581,9856,97.21
2,ST,9616,9856,97.56
3,CHILDREN,8864,9856,89.94
4,WOMEN,9630,9856,97.71


In [87]:
# ============================================================
# CELL 83 — GROUP-LEVEL MISSINGNESS
# ============================================================

missingness_records = []

for crime_group in merge_order:

    prefix = f"{crime_group}_"

    group_columns = [
        column
        for column in integrated_df.columns
        if column.startswith(prefix)
    ]

    if not group_columns:
        continue

    missing_cells = (
        integrated_df[
            group_columns
        ]
        .isna()
        .sum()
        .sum()
    )

    total_cells = (
        integrated_df.shape[0]
        *
        len(group_columns)
    )

    missingness_records.append({

        "crime_group":
            crime_group,

        "feature_count":
            len(group_columns),

        "missing_cells":
            int(missing_cells),

        "missing_pct":
            round(
                100 *
                missing_cells /
                total_cells,
                2
            )
    })


missingness_by_group = pd.DataFrame(
    missingness_records
)

display(
    missingness_by_group
)

,crime_group,feature_count,missing_cells,missing_pct
0,IPC,118,831796,71.52
1,SC,73,585094,81.32
2,ST,73,584744,81.27
3,CHILDREN,14,30991,22.46
4,WOMEN,63,515662,83.05


In [88]:
# ============================================================
# CELL 85 — SAVE INTEGRATED DISTRICT DATASET
# ============================================================

integrated_output = (
    PROCESSED_DATA_DIR /
    "district_integrated_raw.csv"
)

integrated_df.to_csv(
    integrated_output,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("INTEGRATED DISTRICT DATASET SAVED")
print("=" * 80)

print(
    f"\nPath:"
)

print(
    integrated_output.resolve()
)

print(
    f"\nShape:"
    f" {integrated_df.shape[0]:,} rows × "
    f"{integrated_df.shape[1]:,} columns"
)

INTEGRATED DISTRICT DATASET SAVED

Path:
D:\Major_Project\Crime_Analysis\data\processed\district_integrated_raw.csv

Shape: 9,856 rows × 344 columns


In [89]:
# ============================================================
# CELL 86 — FINAL INTEGRATION AUDIT
# ============================================================

print("=" * 80)
print("FINAL DISTRICT INTEGRATION AUDIT")
print("=" * 80)

print(
    f"\nIntegrated rows    : "
    f"{len(integrated_df):,}"
)

print(
    f"Integrated columns : "
    f"{len(integrated_df.columns):,}"
)

print(
    f"Unique keys        : "
    f"{integrated_df[integration_key].drop_duplicates().shape[0]:,}"
)

print(
    f"Duplicate keys     : "
    f"{integrated_df.duplicated(integration_key).sum():,}"
)

print(
    f"Missing cells      : "
    f"{integrated_df.isna().sum().sum():,}"
)

print(
    f"Memory usage       : "
    f"{integrated_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print("\n✓ Integration completed.")

FINAL DISTRICT INTEGRATION AUDIT

Integrated rows    : 9,856
Integrated columns : 344
Unique keys        : 9,856
Duplicate keys     : 0
Missing cells      : 2,548,287
Memory usage       : 26.96 MB

✓ Integration completed.


In [90]:
# ============================================================
# CELL 87 — STRUCTURAL VS WITHIN-GROUP MISSINGNESS
# ============================================================

print("=" * 80)
print("STRUCTURAL VS WITHIN-GROUP MISSINGNESS")
print("=" * 80)

missingness_diagnosis = []

for crime_group in merge_order:

    if crime_group not in combined_group_tables:
        continue

    group_df = combined_group_tables[
        crime_group
    ].copy()

    # --------------------------------------------------------
    # Keys that actually exist in this crime group
    # --------------------------------------------------------

    group_keys = (
        group_df[
            integration_key
        ]
        .drop_duplicates()
    )

    group_key_set = set(
        map(
            tuple,
            group_keys.values
        )
    )

    # --------------------------------------------------------
    # Features belonging to this group
    # --------------------------------------------------------

    prefix = f"{crime_group}_"

    group_columns = [
        column
        for column in integrated_df.columns
        if column.startswith(prefix)
    ]

    # --------------------------------------------------------
    # Determine whether each integrated row has a source
    # record for this crime group.
    # --------------------------------------------------------

    row_keys = list(
        map(
            tuple,
            integrated_df[
                integration_key
            ].values
        )
    )

    group_present = pd.Series(
        [
            key in group_key_set
            for key in row_keys
        ],
        index=integrated_df.index
    )

    group_absent_rows = (
        ~group_present
    ).sum()

    group_present_rows = (
        group_present
    ).sum()

    # --------------------------------------------------------
    # Missing values among rows WHERE THE GROUP EXISTS
    # --------------------------------------------------------

    if group_present_rows > 0:

        present_data = integrated_df.loc[
            group_present,
            group_columns
        ]

        within_group_missing_cells = (
            present_data
            .isna()
            .sum()
            .sum()
        )

        total_present_cells = (
            present_data.shape[0]
            *
            present_data.shape[1]
        )

        within_group_missing_pct = (
            100
            *
            within_group_missing_cells
            /
            total_present_cells
        )

    else:

        within_group_missing_cells = 0
        within_group_missing_pct = 0

    # --------------------------------------------------------
    # Structural missingness
    # --------------------------------------------------------

    structural_missing_cells = (
        group_absent_rows
        *
        len(group_columns)
    )

    total_integrated_cells = (
        len(integrated_df)
        *
        len(group_columns)
    )

    structural_missing_pct = (
        100
        *
        structural_missing_cells
        /
        total_integrated_cells
    )

    missingness_diagnosis.append({

        "crime_group":
            crime_group,

        "integrated_rows":
            len(integrated_df),

        "group_present_rows":
            int(group_present_rows),

        "group_absent_rows":
            int(group_absent_rows),

        "feature_count":
            len(group_columns),

        "structural_missing_pct":
            round(
                structural_missing_pct,
                2
            ),

        "within_group_missing_pct":
            round(
                within_group_missing_pct,
                2
            )
    })


missingness_diagnosis_df = pd.DataFrame(
    missingness_diagnosis
)

display(
    missingness_diagnosis_df
)

STRUCTURAL VS WITHIN-GROUP MISSINGNESS


,crime_group,integrated_rows,group_present_rows,group_absent_rows,feature_count,structural_missing_pct,within_group_missing_pct
0,IPC,9856,9631,225,118,2.28,70.86
1,SC,9856,9581,275,73,2.79,80.78
2,ST,9856,9616,240,73,2.44,80.80
3,CHILDREN,9856,8864,992,14,10.06,13.78
4,WOMEN,9856,9630,226,63,2.29,82.65


In [91]:
# ============================================================
# CELL 88 — CRIME GROUP COVERAGE BY YEAR
# ============================================================

year_coverage_records = []

for crime_group in merge_order:

    if crime_group not in combined_group_tables:
        continue

    group_df = combined_group_tables[
        crime_group
    ]

    coverage = (
        group_df
        .groupby("YEAR")
        .size()
        .reset_index(
            name="source_rows"
        )
    )

    coverage[
        "crime_group"
    ] = crime_group

    year_coverage_records.append(
        coverage
    )


year_coverage_df = pd.concat(
    year_coverage_records,
    ignore_index=True
)

year_coverage_df = (
    year_coverage_df[
        [
            "crime_group",
            "YEAR",
            "source_rows"
        ]
    ]
    .sort_values(
        [
            "YEAR",
            "crime_group"
        ]
    )
)

display(
    year_coverage_df
)

,crime_group,YEAR,source_rows
42,CHILDREN,2001,651
0,IPC,2001,651
14,SC,2001,648
28,ST,2001,648
55,WOMEN,2001,651
...,...,...,...
67,WOMEN,2013,770
13,IPC,2014,729
27,SC,2014,728
41,ST,2014,728


In [92]:
# ============================================================
# CELL 89 — CRIME GROUP AVAILABILITY MATRIX
# ============================================================

group_availability = integrated_df[
    integration_key
].copy()

for crime_group in merge_order:

    if crime_group not in combined_group_tables:
        continue

    group_keys = set(
        map(
            tuple,
            combined_group_tables[
                crime_group
            ][integration_key]
            .drop_duplicates()
            .values
        )
    )

    row_keys = list(
        map(
            tuple,
            integrated_df[
                integration_key
            ].values
        )
    )

    group_availability[
        f"{crime_group}_AVAILABLE"
    ] = [
        key in group_keys
        for key in row_keys
    ]


display(
    group_availability.head(20)
)

,STATE,UNIT_NAME,YEAR,IPC_AVAILABLE,SC_AVAILABLE,ST_AVAILABLE,CHILDREN_AVAILABLE,WOMEN_AVAILABLE
0,ANDHRA PRADESH,ADILABAD,2001,True,True,True,True,True
1,ANDHRA PRADESH,ANANTAPUR,2001,True,True,True,True,True
2,ANDHRA PRADESH,CHITTOOR,2001,True,True,True,True,True
3,ANDHRA PRADESH,CUDDAPAH,2001,True,True,True,True,True
4,ANDHRA PRADESH,EAST GODAVARI,2001,True,True,True,True,True
5,ANDHRA PRADESH,GUNTAKAL RLY.,2001,True,True,True,True,True
6,ANDHRA PRADESH,GUNTUR,2001,True,True,True,True,True
7,ANDHRA PRADESH,KARIMNAGAR,2001,True,True,True,True,True
8,ANDHRA PRADESH,KHAMMAM,2001,True,True,True,True,True
9,ANDHRA PRADESH,KRISHNA,2001,True,True,True,True,True


In [93]:
# ============================================================
# CELL 90 — AVAILABILITY COMBINATIONS
# ============================================================

availability_columns = [
    f"{group}_AVAILABLE"
    for group in merge_order
]

availability_patterns = (
    group_availability[
        availability_columns
    ]
    .astype(int)
    .astype(str)
    .agg(
        "".join,
        axis=1
    )
)

availability_summary = (
    availability_patterns
    .value_counts()
    .reset_index()
)

availability_summary.columns = [
    "availability_pattern",
    "row_count"
]

display(
    availability_summary
)

,availability_pattern,row_count
0,11111,8664
1,11101,737
2,10000,184
3,01111,151
4,10101,35
5,01110,28
6,00001,27
7,00010,13
8,10001,8
9,00011,5


In [96]:
# ============================================================
# CELL 93 — ADD SOURCE AVAILABILITY FLAGS
# ============================================================

for column in availability_columns:

    integrated_df[column] = (
        group_availability[column]
        .values
    )

print(
    "✓ Crime-group availability flags added."
)

display(
    integrated_df[
        integration_key
        + availability_columns
    ].head(10)
)

✓ Crime-group availability flags added.


,STATE,UNIT_NAME,YEAR,IPC_AVAILABLE,SC_AVAILABLE,ST_AVAILABLE,CHILDREN_AVAILABLE,WOMEN_AVAILABLE
0,ANDHRA PRADESH,ADILABAD,2001,True,True,True,True,True
1,ANDHRA PRADESH,ANANTAPUR,2001,True,True,True,True,True
2,ANDHRA PRADESH,CHITTOOR,2001,True,True,True,True,True
3,ANDHRA PRADESH,CUDDAPAH,2001,True,True,True,True,True
4,ANDHRA PRADESH,EAST GODAVARI,2001,True,True,True,True,True
5,ANDHRA PRADESH,GUNTAKAL RLY.,2001,True,True,True,True,True
6,ANDHRA PRADESH,GUNTUR,2001,True,True,True,True,True
7,ANDHRA PRADESH,KARIMNAGAR,2001,True,True,True,True,True
8,ANDHRA PRADESH,KHAMMAM,2001,True,True,True,True,True
9,ANDHRA PRADESH,KRISHNA,2001,True,True,True,True,True


In [94]:
# ============================================================
# CELL 91 — WITHIN-SOURCE MISSING VALUES
# ============================================================

within_source_missing_records = []

for crime_group, df in combined_group_tables.items():

    feature_columns = [
        column
        for column in df.columns
        if column not in integration_key
    ]

    for column in feature_columns:

        missing_count = int(
            df[column].isna().sum()
        )

        within_source_missing_records.append({

            "crime_group":
                crime_group,

            "feature":
                column,

            "rows":
                len(df),

            "missing_count":
                missing_count,

            "missing_pct":
                round(
                    100
                    *
                    missing_count
                    /
                    len(df),
                    2
                )
        })


within_source_missing_df = pd.DataFrame(
    within_source_missing_records
)

display(
    within_source_missing_df
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .head(50)
)

,crime_group,feature,rows,missing_count,missing_pct
340,WOMEN,WOMEN_Total Crimes against Women,9630,8902,92.44
312,WOMEN,WOMEN_Culpable Homicide not amounting to Murder,9630,8902,92.44
285,WOMEN,WOMEN_Custodial Rape,9630,8902,92.44
286,WOMEN,WOMEN_Custodial_Gang Rape,9630,8902,92.44
287,WOMEN,WOMEN_Custodial_Other Rape,9630,8902,92.44
288,WOMEN,WOMEN_Rape other than Custodial,9630,8902,92.44
289,WOMEN,WOMEN_Rape_Gang Rape,9630,8902,92.44
290,WOMEN,WOMEN_Rape_Others,9630,8902,92.44
291,WOMEN,WOMEN_Attempt to commit Rape,9630,8902,92.44
292,WOMEN,WOMEN_Kidnapping & Abduction_Total,9630,8902,92.44


In [95]:
# ============================================================
# CELL 92 — WITHIN-SOURCE MISSINGNESS SUMMARY
# ============================================================

within_source_summary = (
    within_source_missing_df
    .groupby("crime_group")
    .agg(
        features_with_missing_values=(
            "missing_count",
            lambda x: int(
                (x > 0).sum()
            )
        ),

        total_missing_cells=(
            "missing_count",
            "sum"
        ),

        maximum_feature_missing_pct=(
            "missing_pct",
            "max"
        ),

        average_feature_missing_pct=(
            "missing_pct",
            "mean"
        )
    )
    .reset_index()
)

within_source_summary[
    "average_feature_missing_pct"
] = within_source_summary[
    "average_feature_missing_pct"
].round(2)

display(
    within_source_summary
)

,crime_group,features_with_missing_values,total_missing_cells,maximum_feature_missing_pct,average_feature_missing_pct
0,CHILDREN,13,17103,91.71,13.78
1,IPC,118,805246,92.43,70.86
2,SC,73,565019,92.40,80.78
3,ST,73,567224,92.43,80.81
4,WOMEN,60,501424,92.44,82.65


In [97]:
# ============================================================
# CELL 94 — FEATURE AVAILABILITY ACROSS SOURCE FILES
# ============================================================

feature_availability_records = []

for filename, df in integration_tables.items():

    # Get the crime group
    crime_group = (
        core_district_data[
            filename
        ]["CRIME_GROUP"]
        .iloc[0]
    )

    # Features actually present in this source file
    source_features = [
        column
        for column in df.columns
        if column not in integration_key
    ]

    for feature in source_features:

        feature_availability_records.append({

            "filename":
                filename,

            "crime_group":
                crime_group,

            "feature":
                feature,

            "source_rows":
                len(df),

            "missing_values_in_source":
                int(
                    df[feature]
                    .isna()
                    .sum()
                ),

            "missing_pct_in_source":
                round(
                    100
                    * df[feature].isna().mean(),
                    2
                )
        })


feature_availability_df = pd.DataFrame(
    feature_availability_records
)

print("=" * 80)
print("FEATURE AVAILABILITY ACROSS SOURCE FILES")
print("=" * 80)

display(
    feature_availability_df.head(50)
)

FEATURE AVAILABILITY ACROSS SOURCE FILES


,filename,crime_group,feature,source_rows,missing_values_in_source,missing_pct_in_source
0,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_MURDER,8132,0,0.0
1,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_ATTEMPT TO MURDER,8132,0,0.0
2,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,8132,0,0.0
3,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_RAPE,8132,0,0.0
4,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_CUSTODIAL RAPE,8132,0,0.0
5,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_OTHER RAPE,8132,0,0.0
6,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_KIDNAPPING & ABDUCTION,8132,0,0.0
7,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,8132,0,0.0
8,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_KIDNAPPING AND ABDUCTION OF OTHERS,8132,0,0.0
9,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC,IPC_DACOITY,8132,0,0.0


In [98]:
# ============================================================
# CELL 95 — SCHEMA EVOLUTION AUDIT
# ============================================================

schema_evolution_records = []

for crime_group in merge_order:

    group_data = (
        feature_availability_df[
            feature_availability_df[
                "crime_group"
            ]
            == crime_group
        ]
    )

    if group_data.empty:
        continue

    feature_file_counts = (
        group_data
        .groupby("feature")["filename"]
        .nunique()
        .reset_index(
            name="source_file_count"
        )
    )

    total_files = (
        group_data["filename"]
        .nunique()
    )

    feature_file_counts[
        "total_group_files"
    ] = total_files

    feature_file_counts[
        "availability_pct"
    ] = (
        100
        * feature_file_counts[
            "source_file_count"
        ]
        /
        total_files
    ).round(2)

    feature_file_counts[
        "crime_group"
    ] = crime_group

    schema_evolution_records.append(
        feature_file_counts
    )


schema_evolution_df = pd.concat(
    schema_evolution_records,
    ignore_index=True
)

display(
    schema_evolution_df
    .sort_values(
        [
            "crime_group",
            "availability_pct"
        ]
    )
)

,feature,source_file_count,total_group_files,availability_pct,crime_group
268,CHILDREN_Infanticid,1,2,50.00,CHILDREN
270,CHILDREN_Murder,1,2,50.00,CHILDREN
272,CHILDREN_Other murder,1,2,50.00,CHILDREN
264,CHILDREN_Abetment of suicide,2,2,100.00,CHILDREN
265,CHILDREN_Buying of girls for prostitution,2,2,100.00,CHILDREN
...,...,...,...,...,...
316,WOMEN_Insult to modesty of Women,2,3,66.67,WOMEN
323,WOMEN_Kidnapping and Abduction,2,3,66.67,WOMEN
291,WOMEN_Cruelty by Husband or his Relatives,3,3,100.00,WOMEN
299,WOMEN_Dowry Deaths,3,3,100.00,WOMEN


In [99]:
# ============================================================
# CELL 96 — SCHEMA ABSENCE VS REAL SOURCE NaN
# ============================================================

schema_missingness_records = []

for crime_group in merge_order:

    group_files = [
        filename
        for filename in integration_tables
        if (
            core_district_data[
                filename
            ]["CRIME_GROUP"].iloc[0]
            == crime_group
        )
    ]

    for filename in group_files:

        df = integration_tables[
            filename
        ]

        for feature in df.columns:

            if feature in integration_key:
                continue

            schema_missingness_records.append({

                "crime_group":
                    crime_group,

                "filename":
                    filename,

                "feature":
                    feature,

                "source_rows":
                    len(df),

                "source_nan_count":
                    int(
                        df[feature]
                        .isna()
                        .sum()
                    ),

                "source_nan_pct":
                    round(
                        100
                        * df[feature].isna().mean(),
                        2
                    )
            })


source_nan_audit = pd.DataFrame(
    schema_missingness_records
)

display(
    source_nan_audit
    .sort_values(
        "source_nan_pct",
        ascending=False
    )
    .head(50)
)

,crime_group,filename,feature,source_rows,source_nan_count,source_nan_pct
320,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Procuration of minor girls,8129,10,0.12
315,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Rape,8129,10,0.12
323,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Prohibition of child marriage act,8129,10,0.12
322,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Selling of girls for prostitution,8129,10,0.12
321,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Buying of girls for prostitution,8129,10,0.12
319,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Exposure and abandonment,8129,10,0.12
318,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Abetment of suicide,8129,10,0.12
317,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Foeticide,8129,10,0.12
316,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Kidnapping and Abduction,8129,10,0.12
314,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Murder,8129,10,0.12


In [100]:
# ============================================================
# CELL 97 — GENUINE SOURCE-LEVEL MISSINGNESS
# ============================================================

genuine_missing_summary = (
    source_nan_audit[
        source_nan_audit[
            "source_nan_count"
        ] > 0
    ]
    .sort_values(
        "source_nan_pct",
        ascending=False
    )
)

print("=" * 80)
print("SOURCE-LEVEL MISSING VALUES")
print("=" * 80)

display(
    genuine_missing_summary.head(100)
)

SOURCE-LEVEL MISSING VALUES


,crime_group,filename,feature,source_rows,source_nan_count,source_nan_pct
314,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Murder,8129,10,0.12
315,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Rape,8129,10,0.12
316,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Kidnapping and Abduction,8129,10,0.12
317,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Foeticide,8129,10,0.12
318,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Abetment of suicide,8129,10,0.12
319,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Exposure and abandonment,8129,10,0.12
320,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Procuration of minor girls,8129,10,0.12
321,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Buying of girls for prostitution,8129,10,0.12
322,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Selling of girls for prostitution,8129,10,0.12
323,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Prohibition of child marriage act,8129,10,0.12


In [101]:

# ============================================================
# CELL 98 — SAMPLE ACTUAL SOURCE NaN RECORDS
# ============================================================

for _, row in (
    genuine_missing_summary
    .head(10)
    .iterrows()
):

    filename = row["filename"]
    feature = row["feature"]

    df = integration_tables[
        filename
    ]

    missing_rows = df[
        df[feature].isna()
    ]

    if missing_rows.empty:
        continue

    print("\n" + "=" * 90)
    print(
        f"FILE: {filename}"
    )
    print(
        f"FEATURE: {feature}"
    )
    print(
        f"Missing rows: {len(missing_rows)}"
    )
    print("=" * 90)

    display(
        missing_rows[
            integration_key + [feature]
        ].head(10)
    )


FILE: 03_District_wise_crimes_committed_against_children_2001_2012.csv
FEATURE: CHILDREN_Murder
Missing rows: 10


,STATE,UNIT_NAME,YEAR,CHILDREN_Murder
8256,ASSAM,BAKSA,2012,NaN
8258,ASSAM,BIEO,2012,NaN
8260,ASSAM,C.I.D.,2012,NaN
8262,ASSAM,CHIRANG,2012,NaN
8267,ASSAM,G.R.P.,2012,NaN
8272,ASSAM,HAMREN,2012,NaN
8283,ASSAM,R.P.O.,2012,NaN
8287,ASSAM,UDALGURI,2012,NaN
8472,JHARKHAND,BOKARO,2012,NaN
8712,ODISHA,JHARSUGUDA,2012,NaN



FILE: 03_District_wise_crimes_committed_against_children_2001_2012.csv
FEATURE: CHILDREN_Rape
Missing rows: 10


,STATE,UNIT_NAME,YEAR,CHILDREN_Rape
8256,ASSAM,BAKSA,2012,NaN
8258,ASSAM,BIEO,2012,NaN
8260,ASSAM,C.I.D.,2012,NaN
8262,ASSAM,CHIRANG,2012,NaN
8267,ASSAM,G.R.P.,2012,NaN
8272,ASSAM,HAMREN,2012,NaN
8283,ASSAM,R.P.O.,2012,NaN
8287,ASSAM,UDALGURI,2012,NaN
8472,JHARKHAND,BOKARO,2012,NaN
8712,ODISHA,JHARSUGUDA,2012,NaN



FILE: 03_District_wise_crimes_committed_against_children_2001_2012.csv
FEATURE: CHILDREN_Kidnapping and Abduction
Missing rows: 10


...

,STATE,UNIT_NAME,YEAR,CHILDREN_Buying of girls for prostitution
8256,ASSAM,BAKSA,2012,NaN
8258,ASSAM,BIEO,2012,NaN
8260,ASSAM,C.I.D.,2012,NaN
8262,ASSAM,CHIRANG,2012,NaN
8267,ASSAM,G.R.P.,2012,NaN
8272,ASSAM,HAMREN,2012,NaN
8283,ASSAM,R.P.O.,2012,NaN
8287,ASSAM,UDALGURI,2012,NaN
8472,JHARKHAND,BOKARO,2012,NaN
8712,ODISHA,JHARSUGUDA,2012,NaN



FILE: 03_District_wise_crimes_committed_against_children_2001_2012.csv
FEATURE: CHILDREN_Selling of girls for prostitution
Missing rows: 10


,STATE,UNIT_NAME,YEAR,CHILDREN_Selling of girls for prostitution
8256,ASSAM,BAKSA,2012,NaN
8258,ASSAM,BIEO,2012,NaN
8260,ASSAM,C.I.D.,2012,NaN
8262,ASSAM,CHIRANG,2012,NaN
8267,ASSAM,G.R.P.,2012,NaN
8272,ASSAM,HAMREN,2012,NaN
8283,ASSAM,R.P.O.,2012,NaN
8287,ASSAM,UDALGURI,2012,NaN
8472,JHARKHAND,BOKARO,2012,NaN
8712,ODISHA,JHARSUGUDA,2012,NaN



FILE: 03_District_wise_crimes_committed_against_children_2001_2012.csv
FEATURE: CHILDREN_Prohibition of child marriage act
Missing rows: 10


,STATE,UNIT_NAME,YEAR,CHILDREN_Prohibition of child marriage act
8256,ASSAM,BAKSA,2012,NaN
8258,ASSAM,BIEO,2012,NaN
8260,ASSAM,C.I.D.,2012,NaN
8262,ASSAM,CHIRANG,2012,NaN
8267,ASSAM,G.R.P.,2012,NaN
8272,ASSAM,HAMREN,2012,NaN
8283,ASSAM,R.P.O.,2012,NaN
8287,ASSAM,UDALGURI,2012,NaN
8472,JHARKHAND,BOKARO,2012,NaN
8712,ODISHA,JHARSUGUDA,2012,NaN


In [102]:
# ============================================================
# CELL 99 — FEATURE SCHEMA SUMMARY
# ============================================================

schema_summary_records = []

for crime_group in merge_order:

    group_schema = (
        schema_evolution_df[
            schema_evolution_df[
                "crime_group"
            ]
            == crime_group
        ]
    )

    schema_summary_records.append({

        "crime_group":
            crime_group,

        "source_files":
            group_schema[
                "total_group_files"
            ].max(),

        "unique_features":
            group_schema[
                "feature"
            ].nunique(),

        "features_in_all_files":
            int(
                (
                    group_schema[
                        "availability_pct"
                    ]
                    == 100
                ).sum()
            ),

        "features_in_some_files":
            int(
                (
                    group_schema[
                        "availability_pct"
                    ]
                    < 100
                ).sum()
            )
    })


schema_summary_df = pd.DataFrame(
    schema_summary_records
)

display(
    schema_summary_df
)

,crime_group,source_files,unique_features,features_in_all_files,features_in_some_files
0,IPC,3,118,0,118
1,SC,3,73,0,73
2,ST,3,73,0,73
3,CHILDREN,2,14,11,3
4,WOMEN,3,63,3,60


In [103]:
# ============================================================
# CELL 100 — FEATURE METADATA
# ============================================================

feature_metadata_records = []

for crime_group in merge_order:

    group_schema = (
        schema_evolution_df[
            schema_evolution_df[
                "crime_group"
            ] == crime_group
        ]
        .copy()
    )

    for _, row in group_schema.iterrows():

        feature_metadata_records.append({

            "crime_group":
                crime_group,

            "feature":
                row["feature"],

            "source_file_count":
                int(
                    row["source_file_count"]
                ),

            "total_group_files":
                int(
                    row["total_group_files"]
                ),

            "availability_pct":
                float(
                    row["availability_pct"]
                ),

            "available_in_all_files":
                bool(
                    row["availability_pct"]
                    == 100
                )
        })


feature_metadata_df = pd.DataFrame(
    feature_metadata_records
)

print("=" * 80)
print("FEATURE METADATA")
print("=" * 80)

display(
    feature_metadata_df.head(30)
)

FEATURE METADATA


,crime_group,feature,source_file_count,total_group_files,availability_pct,available_in_all_files
0,IPC,IPC_ARSON,2,3,66.67,False
1,IPC,IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,2,3,66.67,False
2,IPC,IPC_ATTEMPT TO MURDER,2,3,66.67,False
3,IPC,IPC_AUTO THEFT,2,3,66.67,False
4,IPC,IPC_Acid attack,1,3,33.33,False
5,IPC,IPC_Arson,1,3,33.33,False
6,IPC,IPC_Assault on Women with intent to outrage her Modesty,1,3,33.33,False
7,IPC,IPC_Assault or use of criminal force to women with intent to Disrobe,1,3,33.33,False
8,IPC,IPC_At Office premises,1,3,33.33,False
9,IPC,IPC_Attempt to Acid Attack,1,3,33.33,False


In [104]:
# ============================================================
# CELL 101 — FEATURE COVERAGE CLASSIFICATION
# ============================================================

def classify_feature_coverage(
    availability_pct
):

    if availability_pct == 100:
        return "COMMON_ACROSS_ALL_SOURCE_FILES"

    if availability_pct > 0:
        return "SCHEMA_VARIABLE_ACROSS_SOURCE_FILES"

    return "NOT_AVAILABLE"


feature_metadata_df[
    "feature_coverage_type"
] = (
    feature_metadata_df[
        "availability_pct"
    ]
    .apply(
        classify_feature_coverage
    )
)

display(
    feature_metadata_df[
        [
            "crime_group",
            "feature",
            "availability_pct",
            "feature_coverage_type"
        ]
    ]
)

,crime_group,feature,availability_pct,feature_coverage_type
0,IPC,IPC_ARSON,66.67,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
1,IPC,IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,66.67,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
2,IPC,IPC_ATTEMPT TO MURDER,66.67,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
3,IPC,IPC_AUTO THEFT,66.67,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
4,IPC,IPC_Acid attack,33.33,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
...,...,...,...,...
336,WOMEN,WOMEN_Sexual Harassment,33.33,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
337,WOMEN,WOMEN_Stalking,33.33,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
338,WOMEN,WOMEN_Total Crimes against Women,33.33,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES
339,WOMEN,WOMEN_UnNatural Offences,33.33,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES


In [105]:
# ============================================================
# CELL 102 — FEATURE COVERAGE SUMMARY
# ============================================================

feature_coverage_summary = (
    feature_metadata_df
    .groupby(
        [
            "crime_group",
            "feature_coverage_type"
        ]
    )
    .size()
    .reset_index(
        name="feature_count"
    )
)

display(
    feature_coverage_summary
)

,crime_group,feature_coverage_type,feature_count
0,CHILDREN,COMMON_ACROSS_ALL_SOURCE_FILES,11
1,CHILDREN,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES,3
2,IPC,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES,118
3,SC,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES,73
4,ST,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES,73
5,WOMEN,COMMON_ACROSS_ALL_SOURCE_FILES,3
6,WOMEN,SCHEMA_VARIABLE_ACROSS_SOURCE_FILES,60


In [106]:
# ============================================================
# CELL 103 — FEATURE-YEAR AVAILABILITY
# ============================================================

feature_year_records = []

for filename, df in integration_tables.items():

    crime_group = (
        core_district_data[
            filename
        ]["CRIME_GROUP"]
        .iloc[0]
    )

    # Determine years actually present in this source
    years = sorted(
        df["YEAR"]
        .dropna()
        .unique()
    )

    feature_columns = [
        column
        for column in df.columns
        if column not in integration_key
    ]

    for feature in feature_columns:

        for year in years:

            year_data = df[
                df["YEAR"] == year
            ]

            feature_year_records.append({

                "crime_group":
                    crime_group,

                "filename":
                    filename,

                "feature":
                    feature,

                "year":
                    year,

                "rows_for_year":
                    len(year_data),

                "missing_count":
                    int(
                        year_data[
                            feature
                        ].isna().sum()
                    ),

                "feature_present_in_schema":
                    True
            })


feature_year_availability_df = pd.DataFrame(
    feature_year_records
)

display(
    feature_year_availability_df.head(50)
)

,crime_group,filename,feature,year,rows_for_year,missing_count,feature_present_in_schema
0,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2001,651,0,True
1,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2002,654,0,True
2,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2003,660,0,True
3,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2004,661,0,True
4,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2005,665,0,True
5,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2006,667,0,True
6,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2007,669,0,True
7,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2008,685,0,True
8,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2009,690,0,True
9,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,2010,697,0,True


In [107]:
# ============================================================
# CELL 104 — FEATURE TEMPORAL COVERAGE
# ============================================================

feature_temporal_coverage = (
    feature_year_availability_df
    .groupby(
        [
            "crime_group",
            "feature"
        ]
    )
    .agg(
        first_available_year=(
            "year",
            "min"
        ),

        last_available_year=(
            "year",
            "max"
        ),

        years_available=(
            "year",
            "nunique"
        )
    )
    .reset_index()
)

display(
    feature_temporal_coverage
)

,crime_group,feature,first_available_year,last_available_year,years_available
0,CHILDREN,CHILDREN_Abetment of suicide,2001,2013,13
1,CHILDREN,CHILDREN_Buying of girls for prostitution,2001,2013,13
2,CHILDREN,CHILDREN_Exposure and abandonment,2001,2013,13
3,CHILDREN,CHILDREN_Foeticide,2001,2013,13
4,CHILDREN,CHILDREN_Infanticid,2013,2013,1
...,...,...,...,...,...
336,WOMEN,WOMEN_Sexual Harassment,2014,2014,1
337,WOMEN,WOMEN_Stalking,2014,2014,1
338,WOMEN,WOMEN_Total Crimes against Women,2014,2014,1
339,WOMEN,WOMEN_UnNatural Offences,2014,2014,1


In [108]:
# ============================================================
# CELL 105 — SAVE SCHEMA METADATA
# ============================================================

schema_output_dir = (
    OUTPUTS_DIR /
    "tables"
)

schema_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

feature_metadata_path = (
    schema_output_dir /
    "integrated_feature_metadata.csv"
)

feature_temporal_path = (
    schema_output_dir /
    "feature_temporal_coverage.csv"
)

feature_metadata_df.to_csv(
    feature_metadata_path,
    index=False,
    encoding="utf-8-sig"
)

feature_temporal_coverage.to_csv(
    feature_temporal_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("SCHEMA METADATA SAVED")
print("=" * 80)

print(
    "\nFeature metadata:"
)

print(
    feature_metadata_path.resolve()
)

print(
    "\nFeature temporal coverage:"
)

print(
    feature_temporal_path.resolve()
)

NameError: name 'OUTPUTS_DIR' is not defined

In [109]:
# ============================================================
# CELL 105 — SAVE SCHEMA METADATA
# ============================================================

# Use the existing tables directory created earlier
schema_output_dir = TABLES_DIR

schema_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

feature_metadata_path = (
    schema_output_dir /
    "integrated_feature_metadata.csv"
)

feature_temporal_path = (
    schema_output_dir /
    "feature_temporal_coverage.csv"
)

# ------------------------------------------------------------
# Save feature metadata
# ------------------------------------------------------------

feature_metadata_df.to_csv(
    feature_metadata_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Save temporal coverage
# ------------------------------------------------------------

feature_temporal_coverage.to_csv(
    feature_temporal_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 80)
print("SCHEMA METADATA SAVED")
print("=" * 80)

print(
    "\nFeature metadata:"
)

print(
    feature_metadata_path.resolve()
)

print(
    "\nFeature temporal coverage:"
)

print(
    feature_temporal_path.resolve()
)

print("\n✓ Both metadata files saved successfully.")

SCHEMA METADATA SAVED

Feature metadata:
D:\Major_Project\Crime_Analysis\outputs\tables\integrated_feature_metadata.csv

Feature temporal coverage:
D:\Major_Project\Crime_Analysis\outputs\tables\feature_temporal_coverage.csv

✓ Both metadata files saved successfully.


In [110]:
# ============================================================
# CELL 106 — CANONICAL FEATURE INVENTORY
# ============================================================

import re

def normalize_feature_label(feature_name, crime_group=None):

    value = str(feature_name).strip()

    # Remove crime-group prefix if present
    if crime_group:
        prefix = f"{crime_group}_"

        if value.upper().startswith(prefix.upper()):
            value = value[len(prefix):]

    # Normalize text mechanically
    value = value.upper()

    value = re.sub(
        r"[^A-Z0-9]+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    value = value.strip("_")

    return value


feature_inventory_records = []

for filename, df in integration_tables.items():

    crime_group = (
        core_district_data[
            filename
        ]["CRIME_GROUP"]
        .iloc[0]
    )

    for feature in df.columns:

        if feature in integration_key:
            continue

        normalized_label = (
            normalize_feature_label(
                feature,
                crime_group
            )
        )

        feature_inventory_records.append({

            "crime_group":
                crime_group,

            "filename":
                filename,

            "source_feature":
                feature,

            "normalized_feature":
                normalized_label
        })


feature_inventory_df = pd.DataFrame(
    feature_inventory_records
)

print("=" * 80)
print("CANONICAL FEATURE INVENTORY")
print("=" * 80)

display(
    feature_inventory_df.head(50)
)

CANONICAL FEATURE INVENTORY


,crime_group,filename,source_feature,normalized_feature
0,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_MURDER,MURDER
1,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_ATTEMPT TO MURDER,ATTEMPT_TO_MURDER
2,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER
3,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_RAPE,RAPE
4,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_CUSTODIAL RAPE,CUSTODIAL_RAPE
5,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_OTHER RAPE,OTHER_RAPE
6,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_KIDNAPPING & ABDUCTION,KIDNAPPING_ABDUCTION
7,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS,KIDNAPPING_AND_ABDUCTION_OF_WOMEN_AND_GIRLS
8,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_KIDNAPPING AND ABDUCTION OF OTHERS,KIDNAPPING_AND_ABDUCTION_OF_OTHERS
9,IPC,01_District_wise_crimes_committed_IPC_2001_2012.csv,IPC_DACOITY,DACOITY


In [111]:
# ============================================================
# CELL 107 — EXACT NORMALIZED FEATURE MATCHES
# ============================================================

normalized_feature_matches = (
    feature_inventory_df
    .groupby(
        [
            "crime_group",
            "normalized_feature"
        ]
    )
    .agg(
        source_feature_count=(
            "source_feature",
            "nunique"
        ),

        source_features=(
            "source_feature",
            lambda x: " | ".join(
                sorted(
                    x.astype(str).unique()
                )
            )
        ),

        source_files=(
            "filename",
            lambda x: " | ".join(
                sorted(
                    x.astype(str).unique()
                )
            )
        )
    )
    .reset_index()
)

# Only show features appearing under multiple
# source representations/files
exact_cross_source_matches = (
    normalized_feature_matches[
        normalized_feature_matches[
            "source_feature_count"
        ] > 1
    ]
    .sort_values(
        [
            "crime_group",
            "normalized_feature"
        ]
    )
)

print("=" * 80)
print("EXACT NORMALIZED CROSS-SOURCE MATCHES")
print("=" * 80)

display(
    exact_cross_source_matches
)

EXACT NORMALIZED CROSS-SOURCE MATCHES


,crime_group,normalized_feature,source_feature_count,source_features,source_files
15,IPC,ARSON,2,IPC_ARSON | IPC_Arson,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
16,IPC,ASSAULT_ON_WOMEN_WITH_INTENT_TO_OUTRAGE_HER_MODESTY,2,IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY | IPC_Assault on Women with intent to outrage her Modesty,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
24,IPC,AUTO_THEFT,2,IPC_AUTO THEFT | IPC_Auto Theft,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
26,IPC,CAUSING_DEATH_BY_NEGLIGENCE,2,IPC_CAUSING DEATH BY NEGLIGENCE | IPC_Causing Death by Negligence,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
27,IPC,CHEATING,2,IPC_CHEATING | IPC_Cheating,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
34,IPC,CRIMINAL_BREACH_OF_TRUST,2,IPC_CRIMINAL BREACH OF TRUST | IPC_Criminal Breach of Trust,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
37,IPC,CRUELTY_BY_HUSBAND_OR_HIS_RELATIVES,2,IPC_CRUELTY BY HUSBAND OR HIS RELATIVES | IPC_Cruelty by Husband or his Relatives,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
38,IPC,CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER,2,IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER | IPC_Culpable Homicide not amounting to Murder,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
41,IPC,CUSTODIAL_RAPE,2,IPC_CUSTODIAL RAPE | IPC_Custodial Rape,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...
42,IPC,DACOITY,2,IPC_DACOITY | IPC_Dacoity,01_District_wise_crimes_committed_IPC_2001_2012.csv | 01_District_wise_crimes_committed_IPC_2013.csv | 01_District_w...


In [112]:
# ============================================================
# CELL 108 — FEATURE INVENTORY BY SOURCE FILE
# ============================================================

feature_by_file = (
    feature_inventory_df[
        [
            "crime_group",
            "filename",
            "source_feature",
            "normalized_feature"
        ]
    ]
    .sort_values(
        [
            "crime_group",
            "filename",
            "source_feature"
        ]
    )
)

display(
    feature_by_file
)

,crime_group,filename,source_feature,normalized_feature
318,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Abetment of suicide,ABETMENT_OF_SUICIDE
321,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Buying of girls for prostitution,BUYING_OF_GIRLS_FOR_PROSTITUTION
319,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Exposure and abandonment,EXPOSURE_AND_ABANDONMENT
317,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Foeticide,FOETICIDE
316,CHILDREN,03_District_wise_crimes_committed_against_children_2001_2012.csv,CHILDREN_Kidnapping and Abduction,KIDNAPPING_AND_ABDUCTION
...,...,...,...,...
369,WOMEN,42_District_wise_crimes_committed_against_women_2014.csv,WOMEN_Sexual Harassment,SEXUAL_HARASSMENT
372,WOMEN,42_District_wise_crimes_committed_against_women_2014.csv,WOMEN_Stalking,STALKING
411,WOMEN,42_District_wise_crimes_committed_against_women_2014.csv,WOMEN_Total Crimes against Women,TOTAL_CRIMES_AGAINST_WOMEN
398,WOMEN,42_District_wise_crimes_committed_against_women_2014.csv,WOMEN_UnNatural Offences,UNNATURAL_OFFENCES


In [113]:
# ============================================================
# CELL 110 — FEATURE MAPPING TEMPLATE
# ============================================================

feature_mapping_df = (
    feature_inventory_df[
        [
            "crime_group",
            "source_feature",
            "normalized_feature"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "crime_group",
            "normalized_feature"
        ]
    )
    .copy()
)

# Default:
# preserve the normalized feature name as the candidate
# canonical feature.
feature_mapping_df[
    "canonical_feature"
] = (
    feature_mapping_df[
        "normalized_feature"
    ]
)

# Manual review status
feature_mapping_df[
    "mapping_status"
] = "AUTO_NORMALIZED_ONLY"

# Keep the source variable unchanged
feature_mapping_df[
    "mapping_notes"
] = ""

print("=" * 80)
print("FEATURE MAPPING TEMPLATE")
print("=" * 80)

display(
    feature_mapping_df
)

FEATURE MAPPING TEMPLATE


,crime_group,source_feature,normalized_feature,canonical_feature,mapping_status,mapping_notes
318,CHILDREN,CHILDREN_Abetment of suicide,ABETMENT_OF_SUICIDE,ABETMENT_OF_SUICIDE,AUTO_NORMALIZED_ONLY,
321,CHILDREN,CHILDREN_Buying of girls for prostitution,BUYING_OF_GIRLS_FOR_PROSTITUTION,BUYING_OF_GIRLS_FOR_PROSTITUTION,AUTO_NORMALIZED_ONLY,
319,CHILDREN,CHILDREN_Exposure and abandonment,EXPOSURE_AND_ABANDONMENT,EXPOSURE_AND_ABANDONMENT,AUTO_NORMALIZED_ONLY,
317,CHILDREN,CHILDREN_Foeticide,FOETICIDE,FOETICIDE,AUTO_NORMALIZED_ONLY,
326,CHILDREN,CHILDREN_Infanticid,INFANTICID,INFANTICID,AUTO_NORMALIZED_ONLY,
...,...,...,...,...,...,...
369,WOMEN,WOMEN_Sexual Harassment,SEXUAL_HARASSMENT,SEXUAL_HARASSMENT,AUTO_NORMALIZED_ONLY,
372,WOMEN,WOMEN_Stalking,STALKING,STALKING,AUTO_NORMALIZED_ONLY,
411,WOMEN,WOMEN_Total Crimes against Women,TOTAL_CRIMES_AGAINST_WOMEN,TOTAL_CRIMES_AGAINST_WOMEN,AUTO_NORMALIZED_ONLY,
398,WOMEN,WOMEN_UnNatural Offences,UNNATURAL_OFFENCES,UNNATURAL_OFFENCES,AUTO_NORMALIZED_ONLY,


In [114]:
# ============================================================
# CELL 111 — SAVE FEATURE MAPPING TEMPLATE
# ============================================================

feature_mapping_path = (
    TABLES_DIR /
    "feature_mapping_template.csv"
)

feature_mapping_df.to_csv(
    feature_mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("FEATURE MAPPING TEMPLATE SAVED")
print("=" * 80)

print(
    feature_mapping_path.resolve()
)

FEATURE MAPPING TEMPLATE SAVED
D:\Major_Project\Crime_Analysis\outputs\tables\feature_mapping_template.csv


In [115]:
# ============================================================
# CELL 112 — APPLY SAFE CANONICAL FEATURE NAMES
# ============================================================

# Make a working copy
canonical_feature_mapping = (
    feature_mapping_df.copy()
)

# ------------------------------------------------------------
# Only exact mechanical normalization is automatically
# accepted at this stage.
# ------------------------------------------------------------

canonical_feature_mapping[
    "canonical_feature"
] = (
    canonical_feature_mapping[
        "normalized_feature"
    ]
)

canonical_feature_mapping[
    "mapping_status"
] = "EXACT_NORMALIZED_MATCH"

# ------------------------------------------------------------
# No semantic assumptions have been made.
# ------------------------------------------------------------

canonical_feature_mapping[
    "mapping_notes"
] = (
    "Canonicalized by exact mechanical "
    "name normalization only."
)

print("=" * 80)
print("SAFE CANONICAL FEATURE MAPPING")
print("=" * 80)

display(
    canonical_feature_mapping
)

SAFE CANONICAL FEATURE MAPPING


,crime_group,source_feature,normalized_feature,canonical_feature,mapping_status,mapping_notes
318,CHILDREN,CHILDREN_Abetment of suicide,ABETMENT_OF_SUICIDE,ABETMENT_OF_SUICIDE,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
321,CHILDREN,CHILDREN_Buying of girls for prostitution,BUYING_OF_GIRLS_FOR_PROSTITUTION,BUYING_OF_GIRLS_FOR_PROSTITUTION,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
319,CHILDREN,CHILDREN_Exposure and abandonment,EXPOSURE_AND_ABANDONMENT,EXPOSURE_AND_ABANDONMENT,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
317,CHILDREN,CHILDREN_Foeticide,FOETICIDE,FOETICIDE,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
326,CHILDREN,CHILDREN_Infanticid,INFANTICID,INFANTICID,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
...,...,...,...,...,...,...
369,WOMEN,WOMEN_Sexual Harassment,SEXUAL_HARASSMENT,SEXUAL_HARASSMENT,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
372,WOMEN,WOMEN_Stalking,STALKING,STALKING,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
411,WOMEN,WOMEN_Total Crimes against Women,TOTAL_CRIMES_AGAINST_WOMEN,TOTAL_CRIMES_AGAINST_WOMEN,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.
398,WOMEN,WOMEN_UnNatural Offences,UNNATURAL_OFFENCES,UNNATURAL_OFFENCES,EXACT_NORMALIZED_MATCH,Canonicalized by exact mechanical name normalization only.


In [116]:
# ============================================================
# CELL 113 — CANONICAL FEATURE COLLISION AUDIT
# ============================================================

canonical_collision_audit = (
    canonical_feature_mapping
    .groupby(
        [
            "crime_group",
            "canonical_feature"
        ]
    )
    .agg(
        source_feature_count=(
            "source_feature",
            "nunique"
        ),

        source_features=(
            "source_feature",
            lambda x: " | ".join(
                sorted(
                    x.astype(str).unique()
                )
            )
        )
    )
    .reset_index()
)

canonical_collisions = (
    canonical_collision_audit[
        canonical_collision_audit[
            "source_feature_count"
        ] > 1
    ]
    .sort_values(
        [
            "crime_group",
            "canonical_feature"
        ]
    )
)

print("=" * 80)
print("CANONICAL FEATURE COLLISION AUDIT")
print("=" * 80)

print(
    f"\nCanonical features with "
    f"multiple source representations: "
    f"{len(canonical_collisions)}"
)

display(
    canonical_collisions
)

CANONICAL FEATURE COLLISION AUDIT

Canonical features with multiple source representations: 18


,crime_group,canonical_feature,source_feature_count,source_features
15,IPC,ARSON,2,IPC_ARSON | IPC_Arson
16,IPC,ASSAULT_ON_WOMEN_WITH_INTENT_TO_OUTRAGE_HER_MODESTY,2,IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY | IPC_Assault on Women with intent to outrage her Modesty
24,IPC,AUTO_THEFT,2,IPC_AUTO THEFT | IPC_Auto Theft
26,IPC,CAUSING_DEATH_BY_NEGLIGENCE,2,IPC_CAUSING DEATH BY NEGLIGENCE | IPC_Causing Death by Negligence
27,IPC,CHEATING,2,IPC_CHEATING | IPC_Cheating
34,IPC,CRIMINAL_BREACH_OF_TRUST,2,IPC_CRIMINAL BREACH OF TRUST | IPC_Criminal Breach of Trust
37,IPC,CRUELTY_BY_HUSBAND_OR_HIS_RELATIVES,2,IPC_CRUELTY BY HUSBAND OR HIS RELATIVES | IPC_Cruelty by Husband or his Relatives
38,IPC,CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER,2,IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER | IPC_Culpable Homicide not amounting to Murder
41,IPC,CUSTODIAL_RAPE,2,IPC_CUSTODIAL RAPE | IPC_Custodial Rape
42,IPC,DACOITY,2,IPC_DACOITY | IPC_Dacoity


In [117]:
# ============================================================
# CELL 114 — AMBIGUOUS CANONICAL COLLISIONS
# ============================================================

ambiguous_collisions = (
    canonical_collisions[
        canonical_collisions[
            "source_feature_count"
        ] > 1
    ]
    .copy()
)

# Mark for review
ambiguous_collisions[
    "review_required"
] = True

ambiguous_collisions[
    "review_reason"
] = (
    "Multiple source column names "
    "map to the same normalized feature. "
    "Verify semantic equivalence before "
    "using as a single modelling feature."
)

display(
    ambiguous_collisions
)

,crime_group,canonical_feature,source_feature_count,source_features,review_required,review_reason
15,IPC,ARSON,2,IPC_ARSON | IPC_Arson,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
16,IPC,ASSAULT_ON_WOMEN_WITH_INTENT_TO_OUTRAGE_HER_MODESTY,2,IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY | IPC_Assault on Women with intent to outrage her Modesty,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
24,IPC,AUTO_THEFT,2,IPC_AUTO THEFT | IPC_Auto Theft,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
26,IPC,CAUSING_DEATH_BY_NEGLIGENCE,2,IPC_CAUSING DEATH BY NEGLIGENCE | IPC_Causing Death by Negligence,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
27,IPC,CHEATING,2,IPC_CHEATING | IPC_Cheating,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
34,IPC,CRIMINAL_BREACH_OF_TRUST,2,IPC_CRIMINAL BREACH OF TRUST | IPC_Criminal Breach of Trust,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
37,IPC,CRUELTY_BY_HUSBAND_OR_HIS_RELATIVES,2,IPC_CRUELTY BY HUSBAND OR HIS RELATIVES | IPC_Cruelty by Husband or his Relatives,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
38,IPC,CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER,2,IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER | IPC_Culpable Homicide not amounting to Murder,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
41,IPC,CUSTODIAL_RAPE,2,IPC_CUSTODIAL RAPE | IPC_Custodial Rape,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...
42,IPC,DACOITY,2,IPC_DACOITY | IPC_Dacoity,True,Multiple source column names map to the same normalized feature. Verify semantic equivalence before using as a singl...


In [118]:
# ============================================================
# CELL 115 — SAVE SAFE CANONICAL MAPPING
# ============================================================

canonical_mapping_path = (
    TABLES_DIR /
    "canonical_feature_mapping.csv"
)

canonical_feature_mapping.to_csv(
    canonical_mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("CANONICAL FEATURE MAPPING SAVED")
print("=" * 80)

print(
    canonical_mapping_path.resolve()
)

CANONICAL FEATURE MAPPING SAVED
D:\Major_Project\Crime_Analysis\outputs\tables\canonical_feature_mapping.csv


In [121]:
# ============================================================
# CELL 116 — IDENTIFY CANONICAL COLUMN GROUPS
# ============================================================

canonical_column_groups = {}

for crime_group in merge_order:

    # --------------------------------------------------------
    # Mapping records belonging to this crime group
    # --------------------------------------------------------

    group_mapping = (
        canonical_feature_mapping[
            canonical_feature_mapping[
                "crime_group"
            ] == crime_group
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Each canonical feature may have one or more
    # source representations.
    # --------------------------------------------------------

    for canonical_feature in (
        group_mapping[
            "canonical_feature"
        ]
        .dropna()
        .unique()
    ):

        source_features = (
            group_mapping[
                group_mapping[
                    "canonical_feature"
                ] == canonical_feature
            ][
                "source_feature"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        # ----------------------------------------------------
        # IMPORTANT:
        #
        # source_feature already contains the crime-group
        # prefix in your integrated dataset.
        #
        # Example:
        # IPC_Murder
        # IPC_MURDER
        #
        # Therefore we DO NOT prepend IPC_ again.
        # ----------------------------------------------------

        actual_columns = [
            column
            for column in source_features
            if column in integrated_df.columns
        ]

        if actual_columns:

            canonical_key = (
                f"{crime_group}_{canonical_feature}"
            )

            canonical_column_groups[
                canonical_key
            ] = actual_columns


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("CANONICAL COLUMN GROUPS")
print("=" * 80)

total_groups = len(
    canonical_column_groups
)

multi_column_groups = 0

print(
    f"\nCanonical groups identified: "
    f"{total_groups}"
)

for canonical_name, columns in (
    canonical_column_groups.items()
):

    if len(columns) > 1:

        multi_column_groups += 1

        print(
            f"\n{canonical_name}"
        )

        print(
            "  Source columns:"
        )

        for column in columns:

            print(
                f"    - {column}"
            )

print(
    f"\nCanonical groups with multiple "
    f"source representations: "
    f"{multi_column_groups}"
)

CANONICAL COLUMN GROUPS

Canonical groups identified: 323

IPC_ARSON
  Source columns:
    - IPC_ARSON
    - IPC_Arson

IPC_ASSAULT_ON_WOMEN_WITH_INTENT_TO_OUTRAGE_HER_MODESTY
  Source columns:
    - IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY
    - IPC_Assault on Women with intent to outrage her Modesty

IPC_AUTO_THEFT
  Source columns:
    - IPC_AUTO THEFT
    - IPC_Auto Theft

IPC_CAUSING_DEATH_BY_NEGLIGENCE
  Source columns:
    - IPC_CAUSING DEATH BY NEGLIGENCE
    - IPC_Causing Death by Negligence

IPC_CHEATING
  Source columns:
    - IPC_CHEATING
    - IPC_Cheating

IPC_CRIMINAL_BREACH_OF_TRUST
  Source columns:
    - IPC_CRIMINAL BREACH OF TRUST
    - IPC_Criminal Breach of Trust

IPC_CRUELTY_BY_HUSBAND_OR_HIS_RELATIVES
  Source columns:
    - IPC_CRUELTY BY HUSBAND OR HIS RELATIVES
    - IPC_Cruelty by Husband or his Relatives

IPC_CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER
  Source columns:
    - IPC_CULPABLE HOMICIDE NOT AMOUNTING TO MURDER
    - IPC_Culpable Homi

In [122]:
# ============================================================
# CELL 117 — CANONICAL VALUE CONFLICT AUDIT
# ============================================================

canonical_conflicts = []

for canonical_name, columns in (
    canonical_column_groups.items()
):

    # Only groups with multiple source representations
    if len(columns) <= 1:
        continue

    subset = integrated_df[
        columns
    ]

    # --------------------------------------------------------
    # Compare non-null values row by row
    # --------------------------------------------------------

    conflict_count = 0

    for _, row in subset.iterrows():

        non_null_values = (
            row.dropna()
            .tolist()
        )

        # One or zero values cannot conflict
        if len(non_null_values) <= 1:
            continue

        normalized_values = []

        for value in non_null_values:

            try:
                normalized_values.append(
                    float(value)
                )

            except (
                ValueError,
                TypeError
            ):
                normalized_values.append(
                    str(value).strip()
                )

        if len(
            set(normalized_values)
        ) > 1:

            conflict_count += 1

    # --------------------------------------------------------
    # Record only actual conflicts
    # --------------------------------------------------------

    if conflict_count > 0:

        canonical_conflicts.append({

            "canonical_feature":
                canonical_name,

            "source_column_count":
                len(columns),

            "source_columns":
                " | ".join(columns),

            "conflicting_rows":
                conflict_count
        })


canonical_conflicts_df = pd.DataFrame(
    canonical_conflicts
)

print("=" * 80)
print("CANONICAL VALUE CONFLICT AUDIT")
print("=" * 80)

if canonical_conflicts_df.empty:

    print(
        "\n✓ NO CONFLICTING VALUES FOUND."
    )

    print(
        "\nThe multiple source representations "
        "can be safely coalesced."
    )

else:

    print(
        f"\n⚠ CONFLICTING VALUES FOUND: "
        f"{len(canonical_conflicts_df)} "
        f"canonical features"
    )

    display(
        canonical_conflicts_df
    )

CANONICAL VALUE CONFLICT AUDIT

✓ NO CONFLICTING VALUES FOUND.

The multiple source representations can be safely coalesced.


In [123]:
# ============================================================
# CELL 118 — CONSOLIDATE EXACT-EQUIVALENT FEATURES
# ============================================================

canonical_integrated_df = integrated_df.copy()

consolidation_log = []

for canonical_name, columns in (
    canonical_column_groups.items()
):

    # --------------------------------------------------------
    # Single source representation
    # --------------------------------------------------------

    if len(columns) == 1:

        source_column = columns[0]

        # Rename to canonical name where necessary
        if source_column != canonical_name:

            canonical_integrated_df.rename(
                columns={
                    source_column:
                    canonical_name
                },
                inplace=True
            )

            action = "RENAMED"

        else:

            action = "UNCHANGED"

        consolidation_log.append({

            "canonical_feature":
                canonical_name,

            "source_columns":
                source_column,

            "source_column_count":
                1,

            "action":
                action
        })

        continue

    # --------------------------------------------------------
    # Multiple equivalent source representations
    # --------------------------------------------------------

    # Combine the columns from left to right.
    # Since Cell 117 confirmed no conflicts,
    # the first available non-null value is safe.
    
    combined_series = (
        canonical_integrated_df[
            columns
        ]
        .bfill(axis=1)
        .iloc[:, 0]
    )

    canonical_integrated_df[
        canonical_name
    ] = combined_series

    # Remove the old source representations
    columns_to_drop = [
        column
        for column in columns
        if column != canonical_name
    ]

    canonical_integrated_df.drop(
        columns=columns_to_drop,
        inplace=True,
        errors="ignore"
    )

    consolidation_log.append({

        "canonical_feature":
            canonical_name,

        "source_columns":
            " | ".join(columns),

        "source_column_count":
            len(columns),

        "action":
            "COALESCED"
    })


consolidation_log_df = pd.DataFrame(
    consolidation_log
)

print("=" * 80)
print("CANONICAL FEATURE CONSOLIDATION")
print("=" * 80)

print(
    f"\nCanonical groups processed: "
    f"{len(consolidation_log_df)}"
)

print(
    "\nActions:"
)

print(
    consolidation_log_df[
        "action"
    ].value_counts()
)

display(
    consolidation_log_df
)

CANONICAL FEATURE CONSOLIDATION

Canonical groups processed: 323

Actions:
action
RENAMED      303
COALESCED     18
UNCHANGED      2
Name: count, dtype: int64


,canonical_feature,source_columns,source_column_count,action
0,IPC_ACID_ATTACK,IPC_Acid attack,1,RENAMED
1,IPC_ARSON,IPC_ARSON | IPC_Arson,2,COALESCED
2,IPC_ASSAULT_ON_WOMEN_WITH_INTENT_TO_OUTRAGE_HER_MODESTY,IPC_ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY | IPC_Assault on Women with intent to outrage her Modesty,2,COALESCED
3,IPC_ASSAULT_OR_USE_OF_CRIMINAL_FORCE_TO_WOMEN_WITH_INTENT_TO_DISROBE,IPC_Assault or use of criminal force to women with intent to Disrobe,1,RENAMED
4,IPC_ATTEMPT_TO_ACID_ATTACK,IPC_Attempt to Acid Attack,1,RENAMED
...,...,...,...,...
318,WOMEN_SEXUAL_HARASSMENT,WOMEN_Sexual Harassment,1,RENAMED
319,WOMEN_STALKING,WOMEN_Stalking,1,RENAMED
320,WOMEN_TOTAL_CRIMES_AGAINST_WOMEN,WOMEN_Total Crimes against Women,1,RENAMED
321,WOMEN_UNNATURAL_OFFENCES,WOMEN_UnNatural Offences,1,RENAMED


In [124]:
# ============================================================
# CELL 119 — POST-CONSOLIDATION ROW AUDIT
# ============================================================

print("=" * 80)
print("POST-CONSOLIDATION AUDIT")
print("=" * 80)

before_rows = integrated_df.shape[0]
before_columns = integrated_df.shape[1]

after_rows = canonical_integrated_df.shape[0]
after_columns = canonical_integrated_df.shape[1]

print(
    f"\nBefore consolidation:"
)

print(
    f"  Rows    : {before_rows:,}"
)

print(
    f"  Columns : {before_columns:,}"
)

print(
    f"\nAfter consolidation:"
)

print(
    f"  Rows    : {after_rows:,}"
)

print(
    f"  Columns : {after_columns:,}"
)

print(
    f"\nRow difference:"
    f" {after_rows - before_rows:,}"
)

print(
    f"\nColumn reduction:"
    f" {before_columns - after_columns:,}"
)

duplicate_keys = (
    canonical_integrated_df
    .duplicated(
        integration_key
    )
    .sum()
)

print(
    f"\nDuplicate integration keys:"
    f" {duplicate_keys:,}"
)

# ------------------------------------------------------------
# Hard validation
# ------------------------------------------------------------

assert after_rows == before_rows, (
    "ERROR: Row count changed during "
    "canonical consolidation."
)

assert duplicate_keys == 0, (
    "ERROR: Duplicate integration keys "
    "appeared after consolidation."
)

print(
    "\n✓ Row count preserved."
)

print(
    "✓ Integration keys remain unique."
)

POST-CONSOLIDATION AUDIT

Before consolidation:
  Rows    : 9,856
  Columns : 349

After consolidation:
  Rows    : 9,856
  Columns : 331

Row difference: 0

Column reduction: 18

Duplicate integration keys: 0

✓ Row count preserved.
✓ Integration keys remain unique.


In [129]:
# ============================================================
# CELL 120 — CANONICAL IPC FEATURE CHECK
# ============================================================

ipc_columns = [
    column
    for column in canonical_integrated_df.columns
    if column.startswith("IPC_")
]

print("=" * 80)
print("CANONICAL IPC FEATURES")
print("=" * 80)

print(
    f"\nIPC feature count: "
    f"{len(ipc_columns)}"
)

for column in ipc_columns:
    print(
        f"  {column}"
    )

CANONICAL IPC FEATURES

IPC feature count: 101
  IPC_MURDER
  IPC_ATTEMPT_TO_MURDER
  IPC_RAPE
  IPC_OTHER_RAPE
  IPC_KIDNAPPING_AND_ABDUCTION_OF_WOMEN_AND_GIRLS
  IPC_KIDNAPPING_AND_ABDUCTION_OF_OTHERS
  IPC_DACOITY
  IPC_PREPARATION_AND_ASSEMBLY_FOR_DACOITY
  IPC_ROBBERY
  IPC_BURGLARY
  IPC_THEFT
  IPC_OTHER_THEFT
  IPC_RIOTS
  IPC_CHEATING
  IPC_COUNTERFIETING
  IPC_ARSON
  IPC_HURT_GREVIOUS_HURT
  IPC_INSULT_TO_MODESTY_OF_WOMEN
  IPC_IMPORTATION_OF_GIRLS_FROM_FOREIGN_COUNTRIES
  IPC_TOTAL_IPC_CRIMES
  IPC_ATTEMPT_TO_COMMIT_MURDER
  IPC_ATTEMPT_TO_COMMIT_CULPABLE_HOMICIDE
  IPC_CUSTODIAL_GANG_RAPE
  IPC_CUSTODIAL_OTHER_RAPE
  IPC_RAPE_OTHER_THAN_CUSTODIAL
  IPC_RAPE_GANG_RAPE
  IPC_RAPE_OTHERS
  IPC_ATTEMPT_TO_COMMIT_RAPE
  IPC_KIDNAPPING_ABDUCTION_TOTAL
  IPC_KIDNAPPING_ABDUCTION_IN_ORDER_TO_MURDER
  IPC_KIDNAPPING_FOR_RANSOM
  IPC_KIDNAPPING_ABDUCTION_OF_WOMEN_TO_COMPEL_HER_FOR_MARRIAGE
  IPC_OTHER_KIDNAPPING
  IPC_DACOITY_WITH_MURDER
  IPC_OTHER_DACOITY
  IPC_MAKING_PREPARATION_

In [128]:
# ============================================================
# CELL 121 — SAVE CANONICAL INTEGRATED DATASET
# ============================================================

canonical_integrated_path = (
    PROCESSED_DATA_DIR /
    "district_integrated_canonical.csv"
)

canonical_integrated_df.to_csv(
    canonical_integrated_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("CANONICAL INTEGRATED DATASET SAVED")
print("=" * 80)

print(
    "\nPath:"
)

print(
    canonical_integrated_path.resolve()
)

print(
    f"\nShape:"
    f" {canonical_integrated_df.shape[0]:,} rows × "
    f"{canonical_integrated_df.shape[1]:,} columns"
)

CANONICAL INTEGRATED DATASET SAVED

Path:
D:\Major_Project\Crime_Analysis\data\processed\district_integrated_canonical.csv

Shape: 9,856 rows × 331 columns


In [132]:
# ============================================================
# CELL 122 — FINAL DATASET VALIDATION
# ============================================================

print("=" * 80)
print("FINAL CANONICAL DATASET VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Required key columns
# ------------------------------------------------------------

required_keys = [
    "STATE",
    "UNIT_NAME",
    "YEAR"
]

missing_keys = [
    col
    for col in required_keys
    if col not in canonical_integrated_df.columns
]

assert not missing_keys, (
    f"Missing required key columns: {missing_keys}"
)

# ------------------------------------------------------------
# Row/key validation
# ------------------------------------------------------------

duplicate_keys = (
    canonical_integrated_df
    .duplicated(required_keys)
    .sum()
)

assert duplicate_keys == 0, (
    "Duplicate STATE + UNIT_NAME + YEAR keys found."
)

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

row_count = len(canonical_integrated_df)

column_count = len(
    canonical_integrated_df.columns
)

unique_key_count = (
    canonical_integrated_df[
        required_keys
    ]
    .drop_duplicates()
    .shape[0]
)

missing_cell_count = int(
    canonical_integrated_df
    .isna()
    .sum()
    .sum()
)

memory_mb = (
    canonical_integrated_df
    .memory_usage(deep=True)
    .sum()
    / (1024 ** 2)
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    f"\nRows              : {row_count:,}"
)

print(
    f"Columns           : {column_count:,}"
)

print(
    f"Unique keys       : {unique_key_count:,}"
)

print(
    f"Duplicate keys    : {duplicate_keys:,}"
)

print(
    f"Missing cells     : {missing_cell_count:,}"
)

print(
    f"Memory usage      : {memory_mb:.2f} MB"
)

print(
    "\n✓ Canonical dataset passed validation."
)

FINAL CANONICAL DATASET VALIDATION

Rows              : 9,856
Columns           : 331
Unique keys       : 9,856
Duplicate keys    : 0
Missing cells     : 2,370,879
Memory usage      : 25.65 MB

✓ Canonical dataset passed validation.


In [134]:
# ============================================================
# CELL 123 — SAVE FINAL PROJECT OUTPUTS
# ============================================================

# ------------------------------------------------------------
# Final canonical dataset
# ------------------------------------------------------------

final_canonical_path = (
    PROCESSED_DATA_DIR /
    "district_integrated_final.csv"
)

canonical_integrated_df.to_csv(
    final_canonical_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Final data dictionary
# ------------------------------------------------------------

final_dictionary = pd.DataFrame({

    "column_name":
        canonical_integrated_df.columns,

    "data_type": [
        str(
            canonical_integrated_df[col].dtype
        )
        for col in canonical_integrated_df.columns
    ],

    "missing_count": [
        int(
            canonical_integrated_df[col].isna().sum()
        )
        for col in canonical_integrated_df.columns
    ],

    "missing_pct": [
        round(
            100 *
            canonical_integrated_df[col].isna().mean(),
            2
        )
        for col in canonical_integrated_df.columns
    ],

    "unique_values": [
        int(
            canonical_integrated_df[col].nunique(
                dropna=True
            )
        )
        for col in canonical_integrated_df.columns
    ]
})

dictionary_path = (
    TABLES_DIR /
    "final_data_dictionary.csv"
)

final_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Final integration report
# ------------------------------------------------------------

final_report = pd.DataFrame([{
    "dataset":
        "India District Crime Integrated Dataset",

    "rows":
        len(canonical_integrated_df),

    "columns":
        len(canonical_integrated_df.columns),

    "unique_keys":
        canonical_integrated_df[
            required_keys
        ].drop_duplicates().shape[0],

    "duplicate_keys":
        int(duplicate_keys),

    "total_missing_cells":
        int(
            canonical_integrated_df
            .isna()
            .sum()
            .sum()
        )
}])

report_path = (
    TABLES_DIR /
    "final_integration_report.csv"
)

final_report.to_csv(
    report_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("=" * 80)
print("FINAL PROJECT DATASET CREATED")
print("=" * 80)

print(
    "\nDataset:"
)

print(
    final_canonical_path.resolve()
)

print(
    "\nData dictionary:"
)

print(
    dictionary_path.resolve()
)

print(
    "\nIntegration report:"
)

print(
    report_path.resolve()
)

print(
    "\n✓ PHASE 1 — DATA INTEGRATION COMPLETE."
)

FINAL PROJECT DATASET CREATED

Dataset:
D:\Major_Project\Crime_Analysis\data\processed\district_integrated_final.csv

Data dictionary:
D:\Major_Project\Crime_Analysis\outputs\tables\final_data_dictionary.csv

Integration report:
D:\Major_Project\Crime_Analysis\outputs\tables\final_integration_report.csv

✓ PHASE 1 — DATA INTEGRATION COMPLETE.


In [135]:
# ============================================================
# CELL 124 — FINAL CHECKPOINT
# ============================================================

print("=" * 80)
print("PROJECT DATA INTEGRATION — FINAL CHECKPOINT")
print("=" * 80)

print("""
✓ 76 raw CSV files audited
✓ 14 district-level datasets identified
✓ Dataset families classified
✓ Geographic dimensions normalized
✓ Temporal dimensions normalized
✓ Duplicate records investigated
✓ Conflicting duplicates investigated
✓ Core keys validated
✓ 5 crime families integrated
✓ 9,856 unique district-year records created
✓ Schema evolution documented
✓ Feature temporal coverage documented
✓ Canonical feature names created
✓ 18 duplicate source representations consolidated
✓ No canonical value conflicts
✓ 331-column canonical dataset created
✓ Final dataset saved
✓ Data dictionary saved
✓ Integration report saved

NEXT PHASE:
EDA → Feature Engineering → ML/DL → Evaluation
""")

print(
    f"FINAL DATASET: "
    f"{canonical_integrated_df.shape[0]:,} rows × "
    f"{canonical_integrated_df.shape[1]:,} columns"
)

PROJECT DATA INTEGRATION — FINAL CHECKPOINT

✓ 76 raw CSV files audited
✓ 14 district-level datasets identified
✓ Dataset families classified
✓ Geographic dimensions normalized
✓ Temporal dimensions normalized
✓ Duplicate records investigated
✓ Conflicting duplicates investigated
✓ Core keys validated
✓ 5 crime families integrated
✓ 9,856 unique district-year records created
✓ Schema evolution documented
✓ Feature temporal coverage documented
✓ Canonical feature names created
✓ 18 duplicate source representations consolidated
✓ No canonical value conflicts
✓ 331-column canonical dataset created
✓ Final dataset saved
✓ Data dictionary saved
✓ Integration report saved

NEXT PHASE:
EDA → Feature Engineering → ML/DL → Evaluation

FINAL DATASET: 9,856 rows × 331 columns
